## 建立專題環境

這份 notebook 會從資料讀取、資料可讀化、知識圖譜建立、GraphRAG 查詢、Multi-Agent 診斷到 Streamlit 系統輸出，完整示範一個「生成式 AI 製造異常處理輔助系統」。

### 本專題的核心流程如下：

1. 讀取 Excel 原始資料。
2. 將代碼型欄位轉成可讀資料表。
3. 建立設備、異常、SOP、歷史事件與原因之間的知識圖譜。
4. 透過 GraphRAG 取回相關資料。
5. 由多個 Agent 分工產生異常診斷、SOP 建議、備件提醒、人工確認提醒與通報內容。
6. 最後輸出成 Streamlit 互動式系統。


In [1]:
!pip install pandas openpyxl networkx plotly streamlit pyngrok python-dotenv pyvis


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 51.7 MB/s eta 0:00:00


## 安裝套件

本段會安裝專題需要用到的 Python 套件。

主要套件用途如下：
1. `pandas`、`openpyxl`：讀取與整理 Excel 資料。
2. `networkx`：建立知識圖譜。
3. `plotly`：製作互動式圖表。
4. `streamlit`：建立網頁版互動系統。
5. `pyngrok` 或 localtunnel：協助在 Colab 中開啟外部連線。


In [2]:
!pip install pandas openpyxl networkx plotly streamlit pyngrok python-dotenv pyvis


## Excel 資料

本段會上傳並讀取老師提供的 Excel 資料集。

這份 Excel 是整個專題的資料來源，裡面包含異常分類、設備主檔、SOP 流程、感測器快照、歷史異常事件與人工處置經驗等工作表。後續所有分析、查詢與 AI 摘要都會以這份資料為基礎。


In [3]:
from google.colab import files

uploaded = files.upload()

Saving 資料集.xlsx to 資料集.xlsx


### 確認檔案是否上傳成功

本段說明「確認檔案是否上傳成功」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


In [4]:
import os

print(os.listdir())

['.config', '資料集.xlsx', 'sample_data']


### 讀取 Excel 的所有工作表

本段說明「讀取 Excel 的所有工作表」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


In [5]:
import pandas as pd

excel_path = "資料集.xlsx"

xls = pd.ExcelFile(excel_path)

print("這份 Excel 裡面有這些工作表：")
for sheet in xls.sheet_names:
    print(sheet)

這份 Excel 裡面有這些工作表：
01_table_dictionary
02_field_dictionary
03_cate_main
04_cate_mid
05_cate_detail
06_equipment
07_state
08_state_mapping
09_monitor_function
10_parameter_spec
11_sop_main
12_sop_step
13_abnormal_event
14_event_step_check
15_sensor_snapshot
16_attachment
17_roles
18_application_methods
19_embedded_source_index
20_embedded_hrm_bite_sop
21_embedded_heating_rules
22_embedded_heating_actual
23_embedded_kb2_speed_current
24_embedded_kb2_notes
25_embedded_case_summary


### 把每張表讀成 DataFrame

本段說明「把每張表讀成 DataFrame」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


In [6]:
tables = {}

for sheet in xls.sheet_names:
    tables[sheet] = pd.read_excel(excel_path, sheet_name=sheet)

print("已成功讀取所有資料表")
print("資料表數量：", len(tables))

已成功讀取所有資料表
資料表數量： 25


### 每張表有幾筆資料

本段說明「每張表有幾筆資料」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


In [7]:
summary = []

for sheet_name, df in tables.items():
    summary.append({
        "sheet_name": sheet_name,
        "rows": len(df),
        "columns": len(df.columns)
    })

summary_df = pd.DataFrame(summary)
summary_df

,sheet_name,rows,columns
0,01_table_dictionary,21,4
1,02_field_dictionary,33,7
2,03_cate_main,4,4
3,04_cate_mid,7,5
4,05_cate_detail,11,5
5,06_equipment,24,6
6,07_state,64,6
7,08_state_mapping,76,8
8,09_monitor_function,16,7
9,10_parameter_spec,29,13


## 建立我們會用到的主要資料表

本段會把 Excel 中常用的工作表指定成變數，方便後續程式直接呼叫。

這些資料表可以分成幾類：
1. 異常分類與設備主檔。
2. SOP 主檔與 SOP 步驟。
3. 感測器與監控參數。
4. 歷史異常事件與人工處置紀錄。


In [8]:
cate_main = tables["03_cate_main"]
cate_mid = tables["04_cate_mid"]
cate_detail = tables["05_cate_detail"]
equipment = tables["06_equipment"]
state = tables["07_state"]
state_mapping = tables["08_state_mapping"]
monitor_function = tables["09_monitor_function"]
parameter_spec = tables["10_parameter_spec"]
sop_main = tables["11_sop_main"]
sop_step = tables["12_sop_step"]
abnormal_event = tables["13_abnormal_event"]
event_step_check = tables["14_event_step_check"]
sensor_snapshot = tables["15_sensor_snapshot"]
attachment = tables["16_attachment"]
roles = tables["17_roles"]

print("主要資料表已建立完成")

主要資料表已建立完成


## 理解這些表的角色

在正式進行資料處理前，先釐清各張資料表的角色。

本專題不是只使用單一表格，而是透過多張表串接出「設備 → 異常狀況 → SOP → 歷史事件 → 原因與處置」的完整關係。


## 原始異常事件資料

本段先查看原始異常事件資料。

原始資料中常會出現 `equipment_id`、`state_id`、`sop_id` 等代碼。這些代碼適合系統串接，但不適合人直接閱讀，因此後面會建立可讀版資料視圖。


In [9]:
abnormal_event.head()

,event_id,occurred_at,line_area,cate_detail_id,equipment_id,state_id,steel_grade,work_order,product_spec,severity,...,impact_qty,root_cause_category,cause_summary,action_summary,handler_role,assigned_team,closed_at,close_result,sop_id,source_generated_rule
0,EVT00001,2026-03-06 10:23,ROD線,CDT001,EQ006,ST011,S30201,WO26038405,Φ6.5mm,A,...,152.4,設備磨耗,KB2 軸承破裂，初判原因：設備磨耗,更換磨耗零件並復測,班長,甲班,2026-03-06 12:22,已復機,SOP011,generated_six_month_history
1,EVT00002,2026-03-08 21:05,ROD線,CDT001,EQ011,ST023,S316L1,WO26032810,BIC-2.0,A,...,129.7,參數設定錯誤,TMB1 TC Roll破裂，初判原因：參數設定錯誤,依標準值調整參數並鎖定版次,班長,乙班,2026-03-08 23:21,已復機,NaN,generated_six_month_history
2,EVT00003,2026-04-12 13:38,ROD線,CDT004,EQ011,ST024,S41010,WO26049109,Φ8.0mm,A,...,131.3,設備磨耗,TMB1 Guide Roll破裂，初判原因：設備磨耗,更換磨耗零件並復測,製程工程師,製程課,2026-04-12 15:33,已復機,NaN,generated_six_month_history
3,EVT00004,2026-04-18 19:39,ROD線,CDT001,EQ010,ST020,S40976,WO26045032,Φ8.0mm,B,...,87.2,設備磨耗,FB 導器磨溝，初判原因：設備磨耗,更換磨耗零件並復測,製程工程師,丙班,2026-04-18 21:15,已復機,NaN,generated_six_month_history
4,EVT00005,2026-05-03 12:11,ROD線,NaN,EQ021,ST058,S316L1,WO26056880,BIC-2.0,A,...,38.8,保養不足,PLC 通訊中斷，初判原因：保養不足,補做保養並建立預防保全項目,製程工程師,甲班,2026-05-03 12:43,已復機,NaN,generated_six_month_history


### 說明

這時候你會發現問題：

equipment_id = EQ006

state_id = ST011

cate_detail_id = CDT001

sop_id = SOP011

這些 ID 對人來說很難讀，所以我們下一步要把它翻譯成人看得懂的資料。


## 建立「異常事件可讀總表」

本段會建立 `clean_event_view`，也就是異常事件可讀版。

它會把原本事件資料中的設備代碼、異常狀況代碼與 SOP 代碼轉成中文或可理解的名稱，讓後續分析、看板與 AI 診斷可以直接使用。


In [10]:
clean_event_view = abnormal_event.copy()

# 1. 加上設備名稱
clean_event_view = clean_event_view.merge(
    equipment[["equipment_id", "equipment_code", "equipment_name"]],
    on="equipment_id",
    how="left"
)

# 2. 加上異常狀況名稱
clean_event_view = clean_event_view.merge(
    state[["state_id", "state_name", "default_severity"]],
    on="state_id",
    how="left"
)

# 3. 加上細分類名稱
clean_event_view = clean_event_view.merge(
    cate_detail[["cate_detail_id", "name"]].rename(columns={"name": "cate_detail_name"}),
    on="cate_detail_id",
    how="left"
)

# 4. 加上 SOP 名稱
clean_event_view = clean_event_view.merge(
    sop_main[["sop_id", "sop_name", "sop_desc", "owner_role"]],
    on="sop_id",
    how="left"
)

# 5. 挑出重要欄位
clean_event_view = clean_event_view[
    [
        "event_id",
        "occurred_at",
        "line_area",
        "equipment_id",
        "equipment_code",
        "equipment_name",
        "cate_detail_name",
        "state_id",
        "state_name",
        "steel_grade",
        "work_order",
        "product_spec",
        "severity",
        "default_severity",
        "detector",
        "current_status",
        "downtime_min",
        "impact_qty",
        "root_cause_category",
        "cause_summary",
        "action_summary",
        "handler_role",
        "assigned_team",
        "close_result",
        "sop_id",
        "sop_name",
        "sop_desc",
        "owner_role"
    ]
]

clean_event_view.head()

,event_id,occurred_at,line_area,equipment_id,equipment_code,equipment_name,cate_detail_name,state_id,state_name,steel_grade,...,root_cause_category,cause_summary,action_summary,handler_role,assigned_team,close_result,sop_id,sop_name,sop_desc,owner_role
0,EVT00001,2026-03-06 10:23,ROD線,EQ006,KB2,KB2,ROD,ST011,軸承破裂,S30201,...,設備磨耗,KB2 軸承破裂，初判原因：設備磨耗,更換磨耗零件並復測,班長,甲班,已復機,SOP011,KB2-軸承破裂異常處理,KB2發生軸承破裂時的標準處置流程,設備/製程工程師
1,EVT00002,2026-03-08 21:05,ROD線,EQ011,TMB1,TMB1,ROD,ST023,TC Roll破裂,S316L1,...,參數設定錯誤,TMB1 TC Roll破裂，初判原因：參數設定錯誤,依標準值調整參數並鎖定版次,班長,乙班,已復機,NaN,NaN,NaN,NaN
2,EVT00003,2026-04-12 13:38,ROD線,EQ011,TMB1,TMB1,ROD,ST024,Guide Roll破裂,S41010,...,設備磨耗,TMB1 Guide Roll破裂，初判原因：設備磨耗,更換磨耗零件並復測,製程工程師,製程課,已復機,NaN,NaN,NaN,NaN
3,EVT00004,2026-04-18 19:39,ROD線,EQ010,FB,FB,ROD,ST020,導器磨溝,S40976,...,設備磨耗,FB 導器磨溝，初判原因：設備磨耗,更換磨耗零件並復測,製程工程師,丙班,已復機,NaN,NaN,NaN,NaN
4,EVT00005,2026-05-03 12:11,ROD線,EQ021,PLC,PLC,NaN,ST058,通訊中斷,S316L1,...,保養不足,PLC 通訊中斷，初判原因：保養不足,補做保養並建立預防保全項目,製程工程師,甲班,已復機,NaN,NaN,NaN,NaN


### 說明

先解決資料可解讀性不佳，這張 clean_event_view 會是整個專題最重要的基礎表。


## 檢查可讀化後的結果

本段用來檢查 `clean_event_view` 是否成功建立。

重點是確認原本難懂的 ID 是否已經轉換成可讀欄位，例如設備名稱、異常狀況、SOP 名稱、原因分類與處置摘要。


In [11]:
clean_event_view[
    [
        "event_id",
        "occurred_at",
        "line_area",
        "equipment_code",
        "equipment_name",
        "state_name",
        "severity",
        "downtime_min",
        "root_cause_category",
        "cause_summary",
        "action_summary",
        "sop_name"
    ]
].head(10)

,event_id,occurred_at,line_area,equipment_code,equipment_name,state_name,severity,downtime_min,root_cause_category,cause_summary,action_summary,sop_name
0,EVT00001,2026-03-06 10:23,ROD線,KB2,KB2,軸承破裂,A,70,設備磨耗,KB2 軸承破裂，初判原因：設備磨耗,更換磨耗零件並復測,KB2-軸承破裂異常處理
1,EVT00002,2026-03-08 21:05,ROD線,TMB1,TMB1,TC Roll破裂,A,80,參數設定錯誤,TMB1 TC Roll破裂，初判原因：參數設定錯誤,依標準值調整參數並鎖定版次,NaN
2,EVT00003,2026-04-12 13:38,ROD線,TMB1,TMB1,Guide Roll破裂,A,81,設備磨耗,TMB1 Guide Roll破裂，初判原因：設備磨耗,更換磨耗零件並復測,NaN
3,EVT00004,2026-04-18 19:39,ROD線,FB,FB,導器磨溝,B,50,設備磨耗,FB 導器磨溝，初判原因：設備磨耗,更換磨耗零件並復測,NaN
4,EVT00005,2026-05-03 12:11,ROD線,PLC,PLC,通訊中斷,A,19,保養不足,PLC 通訊中斷，初判原因：保養不足,補做保養並建立預防保全項目,NaN
5,EVT00006,2026-04-28 08:57,BAR線,HRM,HRM,軋輥打滑,B,43,設備磨耗,HRM 軋輥打滑，初判原因：設備磨耗,更換磨耗零件並復測,NaN
6,EVT00007,2026-01-21 17:50,BAR線,KB2,KB2,軋輥破裂,A,95,設備磨耗,KB2 軋輥破裂，初判原因：設備磨耗,更換磨耗零件並復測,NaN
7,EVT00008,2026-03-14 18:25,ROD線,KB1,KB1,傳動軸斷裂,A,72,設備磨耗,KB1 傳動軸斷裂，初判原因：設備磨耗,更換磨耗零件並復測,KB1-傳動軸斷裂異常處理
8,EVT00009,2026-04-28 19:54,BAR線,KB1,KB1,Guide Roll破裂,A,62,感測器/通訊異常,KB1 Guide Roll破裂，初判原因：感測器/通訊異常,檢查訊號線、重啟通訊並補登資料,NaN
9,EVT00010,2026-02-14 19:58,ROD線,PLC,PLC,通訊中斷,A,78,人工操作疏失,PLC 通訊中斷，初判原因：人工操作疏失,重新教育與增加防呆檢核,NaN


## 建立「SOP 可讀流程表」

本段會建立 `clean_sop_view`。

原始 SOP 分散在 SOP 主檔與 SOP 步驟表中，因此這裡會把兩者合併，整理出「哪個設備、哪個異常狀況、對應哪份 SOP、有哪些處理步驟」的可讀流程表。


In [12]:
clean_sop_view = sop_step.copy()

# 加上 SOP 主檔資訊
clean_sop_view = clean_sop_view.merge(
    sop_main[
        [
            "sop_id",
            "equipment_id",
            "state_id",
            "sop_name",
            "sop_desc",
            "version",
            "status",
            "owner_role"
        ]
    ],
    on="sop_id",
    how="left"
)

# 加上設備資訊
clean_sop_view = clean_sop_view.merge(
    equipment[
        [
            "equipment_id",
            "equipment_code",
            "equipment_name",
            "line_area"
        ]
    ],
    on="equipment_id",
    how="left"
)

# 加上異常狀況
clean_sop_view = clean_sop_view.merge(
    state[
        [
            "state_id",
            "state_name",
            "default_severity"
        ]
    ],
    on="state_id",
    how="left"
)

# 加上監控項目資訊
clean_sop_view = clean_sop_view.merge(
    monitor_function[
        [
            "monitor_id",
            "monitor_name",
            "subgroup",
            "data_source_system",
            "update_frequency_sec"
        ]
    ],
    on="monitor_id",
    how="left"
)

# 挑選重要欄位
clean_sop_view = clean_sop_view[
    [
        "sop_id",
        "sop_name",
        "sop_desc",
        "equipment_code",
        "equipment_name",
        "line_area",
        "state_name",
        "step_id",
        "parent_step_id",
        "sort_order",
        "branch_label",
        "step_title",
        "step_content",
        "check_type",
        "monitor_id",
        "monitor_name",
        "standard_text",
        "image_needed",
        "safety_note",
        "version",
        "status",
        "owner_role"
    ]
]

clean_sop_view.head(10)

,sop_id,sop_name,sop_desc,equipment_code,equipment_name,line_area,state_name,step_id,parent_step_id,sort_order,...,step_content,check_type,monitor_id,monitor_name,standard_text,image_needed,safety_note,version,status,owner_role
0,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP001,NaN,1,...,HRM打滑,manual,NaN,NaN,異常起點,False,先確認現場安全,v1.0,active,軋製課/現場班長
1,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP002,STEP001,1,...,當支退爐、在軋一支是否打滑,manual,NaN,NaN,確認是否只發生單支或持續發生,False,避免人員靠近入料區,v1.0,active,軋製課/現場班長
2,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP003,STEP002,1,...,確認加熱爐時間與溫度作業,auto,MON001,加熱爐在爐時間,依鋼種有效加熱時間±30分鐘，並確認爐溫,False,NaN,v1.0,active,軋製課/現場班長
3,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP004,STEP002,2,...,持續軋延分析打滑鋼種是否異常,manual,NaN,NaN,若非持續異常可紀錄觀察,False,NaN,v1.0,active,軋製課/現場班長
4,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP005,STEP003,1,...,PI11夾輥作動夾持秒,auto,MON003,PI11夾輥作動夾持秒,3~6秒,False,NaN,v1.0,active,軋製課/現場班長
5,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP006,STEP003,2,...,依據加熱溫度管制作業,manual,NaN,NaN,依加熱爐規範調整,False,NaN,v1.0,active,軋製課/現場班長
6,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP007,STEP005,1,...,HRM水量設定與噴嘴阻塞與角度目視檢查,hybrid,MON004,HRM水量設定,水量1500~1900 L/min，噴嘴角度正常,True,NaN,v1.0,active,軋製課/現場班長
7,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP008,STEP005,2,...,依據鋼種夾持規定秒數或視情況增加秒數,manual,NaN,NaN,依鋼種規定秒數調整,False,NaN,v1.0,active,軋製課/現場班長
8,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP009,STEP007,1,...,軋輥咬入點痕跡,manual,MON006,軋輥咬入點痕跡,咬入點痕跡應正常且無明顯偏移,True,NaN,v1.0,active,軋製課/現場班長
9,SOP001,HRM打滑,HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷,HRM,HRM,ROD線,打滑,STEP010,STEP007,2,...,清理噴水環/角度調整/水量依鋼種設定,manual,NaN,NaN,完成後復查水量與噴嘴,True,NaN,v1.0,active,軋製課/現場班長


### 說明

這張表可以回答：

某個設備發生某個異常時，要執行哪份 SOP？

SOP 有哪些步驟？

每一步要檢查什麼？

哪些步驟有綁感測器？

哪些步驟需要圖片？

有哪些安全提醒？

這是之後 SOP Agent 會用的資料。


## 查看某一份 SOP 的流程

本段會挑選一份 SOP，檢查它的步驟順序與流程內容。

這可以確認 SOP 是否成功從原始資料轉成系統可讀、使用者也能理解的處理流程。


In [13]:
hrm_slip_sop = clean_sop_view[
    clean_sop_view["sop_name"].astype(str).str.contains("HRM打滑", na=False)
].sort_values(["sop_id", "sort_order"])

hrm_slip_sop[
    [
        "sop_id",
        "sop_name",
        "step_id",
        "parent_step_id",
        "sort_order",
        "branch_label",
        "step_title",
        "step_content",
        "monitor_name",
        "standard_text",
        "safety_note"
    ]
]

,sop_id,sop_name,step_id,parent_step_id,sort_order,branch_label,step_title,step_content,monitor_name,standard_text,safety_note
0,SOP001,HRM打滑,STEP001,NaN,1,NaN,根節點,HRM打滑,NaN,異常起點,先確認現場安全
1,SOP001,HRM打滑,STEP002,STEP001,1,->,判斷是否持續打滑,當支退爐、在軋一支是否打滑,NaN,確認是否只發生單支或持續發生,避免人員靠近入料區
2,SOP001,HRM打滑,STEP003,STEP002,1,是,確認加熱條件,確認加熱爐時間與溫度作業,加熱爐在爐時間,依鋼種有效加熱時間±30分鐘，並確認爐溫,NaN
4,SOP001,HRM打滑,STEP005,STEP003,1,正常,檢查夾輥,PI11夾輥作動夾持秒,PI11夾輥作動夾持秒,3~6秒,NaN
6,SOP001,HRM打滑,STEP007,STEP005,1,正常,檢查水量與噴嘴,HRM水量設定與噴嘴阻塞與角度目視檢查,HRM水量設定,水量1500~1900 L/min，噴嘴角度正常,NaN
8,SOP001,HRM打滑,STEP009,STEP007,1,正常,檢查咬入點,軋輥咬入點痕跡,軋輥咬入點痕跡,咬入點痕跡應正常且無明顯偏移,NaN
10,SOP001,HRM打滑,STEP011,STEP009,1,正常,檢查入口檔板,HRM入口檔板作動,入口檔板作動,檔板作動正常且高程一致,NaN
12,SOP001,HRM打滑,STEP013,STEP011,1,正常,復機觀察,開四台隧道加熱軋延第二支，優先410鋼種，70%/70%/70%/60%,NaN,復機後至少觀察2支,NaN
14,SOP001,HRM打滑,STEP015,STEP013,1,正常,結案,持續軋延觀察,NaN,穩定後關閉事件,NaN
3,SOP001,HRM打滑,STEP004,STEP002,2,否,持續觀察,持續軋延分析打滑鋼種是否異常,NaN,若非持續異常可紀錄觀察,NaN


### 說明

將原本分散在 sop_main 與 sop_step 的資料整合，建立可讀化 SOP 流程表，讓系統可以根據設備與異常狀況查詢對應的維修步驟。


## 建立「強化 SOP 可讀流程表」

這一段不會取代原本的 `clean_sop_view`，而是在它的基礎上新增 `sop_detail_view`。

原本 SOP 只是步驟表，這裡進一步補上：

1. SOP 階段：異常確認、原因排查、處置修正、復機確認、紀錄回饋。
2. 判斷方式中文化：人工確認、系統自動判斷、系統輔助加人工確認。
3. 佐證資料：數值截圖、現場照片、人工備註。
4. 異常時下一步：讓 SOP 不只是列步驟，而是變成現場處理導引。


In [14]:
# =========================================================
# 建立強化版 SOP 可讀資料表 sop_detail_view
# 目的：保留原本 clean_sop_view，同時增加 SOP 階段、人工/自動判斷、佐證資料與下一步提示
# =========================================================

import pandas as pd


def classify_sop_stage(step_title, step_content):
    """
    依照 SOP 步驟文字，將步驟粗略歸類到現場處理階段。
    這是展示用的可讀化分類，不會改動原始 SOP 表。
    """
    text = f"{step_title} {step_content}"

    if any(k in text for k in ["根節點", "確認是否", "判斷是否", "確認異常", "持續", "觀察", "是否仍"]):
        return "1. 異常確認"

    if any(k in text for k in ["安全", "停機", "隔離", "防護", "通知", "危險"]):
        return "2. 安全處置"

    if any(k in text for k in ["檢查", "確認", "比對", "量測", "查看", "判斷", "監控"]):
        return "3. 原因排查"

    if any(k in text for k in ["調整", "更換", "清除", "修正", "處理", "復歸", "退爐", "降速"]):
        return "4. 處置修正"

    if any(k in text for k in ["復機", "試軋", "恢復", "結案", "再確認", "OK"]):
        return "5. 復機確認"

    if any(k in text for k in ["紀錄", "回報", "上傳", "改善", "修訂", "回填"]):
        return "6. 紀錄回饋"

    return "3. 原因排查"


def translate_check_type(check_type):
    """將 check_type 轉成現場人員看得懂的中文。"""
    value = str(check_type).lower()

    if value == "auto":
        return "系統自動判斷"
    if value == "manual":
        return "人工確認"
    if value == "hybrid":
        return "系統輔助 + 人工確認"
    return "未標示，需人工確認"


def build_evidence_required(row):
    """依照監控項目、圖片需求與安全提醒，推估應留下的佐證資料。"""
    evidence = []

    if pd.notna(row.get("monitor_name")):
        evidence.append("系統參數截圖 / 感測數值")

    image_needed = str(row.get("image_needed", "")).lower()
    if image_needed in ["true", "1", "yes", "y"]:
        evidence.append("現場照片")

    if pd.notna(row.get("safety_note")):
        evidence.append("安全確認紀錄")

    if not evidence:
        evidence.append("人工確認備註")

    return "、".join(evidence)


def build_next_action_hint(row):
    """依照 SOP 文字產生異常時的下一步提示。"""
    title = str(row.get("step_title", ""))
    content = str(row.get("step_content", ""))
    text = title + " " + content

    if any(k in text for k in ["加熱", "溫度", "在爐"]):
        return "若加熱條件異常，請先比對鋼種標準，必要時調整加熱參數或退爐。"

    if any(k in text for k in ["夾輥", "PI11", "咬入"]):
        return "若夾持秒數或咬入狀況異常，請確認夾輥作動、咬入點與相關機構。"

    if any(k in text for k in ["水量", "噴嘴", "冷卻"]):
        return "若水量或噴嘴異常，請確認水量設定、噴嘴阻塞與噴嘴角度。"

    if any(k in text for k in ["軸承", "傳動", "馬達", "電流", "Guide", "軸心"]):
        return "若設備機構或電流異常，請通知設備工程師進一步檢查。"

    if any(k in text for k in ["訊號", "PLC", "sensor", "通訊", "Scanner", "Encoder"]):
        return "若訊號異常，請確認 PLC、通訊狀態與感測器連線。"

    return "若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。"


sop_detail_view = clean_sop_view.copy()

sop_detail_view["sop_stage"] = sop_detail_view.apply(
    lambda row: classify_sop_stage(row.get("step_title", ""), row.get("step_content", "")),
    axis=1
)

sop_detail_view["check_method_label"] = sop_detail_view["check_type"].apply(translate_check_type)

sop_detail_view["evidence_required"] = sop_detail_view.apply(build_evidence_required, axis=1)

sop_detail_view["next_action_hint"] = sop_detail_view.apply(build_next_action_hint, axis=1)

sop_detail_view["readable_step"] = sop_detail_view.apply(
    lambda row: (
        f"{row.get('step_title', '')}：{row.get('step_content', '')}\n"
        f"判斷方式：{row.get('check_method_label', '')}\n"
        f"監控項目：{row.get('monitor_name', '無') if pd.notna(row.get('monitor_name')) else '無'}\n"
        f"判斷標準：{row.get('standard_text', '未提供') if pd.notna(row.get('standard_text')) else '未提供'}\n"
        f"需留存佐證：{row.get('evidence_required', '')}\n"
        f"異常時建議：{row.get('next_action_hint', '')}"
    ),
    axis=1
)

print("sop_detail_view 已建立，資料筆數：", len(sop_detail_view))
sop_detail_view[[
    "sop_id", "sop_name", "equipment_code", "state_name", "sop_stage",
    "step_title", "check_method_label", "monitor_name", "evidence_required", "next_action_hint"
]].head(10)


sop_detail_view 已建立，資料筆數： 169


,sop_id,sop_name,equipment_code,state_name,sop_stage,step_title,check_method_label,monitor_name,evidence_required,next_action_hint
0,SOP001,HRM打滑,HRM,打滑,1. 異常確認,根節點,人工確認,NaN,安全確認紀錄,若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。
1,SOP001,HRM打滑,HRM,打滑,1. 異常確認,判斷是否持續打滑,人工確認,NaN,安全確認紀錄,若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。
2,SOP001,HRM打滑,HRM,打滑,3. 原因排查,確認加熱條件,系統自動判斷,加熱爐在爐時間,系統參數截圖 / 感測數值,若加熱條件異常，請先比對鋼種標準，必要時調整加熱參數或退爐。
3,SOP001,HRM打滑,HRM,打滑,1. 異常確認,持續觀察,人工確認,NaN,人工確認備註,若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。
4,SOP001,HRM打滑,HRM,打滑,3. 原因排查,檢查夾輥,系統自動判斷,PI11夾輥作動夾持秒,系統參數截圖 / 感測數值,若夾持秒數或咬入狀況異常，請確認夾輥作動、咬入點與相關機構。
5,SOP001,HRM打滑,HRM,打滑,4. 處置修正,修正加熱條件,人工確認,NaN,人工確認備註,若加熱條件異常，請先比對鋼種標準，必要時調整加熱參數或退爐。
6,SOP001,HRM打滑,HRM,打滑,3. 原因排查,檢查水量與噴嘴,系統輔助 + 人工確認,HRM水量設定,系統參數截圖 / 感測數值、現場照片,若水量或噴嘴異常，請確認水量設定、噴嘴阻塞與噴嘴角度。
7,SOP001,HRM打滑,HRM,打滑,4. 處置修正,調整夾持秒數,人工確認,NaN,人工確認備註,若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。
8,SOP001,HRM打滑,HRM,打滑,3. 原因排查,檢查咬入點,人工確認,軋輥咬入點痕跡,系統參數截圖 / 感測數值、現場照片,若夾持秒數或咬入狀況異常，請確認夾輥作動、咬入點與相關機構。
9,SOP001,HRM打滑,HRM,打滑,4. 處置修正,清理/調整水量,人工確認,NaN,現場照片,若水量或噴嘴異常，請確認水量設定、噴嘴阻塞與噴嘴角度。


## 建立「感測器判斷可讀表」

本段會建立 `clean_sensor_view`。

它的功能是把感測器快照、監控項目、實際值、標準上下限與判斷結果整理在一起，讓系統可以判斷某些 SOP 步驟是否有數值異常。


In [15]:
clean_sensor_view = sensor_snapshot.copy()

# 加上監控功能名稱
clean_sensor_view = clean_sensor_view.merge(
    monitor_function[
        [
            "monitor_id",
            "monitor_name",
            "subgroup",
            "data_source_system",
            "description"
        ]
    ],
    on="monitor_id",
    how="left"
)

# 加上事件資訊
clean_sensor_view = clean_sensor_view.merge(
    clean_event_view[
        [
            "event_id",
            "occurred_at",
            "equipment_code",
            "equipment_name",
            "state_name",
            "severity",
            "root_cause_category"
        ]
    ],
    on="event_id",
    how="left"
)

# 挑選重要欄位
clean_sensor_view = clean_sensor_view[
    [
        "snapshot_id",
        "event_id",
        "occurred_at",
        "equipment_code",
        "equipment_name",
        "state_name",
        "severity",
        "captured_at",
        "monitor_id",
        "monitor_name",
        "parameter_name",
        "actual_value",
        "unit",
        "spec_lower",
        "spec_upper",
        "judgement",
        "source_system",
        "root_cause_category"
    ]
]

clean_sensor_view.head()

,snapshot_id,event_id,occurred_at,equipment_code,equipment_name,state_name,severity,captured_at,monitor_id,monitor_name,parameter_name,actual_value,unit,spec_lower,spec_upper,judgement,source_system,root_cause_category
0,SNP00001-01,EVT00001,2026-03-06 10:23,KB2,KB2,軸承破裂,A,2026-03-06 10:24,MON001,加熱爐在爐時間,effective_heating_time_min,199.0,min,80.0,230.0,正常,MES,設備磨耗
1,SNP00001-02,EVT00001,2026-03-06 10:23,KB2,KB2,軸承破裂,A,2026-03-06 10:25,MON002,加熱爐溫度,furnace_temperature,1217.0,°C,1080.0,1230.0,正常,MES,設備磨耗
2,SNP00001-03,EVT00001,2026-03-06 10:23,KB2,KB2,軸承破裂,A,2026-03-06 10:26,MON003,PI11夾輥作動夾持秒,pi11_clamp_seconds,4.6,sec,3.0,6.0,正常,manual,設備磨耗
3,SNP00001-04,EVT00001,2026-03-06 10:23,KB2,KB2,軸承破裂,A,2026-03-06 10:27,MON004,HRM水量設定,hrm_water_flow,1632.0,L/min,1500.0,1900.0,正常,manual,設備磨耗
4,SNP00001-05,EVT00001,2026-03-06 10:23,KB2,KB2,軸承破裂,A,2026-03-06 10:28,MON009,Guide Roll振動值,guide_roll_vibration,3.7,mm/s,NaN,6.5,正常,MES,設備磨耗


### 查看某一筆事件的感測器判斷

本段說明「查看某一筆事件的感測器判斷」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


In [16]:
event_id = "EVT00001"

clean_sensor_view[
    clean_sensor_view["event_id"] == event_id
][
    [
        "event_id",
        "equipment_code",
        "state_name",
        "monitor_name",
        "parameter_name",
        "actual_value",
        "unit",
        "spec_lower",
        "spec_upper",
        "judgement"
    ]
]

,event_id,equipment_code,state_name,monitor_name,parameter_name,actual_value,unit,spec_lower,spec_upper,judgement
0,EVT00001,KB2,軸承破裂,加熱爐在爐時間,effective_heating_time_min,199.0,min,80.0,230.0,正常
1,EVT00001,KB2,軸承破裂,加熱爐溫度,furnace_temperature,1217.0,°C,1080.0,1230.0,正常
2,EVT00001,KB2,軸承破裂,PI11夾輥作動夾持秒,pi11_clamp_seconds,4.6,sec,3.0,6.0,正常
3,EVT00001,KB2,軸承破裂,HRM水量設定,hrm_water_flow,1632.0,L/min,1500.0,1900.0,正常
4,EVT00001,KB2,軸承破裂,Guide Roll振動值,guide_roll_vibration,3.7,mm/s,NaN,6.5,正常
5,EVT00001,KB2,軸承破裂,通訊訊號狀態,signal_loss_flag,1.0,bool,0.0,0.0,異常


### 說明

這個之後會給 Diagnosis Agent 使用，AI 不是自己亂猜，而是根據感測器數值是否超出標準來判斷。


## 建立「人工輔助知識表」

這一段把原始檔案中比較細的人工經驗整理出來，不是讓 AI 取代人工，而是讓 AI 告訴現場人員：

1. 人工要檢查什麼。
2. 如果異常可以怎麼處理。
3. 要留下什麼佐證。
4. 什麼情況要升級通報。

主要來源包含：

- `20_embedded_hrm_bite_sop`：HRM 頭端咬不入人工檢查表。
- `24_embedded_kb2_notes`：KB2 晃動擠出案例筆記。
- `25_embedded_case_summary`：實際案例摘要。


In [17]:
# =========================================================
# 建立人工輔助知識表 human_assist_view
# 目的：把原始檔中「人工可以檢查什麼、怎麼處理」整理成 Agent 可查詢的資料
# 重點：不把所有異常都套同一套人工確認事項，而是依設備、異常狀況、SOP 步驟與原始案例建立對應資料
# =========================================================

human_assist_records = []


def _assist_evidence(row):
    evidence = []
    if pd.notna(row.get("monitor_name")):
        evidence.append("系統參數截圖 / 感測數值")
    if str(row.get("image_needed", "")).lower() in ["true", "1", "yes", "y"]:
        evidence.append("現場照片")
    if pd.notna(row.get("safety_note")):
        evidence.append("安全確認紀錄")
    if not evidence:
        evidence.append("人工確認備註")
    return "、".join(evidence)


def _assist_escalation(row):
    title = str(row.get("step_title", ""))
    content = str(row.get("step_content", ""))
    text = title + " " + content

    if any(k in text for k in ["安全", "隔離", "停機", "危險"]):
        return "若涉及人員或設備安全風險，立即停機並通知班長。"
    if any(k in text for k in ["軸承", "傳動", "馬達", "電流", "Guide", "TC Roll"]):
        return "若設備機構異常或處置後仍無法復機，通知設備工程師 / 保全單位。"
    if any(k in text for k in ["加熱", "溫度", "在爐", "鋼種", "製程"]):
        return "若製程條件與鋼種標準不一致，通知製程工程師確認。"
    if any(k in text for k in ["訊號", "PLC", "通訊", "sensor", "感測"]):
        return "若訊號異常、資料缺漏或感測值不可信，通知自動化 / 設備單位確認。"
    return "若依此步驟確認後仍無法改善，通知班長並升級給製程或設備工程師。"


# 1. 先從 SOP 步驟建立「每個設備 + 異常狀況」專屬的人工輔助項目
# 這樣即使沒有內嵌案例，也不會每個異常都顯示同一套人工確認事項。
base_sop_for_assist = sop_detail_view if "sop_detail_view" in globals() else clean_sop_view

for _, row in base_sop_for_assist.iterrows():
    check_type = str(row.get("check_type", "")).lower()
    step_title = str(row.get("step_title", ""))
    step_content = str(row.get("step_content", ""))

    # 過濾掉太空泛的根節點，保留人工 / 混合 / 需要照片 / 有安全提醒 / 無監控但需現場判斷的步驟
    is_manual_related = (
        check_type in ["manual", "hybrid"]
        or str(row.get("image_needed", "")).lower() in ["true", "1", "yes", "y"]
        or pd.notna(row.get("safety_note"))
        or pd.isna(row.get("monitor_name"))
    )

    if not is_manual_related:
        continue

    if step_title.strip() in ["根節點", ""]:
        continue

    human_assist_records.append({
        "source_table": "12_sop_step",
        "equipment_code": row.get("equipment_code"),
        "state_keyword": row.get("state_name"),
        "assist_type": row.get("sop_stage", "SOP人工確認"),
        "check_item": step_title,
        "human_action": step_content,
        "related_monitor_id": row.get("monitor_id"),
        "evidence_required": row.get("evidence_required", _assist_evidence(row)),
        "escalation_rule": _assist_escalation(row),
        "source_note": f"來自 {row.get('sop_id', '')}｜{row.get('sop_name', '')}"
    })


# 2. HRM 頭端咬不入原始人工輔助 SOP：只綁定「頭端咬不入」，避免 HRM 其他異常全部套用同一份檢查表
if "20_embedded_hrm_bite_sop" in tables:
    hrm_assist = tables["20_embedded_hrm_bite_sop"].copy()

    for _, row in hrm_assist.iterrows():
        human_assist_records.append({
            "source_table": "20_embedded_hrm_bite_sop",
            "equipment_code": "HRM",
            "state_keyword": "頭端咬不入",
            "assist_type": "原始檔人工檢查與處置",
            "check_item": row.get("check_item"),
            "human_action": row.get("handling_method"),
            "related_monitor_id": row.get("related_monitor_id"),
            "evidence_required": "現場確認紀錄 / 數值截圖 / 必要時拍照",
            "escalation_rule": "若調整後仍無法改善，通知班長、製程工程師或設備工程師。",
            "source_note": row.get("source_note")
        })


# 3. KB2 案例筆記：只綁定晃動擠出 / 連續 COBBLE / Guide / 電流波動等相關狀況
if "24_embedded_kb2_notes" in tables:
    kb2_notes = tables["24_embedded_kb2_notes"].copy()

    for _, row in kb2_notes.iterrows():
        note_type = str(row.get("note_type", ""))
        content = row.get("content")

        if note_type in ["判斷依據", "原因分析", "處理過程", "後續檢討"]:
            human_assist_records.append({
                "source_table": "24_embedded_kb2_notes",
                "equipment_code": "KB2",
                "state_keyword": "晃動擠出/連續COBBLE/Guide/電流波動",
                "assist_type": f"KB2案例{note_type}",
                "check_item": content if note_type in ["判斷依據", "原因分析"] else "依 KB2 案例進行人工確認",
                "human_action": content if note_type in ["處理過程", "後續檢討"] else "比對速度 / 電流趨勢與現場機構狀態",
                "related_monitor_id": "MON015",
                "evidence_required": "速度/電流趨勢截圖、現場檢查紀錄、維修處置紀錄",
                "escalation_rule": "若出現連續 COBBLE、電流大幅波動、Guide 撐開或軸心異常，通知設備工程師。",
                "source_note": row.get("source_note")
            })


# 4. 實際案例摘要：依原始 case 的設備與異常狀況綁定
if "25_embedded_case_summary" in tables:
    case_summary = tables["25_embedded_case_summary"].copy()

    for _, row in case_summary.iterrows():
        human_assist_records.append({
            "source_table": "25_embedded_case_summary",
            "equipment_code": row.get("equipment_code"),
            "state_keyword": row.get("state_name"),
            "assist_type": "歷史案例處置",
            "check_item": row.get("root_cause_or_key_finding"),
            "human_action": row.get("action_taken"),
            "related_monitor_id": None,
            "evidence_required": "歷史案例比對、處置前後紀錄、復機結果",
            "escalation_rule": "若本次異常與歷史案例相似，優先參考過去有效處置；若無效則升級通報。",
            "source_note": row.get("source_note")
        })

human_assist_view = pd.DataFrame(human_assist_records)

# 去除完全重複的項目，避免畫面過長
if not human_assist_view.empty:
    human_assist_view = human_assist_view.drop_duplicates(
        subset=["equipment_code", "state_keyword", "check_item", "human_action"],
        keep="first"
    ).reset_index(drop=True)

print("人工輔助知識表已建立，資料筆數：", len(human_assist_view))
human_assist_view.head(20)


人工輔助知識表已建立，資料筆數： 199


,source_table,equipment_code,state_keyword,assist_type,check_item,human_action,related_monitor_id,evidence_required,escalation_rule,source_note
0,12_sop_step,HRM,打滑,1. 異常確認,判斷是否持續打滑,當支退爐、在軋一支是否打滑,NaN,安全確認紀錄,若依此步驟確認後仍無法改善，通知班長並升級給製程或設備工程師。,來自 SOP001｜HRM打滑
1,12_sop_step,HRM,打滑,1. 異常確認,持續觀察,持續軋延分析打滑鋼種是否異常,NaN,人工確認備註,若製程條件與鋼種標準不一致，通知製程工程師確認。,來自 SOP001｜HRM打滑
2,12_sop_step,HRM,打滑,4. 處置修正,修正加熱條件,依據加熱溫度管制作業,NaN,人工確認備註,若製程條件與鋼種標準不一致，通知製程工程師確認。,來自 SOP001｜HRM打滑
3,12_sop_step,HRM,打滑,3. 原因排查,檢查水量與噴嘴,HRM水量設定與噴嘴阻塞與角度目視檢查,MON004,系統參數截圖 / 感測數值、現場照片,若依此步驟確認後仍無法改善，通知班長並升級給製程或設備工程師。,來自 SOP001｜HRM打滑
4,12_sop_step,HRM,打滑,4. 處置修正,調整夾持秒數,依據鋼種夾持規定秒數或視情況增加秒數,NaN,人工確認備註,若製程條件與鋼種標準不一致，通知製程工程師確認。,來自 SOP001｜HRM打滑
5,12_sop_step,HRM,打滑,3. 原因排查,檢查咬入點,軋輥咬入點痕跡,MON006,系統參數截圖 / 感測數值、現場照片,若依此步驟確認後仍無法改善，通知班長並升級給製程或設備工程師。,來自 SOP001｜HRM打滑
6,12_sop_step,HRM,打滑,4. 處置修正,清理/調整水量,清理噴水環/角度調整/水量依鋼種設定,NaN,現場照片,若製程條件與鋼種標準不一致，通知製程工程師確認。,來自 SOP001｜HRM打滑
7,12_sop_step,HRM,打滑,3. 原因排查,檢查入口檔板,HRM入口檔板作動,MON007,系統參數截圖 / 感測數值,若依此步驟確認後仍無法改善，通知班長並升級給製程或設備工程師。,來自 SOP001｜HRM打滑
8,12_sop_step,HRM,打滑,4. 處置修正,調整高程,調整高程一致,NaN,安全確認紀錄,若依此步驟確認後仍無法改善，通知班長並升級給製程或設備工程師。,來自 SOP001｜HRM打滑
9,12_sop_step,HRM,打滑,1. 異常確認,復機觀察,開四台隧道加熱軋延第二支，優先410鋼種，70%/70%/70%/60%,NaN,人工確認備註,若製程條件與鋼種標準不一致，通知製程工程師確認。,來自 SOP001｜HRM打滑


## 儲存三張乾淨資料表

本段會把前面整理好的乾淨資料表輸出成檔案，方便 Streamlit 系統使用。

主要輸出包含：
1. 異常事件可讀版。
2. SOP 可讀流程表。
3. 感測器判斷可讀表。
4. 人工輔助知識表。


In [18]:
import os

os.makedirs("output", exist_ok=True)

clean_event_view.to_csv("output/clean_event_view.csv", index=False, encoding="utf-8-sig")
clean_sop_view.to_csv("output/clean_sop_view.csv", index=False, encoding="utf-8-sig")
clean_sensor_view.to_csv("output/clean_sensor_view.csv", index=False, encoding="utf-8-sig")

# 新增：強化 SOP 與人工輔助資料表，供 notebook Agent 使用
if "sop_detail_view" in globals():
    sop_detail_view.to_csv("output/sop_detail_view.csv", index=False, encoding="utf-8-sig")

if "human_assist_view" in globals():
    human_assist_view.to_csv("output/human_assist_view.csv", index=False, encoding="utf-8-sig")

print("已輸出資料表：")
print("output/clean_event_view.csv")
print("output/clean_sop_view.csv")
print("output/clean_sensor_view.csv")
if "sop_detail_view" in globals():
    print("output/sop_detail_view.csv")
if "human_assist_view" in globals():
    print("output/human_assist_view.csv")


已輸出資料表：
output/clean_event_view.csv
output/clean_sop_view.csv
output/clean_sensor_view.csv
output/sop_detail_view.csv
output/human_assist_view.csv


### 說明

**原始 Excel**

**→資料可解讀化**

**→clean_event_view、clean_sop_view、clean_sensor_view**


### 說明

本專題第一步先進行資料可解讀化。由於原始資料中大量使用 equipment_id、state_id、sop_id、monitor_id 等代碼，若直接交給 AI 使用，模型無法有效理解設備、異常狀況、SOP 與監控參數之間的關聯。

因此，我們使用 Pandas 將異常事件表、設備表、異常狀況表、SOP 主檔、SOP 步驟表與感測器快照表進行整合，建立三張可讀化資料表：

第一，clean_event_view，用於呈現每筆異常事件的人可讀資訊，包含設備名稱、異常狀況、嚴重度、停機時間、原因分類、處理摘要與對應 SOP。

第二，clean_sop_view，用於呈現 SOP 的完整流程，包含步驟名稱、處理內容、判斷方式、監控項目、標準文字與安全提醒。

第三，clean_sensor_view，用於呈現異常事件發生當下的感測器數值，並與標準上下限進行比對，判斷該參數是否正常。

透過這個步驟，原本難以理解的製造資料被轉換成 AI 可查詢、可推理、可生成摘要的知識基礎。


## 建立知識圖譜 Graph

本段開始把整理後的表格資料轉成知識圖譜。

知識圖譜的目的，是把原本分散在 Excel 的設備、異常、SOP、SOP 步驟、歷史事件、原因與監控項目，用「節點」和「關係」串起來。


### 說明

目標是把你剛剛整理好的三張表：clean_event_view、clean_sop_view、clean_sensor_view→轉成一個「知識圖譜」。

也就是讓資料從表格變成這種關係：

設備→異常狀況→SOP→SOP 步驟→監控項目→歷史異常事件


## 先建立 NetworkX 圖譜

本段建立一個有方向的 NetworkX 圖譜。

因為資料關係具有方向性，例如：
1. 異常事件發生在某台設備。
2. 設備對應某些異常狀況。
3. 異常狀況對應 SOP。
4. SOP 包含多個 SOP 步驟。
5. SOP 步驟可能檢查某個監控項目。


In [19]:
import networkx as nx

G = nx.DiGraph()

print("已建立一個空的有向知識圖譜")

已建立一個空的有向知識圖譜


### 說明

建立一個有方向的圖譜，因為我們的資料關係有方向，例如：

1. 設備 → 會發生 → 異常狀況

2. 異常狀況 → 對應 → SOP

3. SOP → 包含 → SOP 步驟

4. SOP 步驟 → 檢查 → 監控項目

5. 異常事件 → 發生於 → 設備


## 加入設備節點 Equipment

本段會把每一台設備建立成知識圖譜中的節點。

例如 HRM、KB1、KB2、FB、PLC 等設備都會成為 `Equipment` 節點，之後才能查詢某台設備曾經發生哪些異常。


In [20]:
# 建立設備節點
equipment_nodes = clean_event_view[
    ["equipment_id", "equipment_code", "equipment_name"]
].drop_duplicates()

for _, row in equipment_nodes.iterrows():
    node_id = f"Equipment:{row['equipment_id']}"
    G.add_node(
        node_id,
        node_type="Equipment",
        equipment_id=row["equipment_id"],
        equipment_code=row["equipment_code"],
        name=row["equipment_name"]
    )

print("設備節點數量：", len(equipment_nodes))
print("目前圖譜總節點數：", G.number_of_nodes())

設備節點數量： 23
目前圖譜總節點數： 23


### 把每一台設備變成一個節點。

本段說明「把每一台設備變成一個節點。」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


## 加入異常狀況節點 State

本段會把各種異常狀況建立成 `State` 節點。

例如打滑、軸承破裂、訊號 loss、TC Roll 破裂等，都會變成可以被查詢的異常狀況節點。


In [21]:
# 建立異常狀況節點
state_nodes = clean_event_view[
    ["state_id", "state_name", "default_severity"]
].drop_duplicates()

for _, row in state_nodes.iterrows():
    node_id = f"State:{row['state_id']}"
    G.add_node(
        node_id,
        node_type="State",
        state_id=row["state_id"],
        name=row["state_name"],
        default_severity=row["default_severity"]
    )

print("異常狀況節點數量：", len(state_nodes))
print("目前圖譜總節點數：", G.number_of_nodes())

異常狀況節點數量： 60
目前圖譜總節點數： 83


### 它會把「打滑、軸承破裂、訊號 loss」這些異常狀況變成節點。

本段說明「它會把「打滑、軸承破裂、訊號 loss」這些異常狀況變成節點。」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


## 建立設備與異常狀況的關係

本段會建立設備與異常狀況之間的關係。

例如：
1. HRM 可能發生打滑。
2. KB2 可能發生軸承破裂。
3. FB 可能發生 TC Roll 破裂。

這個關係可以支援下拉選單，只顯示該設備可能發生的異常狀況。


In [22]:
# 建立 Equipment -> State 關係
equipment_state_edges = clean_event_view[
    ["equipment_id", "state_id"]
].drop_duplicates()

for _, row in equipment_state_edges.iterrows():
    source = f"Equipment:{row['equipment_id']}"
    target = f"State:{row['state_id']}"

    if source in G and target in G:
        G.add_edge(
            source,
            target,
            relation="HAS_STATE",
            relation_name="設備可能發生此異常"
        )

print("設備－異常關係數量：", len(equipment_state_edges))
print("目前圖譜總邊數：", G.number_of_edges())

設備－異常關係數量： 60
目前圖譜總邊數： 60


### 說明

建立這種關係：

1. HRM → 打滑

2. KB2 → 軸承破裂

3. TMB1 → TC Roll破裂

也就是系統之後可以知道，某台設備可能會出現哪些異常，這是 GraphRAG 的第一層基礎


## 加入 SOP 節點

本段會把每一份 SOP 主流程建立成知識圖譜節點。

例如 HRM 打滑 SOP、KB2 軸承破裂 SOP、FB-TC Roll 破裂 SOP，都會變成 `SOP` 節點。


In [23]:
# 建立 SOP 節點
sop_nodes = clean_sop_view[
    ["sop_id", "sop_name", "sop_desc", "owner_role", "status", "version"]
].drop_duplicates()

for _, row in sop_nodes.iterrows():
    node_id = f"SOP:{row['sop_id']}"
    G.add_node(
        node_id,
        node_type="SOP",
        sop_id=row["sop_id"],
        name=row["sop_name"],
        description=row["sop_desc"],
        owner_role=row["owner_role"],
        status=row["status"],
        version=row["version"]
    )

print("SOP 節點數量：", len(sop_nodes))
print("目前圖譜總節點數：", G.number_of_nodes())

SOP 節點數量： 21
目前圖譜總節點數： 104


### 它把每一份 SOP 變成節點。

本段說明「它把每一份 SOP 變成節點。」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


## 建立異常狀況與 SOP 的關係

本段會建立異常狀況與 SOP 的對應關係。

也就是讓系統知道：某台設備發生某種異常時，應該查詢哪一份 SOP。


In [24]:
# 建立 State -> SOP 關係
# 注意：clean_sop_view 目前沒有 state_id，所以改從 sop_main 取得 state_id 與 sop_id 的對應關係

state_sop_edges = sop_main[
    ["state_id", "sop_id"]
].dropna().drop_duplicates()

count = 0

for _, row in state_sop_edges.iterrows():
    source = f"State:{row['state_id']}"
    target = f"SOP:{row['sop_id']}"

    if source in G and target in G:
        G.add_edge(
            source,
            target,
            relation="HAS_SOP",
            relation_name="異常狀況對應處理SOP"
        )
        count += 1

print("異常－SOP 關係數量：", count)
print("目前圖譜總邊數：", G.number_of_edges())

異常－SOP 關係數量： 21
目前圖譜總邊數： 81


### 說明

建立：

1. 打滑 → HRM打滑 SOP

2. 軸承破裂 → KB2軸承破裂異常處理 SOP

也就是之後使用者問：HRM 發生打滑怎麼辦？→系統就可以順著圖譜找到 SOP。


## 加入 SOP Step 節點

本段會把 SOP 中的每一個步驟建立成 `SOPStep` 節點。

這樣系統不只知道要用哪份 SOP，也能知道 SOP 裡面每一步要做什麼。


In [25]:
# 建立 SOP Step 節點
sop_step_nodes = clean_sop_view[
    [
        "step_id",
        "sop_id",
        "step_title",
        "step_content",
        "sort_order",
        "branch_label",
        "check_type",
        "standard_text",
        "safety_note",
        "image_needed"
    ]
].drop_duplicates()

for _, row in sop_step_nodes.iterrows():
    node_id = f"SOPStep:{row['step_id']}"
    G.add_node(
        node_id,
        node_type="SOPStep",
        step_id=row["step_id"],
        sop_id=row["sop_id"],
        name=row["step_title"],
        content=row["step_content"],
        sort_order=row["sort_order"],
        branch_label=row["branch_label"],
        check_type=row["check_type"],
        standard_text=row["standard_text"],
        safety_note=row["safety_note"],
        image_needed=row["image_needed"]
    )

print("SOP 步驟節點數量：", len(sop_step_nodes))
print("目前圖譜總節點數：", G.number_of_nodes())

SOP 步驟節點數量： 169
目前圖譜總節點數： 273


### 說明

把 SOP 的每一個步驟都變成節點，這樣之後 AI 才能不只知道「哪份 SOP」，還能知道「具體要做哪些步驟」。


## 建立 SOP 與 SOP Step 的關係

本段會建立 SOP 與 SOP 步驟之間的包含關係。

例如一份 SOP 可能包含：
1. 異常確認。
2. 安全隔離。
3. 原因排查。
4. 執行處置。
5. 復機確認與紀錄回饋。


In [26]:
# 建立 SOP -> SOPStep 關係
sop_step_edges = clean_sop_view[
    ["sop_id", "step_id", "sort_order"]
].drop_duplicates()

for _, row in sop_step_edges.iterrows():
    source = f"SOP:{row['sop_id']}"
    target = f"SOPStep:{row['step_id']}"

    if source in G and target in G:
        G.add_edge(
            source,
            target,
            relation="HAS_STEP",
            relation_name="SOP包含此步驟",
            sort_order=row["sort_order"]
        )

print("SOP－步驟關係數量：", len(sop_step_edges))
print("目前圖譜總邊數：", G.number_of_edges())

SOP－步驟關係數量： 169
目前圖譜總邊數： 250


### 說明

例如：建立HRM打滑 SOP

步驟 1：確認是否持續打滑

步驟 2：檢查加熱條件

步驟 3：檢查水量

步驟 4：檢查軋輥


## 建立 SOP Step 之間的父子流程關係

本段會建立 SOP 步驟之間的先後與分支關係。

這是後續產生 SOP 樹狀圖與逐步互動導覽的基礎。


In [27]:
# 建立 SOPStep -> SOPStep 父子流程關係
step_parent_edges = clean_sop_view[
    ["step_id", "parent_step_id", "branch_label"]
].drop_duplicates()

count = 0

for _, row in step_parent_edges.iterrows():
    child = f"SOPStep:{row['step_id']}"

    if pd.notna(row["parent_step_id"]):
        parent = f"SOPStep:{row['parent_step_id']}"

        if parent in G and child in G:
            G.add_edge(
                parent,
                child,
                relation="NEXT_STEP",
                relation_name="下一個SOP步驟",
                branch_label=row["branch_label"]
            )
            count += 1

print("SOP 步驟父子流程關係數量：", count)
print("目前圖譜總邊數：", G.number_of_edges())

SOP 步驟父子流程關係數量： 148
目前圖譜總邊數： 398


### 說明

SOP 不是單純清單，而是有分支流程：

是否持續打滑？

→是 → 檢查加熱條件
→否 → 持續觀察

這一步會讓你的系統可以呈現「流程邏輯」，而不是只呈現一堆步驟。


## 加入監控項目 Monitor 節點

本段會把監控項目建立成 `Monitor` 節點。

例如加熱爐溫度、在爐時間、PI11 夾持秒數、水量、電流或通訊狀態等，都可以被當成監控或檢查項目。


In [28]:
# 建立監控項目節點
monitor_nodes = clean_sensor_view[
    ["monitor_id", "monitor_name", "source_system"]
].drop_duplicates()

for _, row in monitor_nodes.iterrows():
    node_id = f"Monitor:{row['monitor_id']}"
    G.add_node(
        node_id,
        node_type="Monitor",
        monitor_id=row["monitor_id"],
        name=row["monitor_name"],
        source_system=row["source_system"]
    )

print("監控項目節點數量：", len(monitor_nodes))
print("目前圖譜總節點數：", G.number_of_nodes())

監控項目節點數量： 30
目前圖譜總節點數： 283


### 說明

監控項目變成節點，這些之後會讓 Diagnosis Agent 判斷：

哪些感測器資料異常？


## 建立 SOP Step 與 Monitor 的關係

本段會建立 SOP 步驟與監控項目之間的關係。

如果某個 SOP 步驟可以透過感測器或參數輔助判斷，系統就會建立 `CHECKS` 關係。


In [29]:
# 建立 SOPStep -> Monitor 關係
step_monitor_edges = clean_sop_view[
    ["step_id", "monitor_id"]
].drop_duplicates()

count = 0

for _, row in step_monitor_edges.iterrows():
    if pd.notna(row["monitor_id"]):
        source = f"SOPStep:{row['step_id']}"
        target = f"Monitor:{row['monitor_id']}"

        if source in G and target in G:
            G.add_edge(
                source,
                target,
                relation="CHECKS",
                relation_name="此SOP步驟檢查此監控項目"
            )
            count += 1

print("SOP步驟－監控項目關係數量：", count)
print("目前圖譜總邊數：", G.number_of_edges())

SOP步驟－監控項目關係數量： 59
目前圖譜總邊數： 457


### 專題中「AI 不是只看文字，而是可以結合感測器資料判斷」的重點

本段說明「專題中「AI 不是只看文字，而是可以結合感測器資料判斷」的重點」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


## 加入歷史異常事件 Event 節點

本段會把每一筆歷史異常事件建立成 `Event` 節點。

這些節點記錄了事件發生時間、設備、異常狀況、嚴重度、停機時間、原因與處置方式，是後續診斷與相似案例推薦的重要來源。


In [30]:
# 建立歷史異常事件節點
event_nodes = clean_event_view[
    [
        "event_id",
        "occurred_at",
        "equipment_id",
        "state_id",
        "equipment_code",
        "equipment_name",
        "state_name",
        "severity",
        "downtime_min",
        "root_cause_category",
        "cause_summary",
        "action_summary",
        "close_result"
    ]
].drop_duplicates()

for _, row in event_nodes.iterrows():
    node_id = f"Event:{row['event_id']}"
    G.add_node(
        node_id,
        node_type="Event",
        event_id=row["event_id"],
        occurred_at=str(row["occurred_at"]),
        equipment_code=row["equipment_code"],
        equipment_name=row["equipment_name"],
        state_name=row["state_name"],
        severity=row["severity"],
        downtime_min=row["downtime_min"],
        root_cause_category=row["root_cause_category"],
        cause_summary=row["cause_summary"],
        action_summary=row["action_summary"],
        close_result=row["close_result"]
    )

print("歷史異常事件節點數量：", len(event_nodes))
print("目前圖譜總節點數：", G.number_of_nodes())

歷史異常事件節點數量： 362
目前圖譜總節點數： 645


### 說明

把每一筆異常事件變成節點，每個事件裡面會記錄：

1. 設備

2. 異常狀況

3. 嚴重度

4. 停機時間

5. 原因摘要

6. 處理摘要

7. 結案結果

這就是之後做「相似案例推薦」的資料來源。


## 建立 Event 與 Equipment / State / SOP 的關係

本段會把事件與設備、異常狀況、SOP 連起來。

例如某筆事件發生在 KB2、異常狀況是軸承破裂、使用 SOP011，這些都會成為圖譜中的關係。


In [31]:
# Event -> Equipment / State / SOP 關係
count_equipment = 0
count_state = 0
count_sop = 0

for _, row in clean_event_view.iterrows():
    event_node = f"Event:{row['event_id']}"

    equipment_node = f"Equipment:{row['equipment_id']}"
    state_node = f"State:{row['state_id']}"
    sop_node = f"SOP:{row['sop_id']}"

    if event_node in G and equipment_node in G:
        G.add_edge(
            event_node,
            equipment_node,
            relation="OCCURRED_ON",
            relation_name="事件發生於此設備"
        )
        count_equipment += 1

    if event_node in G and state_node in G:
        G.add_edge(
            event_node,
            state_node,
            relation="HAS_ABNORMAL_STATE",
            relation_name="事件屬於此異常狀況"
        )
        count_state += 1

    if pd.notna(row["sop_id"]) and event_node in G and sop_node in G:
        G.add_edge(
            event_node,
            sop_node,
            relation="USED_SOP",
            relation_name="事件使用此SOP"
        )
        count_sop += 1

print("Event -> Equipment 關係數量：", count_equipment)
print("Event -> State 關係數量：", count_state)
print("Event -> SOP 關係數量：", count_sop)
print("目前圖譜總邊數：", G.number_of_edges())

Event -> Equipment 關係數量： 362
Event -> State 關係數量： 362
Event -> SOP 關係數量： 147
目前圖譜總邊數： 1328


### 說明

例如建立：

1. EVT00001 → 發生於 → KB2

2. EVT00001 → 屬於 → 軸承破裂

3. EVT00001 → 使用 → KB2 軸承破裂 SOP

這樣之後 GraphRAG 可以從事件往外找：

事件 → 設備 → 異常 → SOP → 步驟 → 監控項目


## 加入 Cause 原因節點

本段會把歷史事件中的原因分類建立成 `Cause` 節點。

例如設備磨耗、參數錯誤、感測器異常、加熱條件異常等，之後可以用來統計常見原因。


In [32]:
# 建立原因分類節點
cause_nodes = clean_event_view[
    ["root_cause_category"]
].drop_duplicates()

cause_nodes = cause_nodes[cause_nodes["root_cause_category"].notna()]

for _, row in cause_nodes.iterrows():
    cause_name = row["root_cause_category"]
    node_id = f"Cause:{cause_name}"

    G.add_node(
        node_id,
        node_type="Cause",
        name=cause_name
    )

print("原因分類節點數量：", len(cause_nodes))
print("目前圖譜總節點數：", G.number_of_nodes())

原因分類節點數量： 9
目前圖譜總節點數： 654


## 建立 Event 與 Cause 的關係

本段會把每一筆歷史事件連到它的原因分類。

這樣 Diagnosis Agent 就能根據過去事件的原因分布，推估新事件可能的原因。


In [33]:
# 建立 Event -> Cause 關係
count = 0

for _, row in clean_event_view.iterrows():
    if pd.notna(row["root_cause_category"]):
        event_node = f"Event:{row['event_id']}"
        cause_node = f"Cause:{row['root_cause_category']}"

        if event_node in G and cause_node in G:
            G.add_edge(
                event_node,
                cause_node,
                relation="HAS_CAUSE",
                relation_name="事件原因分類"
            )
            count += 1

print("Event－Cause 關係數量：", count)
print("目前圖譜總邊數：", G.number_of_edges())

Event－Cause 關係數量： 362
目前圖譜總邊數： 1690


### 說明

之後 Diagnosis Agent 可以用這些歷史案例的原因來推測新事件的可能原因。


## 檢查整個知識圖譜規模

本段會檢查知識圖譜中目前有多少節點與關係。

這可以用來確認資料是否成功轉成圖譜，也能觀察哪些資料類型最多。


In [34]:
print("知識圖譜建立完成")
print("總節點數：", G.number_of_nodes())
print("總關係數：", G.number_of_edges())

# 統計各類節點數量
node_type_count = {}

for _, data in G.nodes(data=True):
    node_type = data.get("node_type", "Unknown")
    node_type_count[node_type] = node_type_count.get(node_type, 0) + 1

pd.DataFrame(
    [{"node_type": k, "count": v} for k, v in node_type_count.items()]
).sort_values("count", ascending=False)

知識圖譜建立完成
總節點數： 654
總關係數： 1690


,node_type,count
5,Event,362
3,SOPStep,169
1,State,60
0,Equipment,23
2,SOP,21
4,Monitor,10
6,Cause,9


## 統計各類關係數量

本段會統計知識圖譜中的關係類型。

例如事件發生在哪個設備、事件有什麼異常狀況、SOP 包含哪些步驟、步驟之間如何連接等。


In [35]:
edge_type_count = {}

for source, target, data in G.edges(data=True):
    relation = data.get("relation", "Unknown")
    edge_type_count[relation] = edge_type_count.get(relation, 0) + 1

pd.DataFrame(
    [{"relation": k, "count": v} for k, v in edge_type_count.items()]
).sort_values("count", ascending=False)

,relation,count
6,HAS_ABNORMAL_STATE,362
5,OCCURRED_ON,362
8,HAS_CAUSE,362
2,HAS_STEP,169
3,NEXT_STEP,148
7,USED_SOP,147
0,HAS_STATE,60
4,CHECKS,59
1,HAS_SOP,21


## 把圖譜轉成 nodes / edges 表格

本段會把知識圖譜轉成兩張表：`nodes` 和 `edges`。

其中 `nodes` 代表節點，`edges` 代表節點之間的關係。這兩張表會提供給 Streamlit 頁面呈現知識圖譜統計。


In [36]:
# 輸出 nodes
nodes_data = []

for node_id, data in G.nodes(data=True):
    row = {"node_id": node_id}
    row.update(data)
    nodes_data.append(row)

graph_nodes = pd.DataFrame(nodes_data)

# 輸出 edges
edges_data = []

for source, target, data in G.edges(data=True):
    row = {
        "source": source,
        "target": target
    }
    row.update(data)
    edges_data.append(row)

graph_edges = pd.DataFrame(edges_data)

print("graph_nodes 筆數：", len(graph_nodes))
print("graph_edges 筆數：", len(graph_edges))

graph_nodes.head()

graph_nodes 筆數： 654
graph_edges 筆數： 1690


,node_id,node_type,equipment_id,equipment_code,name,state_id,default_severity,sop_id,description,owner_role,...,event_id,occurred_at,equipment_name,state_name,severity,downtime_min,root_cause_category,cause_summary,action_summary,close_result
0,Equipment:EQ006,Equipment,EQ006,KB2,KB2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Equipment:EQ011,Equipment,EQ011,TMB1,TMB1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Equipment:EQ010,Equipment,EQ010,FB,FB,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Equipment:EQ021,Equipment,EQ021,PLC,PLC,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Equipment:EQ001,Equipment,EQ001,HRM,HRM,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 因為之後 Streamlit 會讀這兩張表。

本段說明「因為之後 Streamlit 會讀這兩張表。」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


## 查看 edges 表

本段用來查看知識圖譜的邊資料。

每一列 edge 都代表兩個節點之間的一條關係，例如事件發生在設備、SOP 包含步驟、步驟檢查監控項目等。


In [37]:
graph_edges.head(20)

,source,target,relation,relation_name,sort_order,branch_label
0,Equipment:EQ006,State:ST011,HAS_STATE,設備可能發生此異常,NaN,NaN
1,Equipment:EQ006,State:ST050,HAS_STATE,設備可能發生此異常,NaN,NaN
2,Equipment:EQ006,State:ST012,HAS_STATE,設備可能發生此異常,NaN,NaN
3,Equipment:EQ006,State:ST051,HAS_STATE,設備可能發生此異常,NaN,NaN
4,Equipment:EQ006,State:ST010,HAS_STATE,設備可能發生此異常,NaN,NaN
5,Equipment:EQ006,State:ST064,HAS_STATE,設備可能發生此異常,NaN,NaN
6,Equipment:EQ011,State:ST023,HAS_STATE,設備可能發生此異常,NaN,NaN
7,Equipment:EQ011,State:ST024,HAS_STATE,設備可能發生此異常,NaN,NaN
8,Equipment:EQ011,State:ST053,HAS_STATE,設備可能發生此異常,NaN,NaN
9,Equipment:EQ011,State:ST027,HAS_STATE,設備可能發生此異常,NaN,NaN


In [38]:
graph_edges["relation"].value_counts()

,count
relation,
HAS_ABNORMAL_STATE,362
OCCURRED_ON,362
HAS_CAUSE,362
HAS_STEP,169
NEXT_STEP,148
USED_SOP,147
HAS_STATE,60
CHECKS,59
HAS_SOP,21


## 儲存圖譜資料

本段會把 `nodes` 和 `edges` 輸出成檔案，讓後續 Streamlit 可以直接讀取並呈現圖譜統計。


In [39]:
graph_nodes.to_csv("output/graph_nodes.csv", index=False, encoding="utf-8-sig")
graph_edges.to_csv("output/graph_edges.csv", index=False, encoding="utf-8-sig")

print("已輸出知識圖譜資料：")
print("output/graph_nodes.csv")
print("output/graph_edges.csv")

已輸出知識圖譜資料：
output/graph_nodes.csv
output/graph_edges.csv


## 做一個簡單的 GraphRAG 查詢函式

本段開始建立 GraphRAG 查詢函式。

GraphRAG 的概念是先從知識圖譜取回相關資料，再把這些資料提供給生成式 AI 產生回答，降低 AI 憑空亂猜的風險。


### 說明

這一步開始進入 GraphRAG。

先做一個函式：輸入事件 ID，系統幫你找出與它相關的資訊。


In [40]:
def graph_context_by_event(event_id, max_steps=2):
    """
    根據 event_id 從知識圖譜找相關上下文。
    max_steps=2 代表往外找兩層關係。
    """

    start_node = f"Event:{event_id}"

    if start_node not in G:
        return f"找不到事件：{event_id}"

    context = []

    event_data = G.nodes[start_node]
    context.append("【異常事件基本資料】")
    context.append(f"事件編號：{event_data.get('event_id')}")
    context.append(f"發生時間：{event_data.get('occurred_at')}")
    context.append(f"設備：{event_data.get('equipment_code')} {event_data.get('equipment_name')}")
    context.append(f"異常狀況：{event_data.get('state_name')}")
    context.append(f"嚴重度：{event_data.get('severity')}")
    context.append(f"停機時間：{event_data.get('downtime_min')} 分鐘")
    context.append(f"原因分類：{event_data.get('root_cause_category')}")
    context.append(f"原因摘要：{event_data.get('cause_summary')}")
    context.append(f"處理摘要：{event_data.get('action_summary')}")
    context.append("")

    # 找出直接鄰居
    context.append("【知識圖譜關聯資料】")

    for neighbor in G.successors(start_node):
        edge_data = G.get_edge_data(start_node, neighbor)
        node_data = G.nodes[neighbor]

        context.append(
            f"- {edge_data.get('relation_name')}：{node_data.get('name', neighbor)}"
        )

        # 再往下一層找
        for second_neighbor in G.successors(neighbor):
            second_edge_data = G.get_edge_data(neighbor, second_neighbor)
            second_node_data = G.nodes[second_neighbor]

            context.append(
                f"  - {second_edge_data.get('relation_name')}：{second_node_data.get('name', second_neighbor)}"
            )

    return "\n".join(context)

### 說明

這個函式就是 GraphRAG 的雛形。

它不是直接叫 AI 回答，而是先根據事件 ID 從圖譜找資料：

事件→ 設備→ 異常狀況→ SOP→ 原因分類

之後這些資料會丟給 LLM 產生回答。


## 測試 GraphRAG 查詢

本段會測試 GraphRAG 查詢結果。

目標是確認系統能根據事件 ID 或設備與異常狀況，找到相關設備、SOP、原因與歷史案例。


In [41]:
print(graph_context_by_event("EVT00001"))

【異常事件基本資料】
事件編號：EVT00001
發生時間：2026-03-06 10:23
設備：KB2 KB2
異常狀況：軸承破裂
嚴重度：A
停機時間：70 分鐘
原因分類：設備磨耗
原因摘要：KB2 軸承破裂，初判原因：設備磨耗
處理摘要：更換磨耗零件並復測

【知識圖譜關聯資料】
- 事件發生於此設備：KB2
  - 設備可能發生此異常：軸承破裂
  - 設備可能發生此異常：軋輥破裂
  - 設備可能發生此異常：傳動軸斷裂
  - 設備可能發生此異常：Guide Roll破裂
  - 設備可能發生此異常：lever斷裂
  - 設備可能發生此異常：晃動擠出/連續COBBLE
- 事件屬於此異常狀況：軸承破裂
  - 異常狀況對應處理SOP：KB2-軸承破裂異常處理
- 事件使用此SOP：KB2-軸承破裂異常處理
  - SOP包含此步驟：異常確認
  - SOP包含此步驟：安全隔離
  - SOP包含此步驟：持續觀察
  - SOP包含此步驟：檢查常見原因
  - SOP包含此步驟：執行處置
  - SOP包含此步驟：升級處理
  - SOP包含此步驟：結案與回饋
- 事件原因分類：設備磨耗


### 說明

系統會先從知識圖譜中取回相關上下文，再交給生成式 AI 產生診斷摘要，因此不是單純讓 AI 憑空回答。


## 做依照設備與異常狀況查詢的 GraphRAG 函式

本段建立更貼近使用情境的查詢函式。

使用者通常不會輸入事件 ID，而是會選擇設備與異常狀況，因此這裡會讓系統能透過設備與異常狀況找到對應資料。


### 說明

實際展示時，使用者可能會輸入：

設備：HRM

異常狀況：打滑

所以我們再做一個查詢函式。


In [42]:
def graph_context_by_equipment_state(equipment_code=None, state_name=None, top_k_events=5):
    """
    根據設備代碼與異常狀況，從圖譜找 SOP、SOP 步驟、歷史案例。
    """

    matched_events = clean_event_view.copy()

    if equipment_code:
        matched_events = matched_events[
            matched_events["equipment_code"].astype(str).str.contains(equipment_code, case=False, na=False)
        ]

    if state_name:
        matched_events = matched_events[
            matched_events["state_name"].astype(str).str.contains(state_name, case=False, na=False)
        ]

    if matched_events.empty:
        return "找不到符合條件的歷史事件。"

    # 取停機時間較短且已結案的案例，當作較好的參考
    case_events = matched_events.sort_values("downtime_min", ascending=True).head(top_k_events)

    context = []
    context.append("【查詢條件】")
    context.append(f"設備：{equipment_code}")
    context.append(f"異常狀況：{state_name}")
    context.append("")

    # 找 SOP
    sop_candidates = matched_events[
        ["sop_id", "sop_name", "sop_desc"]
    ].dropna().drop_duplicates()

    context.append("【可能對應 SOP】")

    if sop_candidates.empty:
        context.append("未找到明確對應 SOP。")
    else:
        for _, row in sop_candidates.iterrows():
            context.append(f"- {row['sop_id']}：{row['sop_name']}，說明：{row['sop_desc']}")

            steps = clean_sop_view[
                clean_sop_view["sop_id"] == row["sop_id"]
            ].sort_values("sort_order")

            for _, step in steps.iterrows():
                context.append(
                    f"  {step['sort_order']}. {step['step_title']}：{step['step_content']}"
                )

                if pd.notna(step["monitor_name"]):
                    context.append(
                        f"     需檢查監控項目：{step['monitor_name']}；標準：{step['standard_text']}"
                    )

                if pd.notna(step["safety_note"]):
                    context.append(
                        f"     安全提醒：{step['safety_note']}"
                    )

    context.append("")
    context.append("【相似歷史案例】")

    for _, row in case_events.iterrows():
        context.append(
            f"- {row['event_id']}｜設備：{row['equipment_code']}｜異常：{row['state_name']}｜"
            f"嚴重度：{row['severity']}｜停機：{row['downtime_min']}分鐘｜"
            f"原因：{row['root_cause_category']}｜處理：{row['action_summary']}"
        )

    return "\n".join(context)

## 測試設備 + 異常查詢

本段會測試設備與異常狀況查詢。

例如輸入 HRM 與打滑，系統應該能取回相關 SOP、歷史案例與可能原因。


In [43]:
clean_event_view[
    ["equipment_code", "state_name"]
].drop_duplicates().head(30)

,equipment_code,state_name
0,KB2,軸承破裂
1,TMB1,TC Roll破裂
2,TMB1,Guide Roll破裂
3,FB,導器磨溝
4,PLC,通訊中斷
5,HRM,軋輥打滑
6,KB2,軋輥破裂
7,KB1,傳動軸斷裂
8,KB1,Guide Roll破裂
11,TMB2,軋輥破裂


In [44]:
print(graph_context_by_equipment_state(equipment_code="HRM", state_name="打滑"))

【查詢條件】
設備：HRM
異常狀況：打滑

【可能對應 SOP】
- SOP001：HRM打滑，說明：HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷
  1. 根節點：HRM打滑
     安全提醒：先確認現場安全
  1. 判斷是否持續打滑：當支退爐、在軋一支是否打滑
     安全提醒：避免人員靠近入料區
  1. 確認加熱條件：確認加熱爐時間與溫度作業
     需檢查監控項目：加熱爐在爐時間；標準：依鋼種有效加熱時間±30分鐘，並確認爐溫
  1. 檢查夾輥：PI11夾輥作動夾持秒
     需檢查監控項目：PI11夾輥作動夾持秒；標準：3~6秒
  1. 檢查水量與噴嘴：HRM水量設定與噴嘴阻塞與角度目視檢查
     需檢查監控項目：HRM水量設定；標準：水量1500~1900 L/min，噴嘴角度正常
  1. 結案：持續軋延觀察
  1. 檢查入口檔板：HRM入口檔板作動
     需檢查監控項目：入口檔板作動；標準：檔板作動正常且高程一致
  1. 檢查咬入點：軋輥咬入點痕跡
     需檢查監控項目：軋輥咬入點痕跡；標準：咬入點痕跡應正常且無明顯偏移
  1. 復機觀察：開四台隧道加熱軋延第二支，優先410鋼種，70%/70%/70%/60%
  2. 調整夾持秒數：依據鋼種夾持規定秒數或視情況增加秒數
  2. 持續觀察：持續軋延分析打滑鋼種是否異常
  2. 修正加熱條件：依據加熱溫度管制作業
  2. 調整高程：調整高程一致
     安全提醒：注意夾捲危害
  2. 清理/調整水量：清理噴水環/角度調整/水量依鋼種設定
  2. 換輥確認：換輥&確認輥型&材質
  2. 再次換輥確認：換輥&確認輥型&材質

【相似歷史案例】
- EVT00255｜設備：HRM｜異常：軋輥打滑｜嚴重度：B｜停機：4分鐘｜原因：人工操作疏失｜處理：重新教育與增加防呆檢核
- EVT00013｜設備：HRM｜異常：打滑｜嚴重度：B｜停機：19分鐘｜原因：原料/鋼種條件差異｜處理：確認工單與鋼種條件，調整加熱/速度設定
- EVT00296｜設備：HRM｜異常：軋輥打滑｜嚴重度：B｜停機：24分鐘｜原因：人工操作疏失｜處理：重新教育與增加防呆檢核
- EVT00058｜設備：HRM｜異常：軋輥打滑｜嚴重度：B｜停機：33分鐘｜原因：人工操

## 把 GraphRAG 查詢結果整理成 LLM Prompt

本段會把 GraphRAG 取回的資料整理成 LLM 可以使用的 Prompt。

Prompt 中會明確限制模型只能根據取回的資料回答，避免生成式 AI 產生不符合資料的內容。


In [45]:
def build_diagnosis_prompt(equipment_code, state_name):
    """
    根據 GraphRAG 查詢結果，建立給 LLM 的診斷提示詞。
    """

    retrieved_context = graph_context_by_equipment_state(
        equipment_code=equipment_code,
        state_name=state_name,
        top_k_events=5
    )

    prompt = f"""
你是一位製造現場設備異常診斷助理。
請根據下方 GraphRAG 知識圖譜查詢結果，產生一份工程師可讀的異常診斷摘要。

請務必遵守：
1. 只能根據提供的資料回答，不要憑空捏造。
2. 如果資料不足，請明確說明「目前資料不足」。
3. 請用繁體中文回答。
4. 請分成以下段落：
   - 異常概況
   - 可能原因
   - 建議檢查順序
   - 對應 SOP
   - 相似歷史案例
   - 建議通報內容

【GraphRAG 查詢結果】
{retrieved_context}

請產生診斷摘要：
"""

    return prompt

## 測試 Prompt 長什麼樣子

本段會查看組合後的 Prompt 內容。

這可以確認資料是否完整放入 Prompt，也可以檢查生成式 AI 的回答範圍是否被清楚限制。


In [46]:
prompt = build_diagnosis_prompt("HRM", "打滑")
print(prompt)


你是一位製造現場設備異常診斷助理。
請根據下方 GraphRAG 知識圖譜查詢結果，產生一份工程師可讀的異常診斷摘要。

請務必遵守：
1. 只能根據提供的資料回答，不要憑空捏造。
2. 如果資料不足，請明確說明「目前資料不足」。
3. 請用繁體中文回答。
4. 請分成以下段落：
   - 異常概況
   - 可能原因
   - 建議檢查順序
   - 對應 SOP
   - 相似歷史案例
   - 建議通報內容

【GraphRAG 查詢結果】
【查詢條件】
設備：HRM
異常狀況：打滑

【可能對應 SOP】
- SOP001：HRM打滑，說明：HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷
  1. 根節點：HRM打滑
     安全提醒：先確認現場安全
  1. 判斷是否持續打滑：當支退爐、在軋一支是否打滑
     安全提醒：避免人員靠近入料區
  1. 確認加熱條件：確認加熱爐時間與溫度作業
     需檢查監控項目：加熱爐在爐時間；標準：依鋼種有效加熱時間±30分鐘，並確認爐溫
  1. 檢查夾輥：PI11夾輥作動夾持秒
     需檢查監控項目：PI11夾輥作動夾持秒；標準：3~6秒
  1. 檢查水量與噴嘴：HRM水量設定與噴嘴阻塞與角度目視檢查
     需檢查監控項目：HRM水量設定；標準：水量1500~1900 L/min，噴嘴角度正常
  1. 結案：持續軋延觀察
  1. 檢查入口檔板：HRM入口檔板作動
     需檢查監控項目：入口檔板作動；標準：檔板作動正常且高程一致
  1. 檢查咬入點：軋輥咬入點痕跡
     需檢查監控項目：軋輥咬入點痕跡；標準：咬入點痕跡應正常且無明顯偏移
  1. 復機觀察：開四台隧道加熱軋延第二支，優先410鋼種，70%/70%/70%/60%
  2. 調整夾持秒數：依據鋼種夾持規定秒數或視情況增加秒數
  2. 持續觀察：持續軋延分析打滑鋼種是否異常
  2. 修正加熱條件：依據加熱溫度管制作業
  2. 調整高程：調整高程一致
     安全提醒：注意夾捲危害
  2. 清理/調整水量：清理噴水環/角度調整/水量依鋼種設定
  2. 換輥確認：換輥&確認輥型&材質
  2. 再次換輥確認：換輥&確認輥型&材質

【相似歷史案例】
- EVT00255｜設備：HRM｜異常：軋

### 說明

系統先透過 GraphRAG 從知識圖譜取回 SOP、監控項目與相似歷史案例，再將查詢結果組合成受限制的 Prompt，要求 LLM 僅能根據資料生成診斷摘要，以降低幻覺風險。


## 先做一個不用 API 的「Demo 版 AI 摘要」

本段會先建立不需要外部 API 的示範版 AI 摘要。

此版本會使用規則式邏輯整理診斷摘要，適合課堂展示與避免 API 額度問題。


In [47]:
def demo_ai_summary(equipment_code, state_name):
    """
    不使用 API 的 Demo 版摘要。
    這不是真正 LLM，但可以先模擬生成式 AI 最終輸出的格式。
    """

    matched_events = clean_event_view.copy()

    matched_events = matched_events[
        matched_events["equipment_code"].astype(str).str.contains(equipment_code, case=False, na=False)
        &
        matched_events["state_name"].astype(str).str.contains(state_name, case=False, na=False)
    ]

    if matched_events.empty:
        return "目前資料不足，找不到符合此設備與異常狀況的歷史資料。"

    sop_candidates = matched_events[
        ["sop_id", "sop_name", "sop_desc"]
    ].dropna().drop_duplicates()

    cause_counts = matched_events["root_cause_category"].value_counts().head(3)
    avg_downtime = matched_events["downtime_min"].mean()
    top_cases = matched_events.sort_values("downtime_min").head(3)

    summary = []

    summary.append("【異常概況】")
    summary.append(
        f"本次查詢的設備為 {equipment_code}，異常狀況為 {state_name}。"
        f"系統在歷史資料中找到 {len(matched_events)} 筆相似事件，"
        f"平均停機時間約為 {avg_downtime:.1f} 分鐘。"
    )
    summary.append("")

    summary.append("【可能原因】")
    if len(cause_counts) > 0:
        for cause, count in cause_counts.items():
            summary.append(f"- {cause}：歷史資料中出現 {count} 次。")
    else:
        summary.append("- 目前資料中沒有足夠的原因分類資訊。")
    summary.append("")

    summary.append("【對應 SOP】")
    if sop_candidates.empty:
        summary.append("- 目前沒有找到明確對應的 SOP。")
    else:
        for _, row in sop_candidates.iterrows():
            summary.append(f"- {row['sop_id']}：{row['sop_name']}。{row['sop_desc']}")

            steps = clean_sop_view[
                clean_sop_view["sop_id"] == row["sop_id"]
            ].sort_values("sort_order")

            for _, step in steps.head(5).iterrows():
                summary.append(f"  {step['sort_order']}. {step['step_title']}：{step['step_content']}")
    summary.append("")

    summary.append("【相似歷史案例】")
    for _, row in top_cases.iterrows():
        summary.append(
            f"- {row['event_id']}：停機 {row['downtime_min']} 分鐘，"
            f"原因為 {row['root_cause_category']}，處理方式為：{row['action_summary']}"
        )
    summary.append("")

    summary.append("【建議通報內容】")
    summary.append(
        f"設備 {equipment_code} 發生 {state_name} 異常。"
        "建議依系統查得之 SOP 進行初步檢查，並同步通知設備維修與製程相關人員。"
        "若感測器參數超出標準範圍，應優先確認對應設備條件並記錄處置結果。"
    )

    return "\n".join(summary)

## 測試 Demo AI 摘要

本段會測試示範版 AI 摘要是否能正常產生結果。

如果這裡可以成功，表示前面的 GraphRAG 查詢與資料整理流程已經能支援診斷摘要。


In [48]:
print(demo_ai_summary("HRM", "打滑"))

【異常概況】
本次查詢的設備為 HRM，異常狀況為 打滑。系統在歷史資料中找到 12 筆相似事件，平均停機時間約為 40.6 分鐘。

【可能原因】
- 人工操作疏失：歷史資料中出現 4 次。
- 原料/鋼種條件差異：歷史資料中出現 3 次。
- 設備磨耗：歷史資料中出現 2 次。

【對應 SOP】
- SOP001：HRM打滑。HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷
  1. 根節點：HRM打滑
  1. 判斷是否持續打滑：當支退爐、在軋一支是否打滑
  1. 確認加熱條件：確認加熱爐時間與溫度作業
  1. 檢查夾輥：PI11夾輥作動夾持秒
  1. 檢查水量與噴嘴：HRM水量設定與噴嘴阻塞與角度目視檢查

【相似歷史案例】
- EVT00255：停機 4 分鐘，原因為 人工操作疏失，處理方式為：重新教育與增加防呆檢核
- EVT00013：停機 19 分鐘，原因為 原料/鋼種條件差異，處理方式為：確認工單與鋼種條件，調整加熱/速度設定
- EVT00296：停機 24 分鐘，原因為 人工操作疏失，處理方式為：重新教育與增加防呆檢核

【建議通報內容】
設備 HRM 發生 打滑 異常。建議依系統查得之 SOP 進行初步檢查，並同步通知設備維修與製程相關人員。若感測器參數超出標準範圍，應優先確認對應設備條件並記錄處置結果。


### 說明

目前你完成了：

Step 1：資料可解讀化

Step 2：建立知識圖譜 Graph

Step 3：GraphRAG 查詢雛形

Step 4：LLM Prompt 建立

Step 5：Demo AI 摘要


### 說明

完成資料可解讀化後，本專題進一步將設備、異常狀況、SOP、SOP 步驟、監控項目與歷史異常事件建立為知識圖譜。圖譜中的節點包含 Equipment、State、SOP、SOPStep、Monitor、Event 與 Cause，關係則包含 HAS_STATE、HAS_SOP、HAS_STEP、NEXT_STEP、CHECKS、OCCURRED_ON、USED_SOP 與 HAS_CAUSE。

透過此設計，系統可以由單一異常事件出發，查詢其發生設備、異常狀況、對應 SOP、SOP 步驟、監控項目與歷史原因分類。這些查詢結果會作為 GraphRAG 的檢索上下文，再交由生成式 AI 產生診斷摘要。相較於直接讓 LLM 回答，本系統能降低模型幻覺，並讓回答更貼近企業內部維修知識。


## 現在要正式加入 5 個 Agent：

1. Diagnosis Agent：判斷可能原因。

2. SOP Agent：整理 SOP 步驟，並強化為階段式流程。

3. Human Assistance Agent：整理原始檔與 SOP 中可由人工執行的檢查、處置、佐證與升級通報條件。

4. Parts Agent：推估可能備件。

5. Notification Agent：產生通報訊息。


## 先確認前面資料都存在

本段會確認前面建立的可讀資料表、知識圖譜與查詢資料是否存在。

這可以避免後面建立 Agent 時因為缺少資料而出錯。


In [49]:
# Step 45：確認目前需要的資料是否存在

required_vars = [
    "clean_event_view",
    "clean_sop_view",
    "clean_sensor_view",
    "sop_detail_view",
    "human_assist_view",
    "graph_nodes",
    "graph_edges",
    "G"
]

for var in required_vars:
    if var in globals():
        print(f"{var} 已存在")
    else:
        print(f"{var} 不存在，請先回前面步驟重新建立")


clean_event_view 已存在
clean_sop_view 已存在
clean_sensor_view 已存在
sop_detail_view 已存在
human_assist_view 已存在
graph_nodes 已存在
graph_edges 已存在
G 已存在


## 建立查詢用的共用函式

本段建立多個 Agent 會共用的查詢工具。

這些函式負責查詢相似歷史事件、對應 SOP 與感測器資料，讓各個 Agent 不需要重複寫查詢邏輯。


In [50]:
# Step 46：建立共用查詢函式

def get_matched_events(equipment_code=None, state_name=None):
    """
    根據設備代碼與異常狀況，找出相似歷史事件。
    """
    df = clean_event_view.copy()

    if equipment_code:
        df = df[
            df["equipment_code"].astype(str).str.contains(
                str(equipment_code), case=False, na=False
            )
        ]

    if state_name:
        df = df[
            df["state_name"].astype(str).str.contains(
                str(state_name), case=False, na=False
            )
        ]

    return df


def get_sop_by_equipment_state(equipment_code=None, state_name=None):
    """
    根據設備與異常狀況，找出可能對應 SOP。
    """
    events = get_matched_events(equipment_code, state_name)

    if events.empty:
        return pd.DataFrame()

    sop_ids = events["sop_id"].dropna().unique().tolist()

    sop_df = clean_sop_view[
        clean_sop_view["sop_id"].isin(sop_ids)
    ].copy()

    return sop_df.sort_values(["sop_id", "sort_order"])


def get_sensor_by_events(event_ids):
    """
    根據 event_id 清單，找出感測器快照資料。
    """
    return clean_sensor_view[
        clean_sensor_view["event_id"].isin(event_ids)
    ].copy()


def format_empty_message(agent_name):
    return f"【{agent_name}】目前資料不足，無法產生完整建議。"

### 說明

建立三個查詢工具：

1. get_matched_events()	找相似歷史事件

2. get_sop_by_equipment_state()	找對應 SOP

3. get_sensor_by_events()	找感測器資料

後面的 Agent 都會呼叫這些工具。


## 建立 Diagnosis Agent

本段建立 Diagnosis Agent。

它負責根據歷史異常事件、原因分類與感測器資料，推估目前異常可能的原因。


## 負責：
 1.判斷可能原因

 2.找出歷史資料中最常見的 root_cause_category

 3.找出感測器異常項目

 4.整理初步診斷依據


In [51]:
# Step 47：Diagnosis Agent

def diagnosis_agent(equipment_code, state_name, top_n_causes=3):
    """
    Diagnosis Agent：根據設備、異常狀況、歷史案例與感測器資料判斷可能原因。
    """

    events = get_matched_events(equipment_code, state_name)

    if events.empty:
        return {
            "agent": "Diagnosis Agent",
            "status": "no_data",
            "text": format_empty_message("Diagnosis Agent"),
            "possible_causes": [],
            "matched_event_count": 0
        }

    # 1. 統計歷史常見原因
    cause_counts = (
        events["root_cause_category"]
        .dropna()
        .value_counts()
        .head(top_n_causes)
    )

    possible_causes = []

    for cause, count in cause_counts.items():
        possible_causes.append({
            "cause": cause,
            "count": int(count),
            "evidence": f"歷史相似事件中出現 {count} 次"
        })

    # 2. 找相似事件的感測器資料
    event_ids = events["event_id"].dropna().unique().tolist()
    sensor_df = get_sensor_by_events(event_ids)

    abnormal_sensors = pd.DataFrame()

    if not sensor_df.empty and "judgement" in sensor_df.columns:
        abnormal_sensors = sensor_df[
            sensor_df["judgement"].astype(str).str.contains(
                "異常|NG|超出|不正常", case=False, na=False
            )
        ]

    # 3. 統計平均停機時間
    avg_downtime = events["downtime_min"].mean()
    max_downtime = events["downtime_min"].max()
    min_downtime = events["downtime_min"].min()

    # 4. 產生文字結果
    lines = []
    lines.append("【Diagnosis Agent｜異常診斷結果】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append(f"找到相似歷史事件：{len(events)} 筆")
    lines.append(f"平均停機時間：約 {avg_downtime:.1f} 分鐘")
    lines.append(f"停機時間範圍：{min_downtime:.0f} 至 {max_downtime:.0f} 分鐘")
    lines.append("")

    lines.append("可能原因排序：")
    if possible_causes:
        for idx, item in enumerate(possible_causes, start=1):
            lines.append(
                f"{idx}. {item['cause']}：{item['evidence']}"
            )
    else:
        lines.append("目前歷史資料中沒有明確的原因分類。")

    lines.append("")
    lines.append("感測器異常觀察：")

    if abnormal_sensors.empty:
        lines.append("目前未從相似事件中整理出明確的感測器異常紀錄。")
    else:
        sensor_summary = (
            abnormal_sensors["monitor_name"]
            .dropna()
            .value_counts()
            .head(5)
        )
        for monitor_name, count in sensor_summary.items():
            lines.append(f"- {monitor_name}：曾出現 {count} 次異常判斷")

    return {
        "agent": "Diagnosis Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "possible_causes": possible_causes,
        "matched_event_count": len(events),
        "avg_downtime": avg_downtime
    }

## 測試 Diagnosis Agent

本段會測試 Diagnosis Agent 的輸出結果。

重點是確認它是否能找出相似事件、常見原因與異常感測器項目。


In [52]:
# Step 48：測試 Diagnosis Agent

diagnosis_result = diagnosis_agent("HRM", "打滑")
print(diagnosis_result["text"])

【Diagnosis Agent｜異常診斷結果】
查詢設備：HRM
異常狀況：打滑
找到相似歷史事件：12 筆
平均停機時間：約 40.6 分鐘
停機時間範圍：4 至 83 分鐘

可能原因排序：
1. 人工操作疏失：歷史相似事件中出現 4 次
2. 原料/鋼種條件差異：歷史相似事件中出現 3 次
3. 設備磨耗：歷史相似事件中出現 2 次

感測器異常觀察：
- Guide Roll振動值：曾出現 4 次異常判斷
- 加熱爐溫度：曾出現 3 次異常判斷
- 通訊訊號狀態：曾出現 3 次異常判斷
- HRM水量設定：曾出現 3 次異常判斷
- PI11夾輥作動夾持秒：曾出現 2 次異常判斷


## 建立 SOP Agent

這版 SOP Agent 不再只是把 SOP 表格列出來，而是會依照 `sop_stage` 分成：

1. 異常確認
2. 安全處置
3. 原因排查
4. 處置修正
5. 復機確認
6. 紀錄回饋

每個步驟也會列出判斷方式、監控項目、判斷標準、需要留下的佐證，以及異常時下一步建議。


### 說明

負責：

1. 找出對應 SOP

2. 整理 SOP 步驟

3. 列出每一步要檢查的監控項目

4. 列出安全提醒


In [53]:
# Step 49：強化版 SOP Agent

def sop_agent(equipment_code, state_name):
    """
    SOP Agent：根據設備與異常狀況，整理較完整的 SOP 執行流程。

    強化重點：
    1. 不只列步驟，而是依階段呈現。
    2. 區分人工確認、系統自動判斷、系統輔助加人工確認。
    3. 顯示監控項目、判斷標準、佐證資料。
    4. 提供異常時下一步建議。
    """

    sop_df = get_sop_by_equipment_state(equipment_code, state_name)

    if sop_df.empty:
        return {
            "agent": "SOP Agent",
            "status": "no_data",
            "text": format_empty_message("SOP Agent"),
            "sop_count": 0,
            "sop_df": sop_df
        }

    # 優先使用前面建立的強化 SOP 表
    if "sop_detail_view" in globals():
        target_sop_ids = sop_df["sop_id"].dropna().unique().tolist()
        detail_df = sop_detail_view[
            sop_detail_view["sop_id"].isin(target_sop_ids)
        ].copy()
    else:
        detail_df = sop_df.copy()

    # 若缺少強化欄位，現場補上，避免單獨執行時報錯
    if "sop_stage" not in detail_df.columns:
        detail_df["sop_stage"] = detail_df.apply(
            lambda row: classify_sop_stage(row.get("step_title", ""), row.get("step_content", "")),
            axis=1
        )

    if "check_method_label" not in detail_df.columns:
        detail_df["check_method_label"] = detail_df["check_type"].apply(translate_check_type)

    if "evidence_required" not in detail_df.columns:
        detail_df["evidence_required"] = detail_df.apply(build_evidence_required, axis=1)

    if "next_action_hint" not in detail_df.columns:
        detail_df["next_action_hint"] = detail_df.apply(build_next_action_hint, axis=1)

    lines = []
    lines.append("【SOP Agent｜強化版 SOP 流程建議】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append("")

    sop_count = detail_df["sop_id"].nunique() if "sop_id" in detail_df.columns else 0
    lines.append(f"找到對應 SOP：{sop_count} 份")
    lines.append("")

    lines.append("【SOP 使用原則】")
    lines.append("1. 先確認異常是否仍持續發生，避免對已消失的短暫異常過度處置。")
    lines.append("2. 涉及人員安全、設備保護或可能擴大損壞時，應先停機或通知班長。")
    lines.append("3. 可自動判斷的項目，應優先比對系統數值與標準範圍。")
    lines.append("4. 需要人工判斷的項目，應留下照片、備註或現場確認紀錄。")
    lines.append("5. 若依 SOP 處置後仍未改善，應升級通知製程工程師與設備工程師。")
    lines.append("")

    for sop_id, group in detail_df.groupby("sop_id"):
        first = group.iloc[0]

        lines.append("=" * 70)
        lines.append(f"【{sop_id}｜{first.get('sop_name', '')}】")
        lines.append("=" * 70)

        if pd.notna(first.get("sop_desc", None)):
            lines.append(f"SOP 說明：{first.get('sop_desc')}")

        if pd.notna(first.get("owner_role", None)):
            lines.append(f"主要負責角色：{first.get('owner_role')}")

        if pd.notna(first.get("version", None)):
            lines.append(f"版本：{first.get('version')}")

        if pd.notna(first.get("status", None)):
            lines.append(f"狀態：{first.get('status')}")

        lines.append("")

        if "sort_order" in group.columns:
            group = group.sort_values(["sop_stage", "sort_order"])
        else:
            group = group.sort_values(["sop_stage"])

        for stage, stage_df in group.groupby("sop_stage", sort=True):
            lines.append(f"【{stage}】")

            for _, step in stage_df.iterrows():
                sort_order = step.get("sort_order", "")
                branch_label = step.get("branch_label", "")

                prefix = f"{int(sort_order)}." if pd.notna(sort_order) else "-"

                if pd.notna(branch_label) and str(branch_label).strip() != "":
                    prefix += f" [{branch_label}]"

                lines.append(f"{prefix} {step.get('step_title', '')}")
                lines.append(f"   執行內容：{step.get('step_content', '')}")
                lines.append(f"   判斷方式：{step.get('check_method_label', '未標示')}")

                monitor_name = step.get("monitor_name", None)
                if pd.notna(monitor_name):
                    lines.append(f"   監控項目：{monitor_name}")
                else:
                    lines.append("   監控項目：無，需由現場人員判斷")

                standard_text = step.get("standard_text", None)
                if pd.notna(standard_text):
                    lines.append(f"   判斷標準：{standard_text}")
                else:
                    lines.append("   判斷標準：資料未提供，需依現場經驗或主管指示確認")

                lines.append(f"   需留存佐證：{step.get('evidence_required', '人工確認備註')}")

                if pd.notna(step.get("safety_note", None)):
                    lines.append(f"   安全提醒：{step.get('safety_note')}")

                lines.append(f"   異常時下一步：{step.get('next_action_hint', '')}")
                lines.append("")

        lines.append("【SOP 結案建議】")
        lines.append("- 紀錄本次異常主因、處置方式、停機時間與是否復機。")
        lines.append("- 若相同異常重複發生，應回饋 SOP 是否需要新增檢查點。")
        lines.append("- 若現場發現 SOP 缺少關鍵判斷標準，應由製程 / 設備工程師補充。")
        lines.append("")

    return {
        "agent": "SOP Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "sop_count": sop_count,
        "sop_df": detail_df
    }


## 測試 SOP Agent

本段會測試 SOP Agent 是否能正確找出對應 SOP，並輸出可讀的處理流程。


In [54]:
# Step 50：測試 SOP Agent

sop_result = sop_agent("HRM", "打滑")
print(sop_result["text"])


【SOP Agent｜強化版 SOP 流程建議】
查詢設備：HRM
異常狀況：打滑

找到對應 SOP：1 份

【SOP 使用原則】
1. 先確認異常是否仍持續發生，避免對已消失的短暫異常過度處置。
2. 涉及人員安全、設備保護或可能擴大損壞時，應先停機或通知班長。
3. 可自動判斷的項目，應優先比對系統數值與標準範圍。
4. 需要人工判斷的項目，應留下照片、備註或現場確認紀錄。
5. 若依 SOP 處置後仍未改善，應升級通知製程工程師與設備工程師。

【SOP001｜HRM打滑】
SOP 說明：HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷
主要負責角色：軋製課/現場班長
版本：v1.0
狀態：active

【1. 異常確認】
1. 根節點
   執行內容：HRM打滑
   判斷方式：人工確認
   監控項目：無，需由現場人員判斷
   判斷標準：異常起點
   需留存佐證：安全確認紀錄
   安全提醒：先確認現場安全
   異常時下一步：若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。

1. [->] 判斷是否持續打滑
   執行內容：當支退爐、在軋一支是否打滑
   判斷方式：人工確認
   監控項目：無，需由現場人員判斷
   判斷標準：確認是否只發生單支或持續發生
   需留存佐證：安全確認紀錄
   安全提醒：避免人員靠近入料區
   異常時下一步：若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。

1. [正常] 復機觀察
   執行內容：開四台隧道加熱軋延第二支，優先410鋼種，70%/70%/70%/60%
   判斷方式：人工確認
   監控項目：無，需由現場人員判斷
   判斷標準：復機後至少觀察2支
   需留存佐證：人工確認備註
   異常時下一步：若加熱條件異常，請先比對鋼種標準，必要時調整加熱參數或退爐。

1. [正常] 結案
   執行內容：持續軋延觀察
   判斷方式：人工確認
   監控項目：無，需由現場人員判斷
   判斷標準：穩定後關閉事件
   需留存佐證：人工確認備註
   異常時下一步：若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。

2. [否] 持續觀察
   執行內容：持續軋延分析打滑鋼種是否異常
   判斷方式

## 建立 Human Assistance Agent

原本的人工輔助如果只寫「請現場人員確認」，會太粗糙。

這個 Agent 會根據原始檔裡的人工檢查表與案例筆記，整理出：

1. 人工需確認什麼。
2. 建議人工處置方式。
3. 可搭配哪些監控項目。
4. 需要留下什麼佐證。
5. 什麼情況要升級通報。


In [55]:
# Step 50-1：Human Assistance Agent

def human_assistance_agent(equipment_code, state_name, max_items=8):
    """
    Human Assistance Agent：
    根據設備與異常狀況，從原始人工處置資料中找出
    現場人員可以檢查什麼、怎麼處理、要留下什麼證據、何時升級通報。
    """

    if "human_assist_view" not in globals() or human_assist_view.empty:
        return {
            "agent": "Human Assistance Agent",
            "status": "no_data",
            "text": "目前尚未建立人工輔助知識表，無法產生人工輔助建議。",
            "assist_df": pd.DataFrame()
        }

    df = human_assist_view.copy()

    # 先用設備篩選
    matched = df[
        df["equipment_code"].astype(str).str.contains(str(equipment_code), case=False, na=False)
    ].copy()

    # 再用異常狀況、檢查項目、處置內容搜尋
    state_matched = matched[
        matched["state_keyword"].astype(str).str.contains(str(state_name), case=False, na=False)
        |
        matched["check_item"].astype(str).str.contains(str(state_name), case=False, na=False)
        |
        matched["human_action"].astype(str).str.contains(str(state_name), case=False, na=False)
    ].copy()

    # 若異常名稱完全對不到，保留同設備資料，避免完全沒有人工輔助建議
    if not state_matched.empty:
        matched = state_matched

    if matched.empty:
        return {
            "agent": "Human Assistance Agent",
            "status": "no_data",
            "text": (
                "【Human Assistance Agent｜人工輔助建議】\n"
                f"查詢設備：{equipment_code}\n"
                f"異常狀況：{state_name}\n\n"
                "目前找不到對應的人工輔助知識，建議回到 SOP 步驟進行人工確認，"
                "並將本次處置經驗補回人工輔助知識庫。"
            ),
            "assist_df": pd.DataFrame()
        }

    matched = matched.head(max_items)

    lines = []
    lines.append("【Human Assistance Agent｜人工輔助建議】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append("")
    lines.append("此 Agent 的目的不是取代現場人員，而是把原始案例中的人工檢查與處置經驗整理成可執行清單。")
    lines.append("")

    for idx, row in matched.reset_index(drop=True).iterrows():
        lines.append(f"【人工輔助項目 {idx + 1}】")
        lines.append(f"類型：{row.get('assist_type', '人工輔助')}")

        if pd.notna(row.get("check_item")):
            lines.append(f"人工需確認：{row.get('check_item')}")
        else:
            lines.append("人工需確認：依現場狀況判斷")

        if pd.notna(row.get("human_action")):
            lines.append(f"建議人工處置：{row.get('human_action')}")
        else:
            lines.append("建議人工處置：目前資料未明確提供，需由班長或工程師判斷")

        if pd.notna(row.get("related_monitor_id")):
            lines.append(f"可搭配監控項目：{row.get('related_monitor_id')}")
        else:
            lines.append("可搭配監控項目：無明確監控項目，偏人工經驗判斷")

        lines.append(f"建議留下佐證：{row.get('evidence_required', '人工確認紀錄')}")
        lines.append(f"升級通報條件：{row.get('escalation_rule', '若處置後仍未改善，需升級通報')}")
        lines.append(f"資料來源：{row.get('source_note', '原始資料')}")
        lines.append("")

    lines.append("【人工輔助總結】")
    lines.append("1. AI 可協助整理檢查順序與歷史處置經驗，但現場安全與最終判斷仍需由人工確認。")
    lines.append("2. 若系統數值正常但現場仍異常，應優先保留人工觀察紀錄與照片。")
    lines.append("3. 若人工處置後恢復正常，應將處置方式回填至異常事件紀錄，作為後續案例推薦依據。")
    lines.append("4. 若相同異常重複發生，應回饋 SOP 是否需要新增檢查點或調整判斷標準。")

    return {
        "agent": "Human Assistance Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "assist_df": matched
    }


## 測試 Human Assistance Agent

本段會測試 Human Assistance Agent 是否能根據設備與異常狀況，整理出對應的人工檢查與處置建議。


In [56]:
# Step 50-2：測試 Human Assistance Agent

human_result = human_assistance_agent("HRM", "頭端咬不入")
print(human_result["text"])


【Human Assistance Agent｜人工輔助建議】
查詢設備：HRM
異常狀況：頭端咬不入

此 Agent 的目的不是取代現場人員，而是把原始案例中的人工檢查與處置經驗整理成可執行清單。

【人工輔助項目 1】
類型：3. 原因排查
人工需確認：異常確認
建議人工處置：確認HRM是否發生「頭端咬不入」，並記錄時間、工單與鋼種
可搭配監控項目：無明確監控項目，偏人工經驗判斷
建議留下佐證：人工確認備註
升級通報條件：若製程條件與鋼種標準不一致，通知製程工程師確認。
資料來源：來自 SOP002｜HRM-頭端咬不入異常處理

【人工輔助項目 2】
類型：2. 安全處置
人工需確認：安全隔離
建議人工處置：若異常可能造成人員或設備風險，先停機並通知班長
可搭配監控項目：無明確監控項目，偏人工經驗判斷
建議留下佐證：安全確認紀錄
升級通報條件：若涉及人員或設備安全風險，立即停機並通知班長。
資料來源：來自 SOP002｜HRM-頭端咬不入異常處理

【人工輔助項目 3】
類型：1. 異常確認
人工需確認：持續觀察
建議人工處置：若為瞬間訊號，先觀察下一支或下一批次
可搭配監控項目：無明確監控項目，偏人工經驗判斷
建議留下佐證：人工確認備註
升級通報條件：若訊號異常、資料缺漏或感測值不可信，通知自動化 / 設備單位確認。
資料來源：來自 SOP002｜HRM-頭端咬不入異常處理

【人工輔助項目 4】
類型：3. 原因排查
人工需確認：檢查常見原因
建議人工處置：依設備點檢表確認頭端咬不入的常見原因：磨耗、偏移、斷裂、參數錯誤或通訊異常
可搭配監控項目：無明確監控項目，偏人工經驗判斷
建議留下佐證：現場照片
升級通報條件：若訊號異常、資料缺漏或感測值不可信，通知自動化 / 設備單位確認。
資料來源：來自 SOP002｜HRM-頭端咬不入異常處理

【人工輔助項目 5】
類型：4. 處置修正
人工需確認：執行處置
建議人工處置：依SOP處置：清理、調整、更換零件或重啟通訊；完成後復測
可搭配監控項目：無明確監控項目，偏人工經驗判斷
建議留下佐證：現場照片
升級通報條件：若訊號異常、資料缺漏或感測值不可信，通知自動化 / 設備單位確認。
資料來源：來自 SOP002｜HRM-頭端咬不入異常處理

【人工輔助項目 6】
類型：2. 安全處置
人工需確認：升級處理
建議

## 建立 Parts Agent

本段建立 Parts Agent。

它會根據異常狀況、SOP 文字、歷史處置紀錄與關鍵字，推估可能需要確認的備件或物料。


### 說明

負責推估可能備件，因為資料裡不一定真的有完整「備件庫存表」，所以我們先做一個合理的規則式 Agent：

1. 根據異常狀況、原因分類、SOP 文字中的關鍵字

2. 推估可能需要確認的備件或物料


In [57]:
# Step 51：Parts Agent

def parts_agent(equipment_code, state_name):
    """
    Parts Agent：根據異常狀況、原因分類與 SOP 內容推估可能備件。
    如果資料中沒有正式備件庫，先以規則式方式推估。
    """

    events = get_matched_events(equipment_code, state_name)
    sop_df = get_sop_by_equipment_state(equipment_code, state_name)

    if events.empty and sop_df.empty:
        return {
            "agent": "Parts Agent",
            "status": "no_data",
            "text": format_empty_message("Parts Agent"),
            "parts": []
        }

    # 將相關文字合併，做關鍵字判斷
    text_pool = []

    if not events.empty:
        for col in ["state_name", "root_cause_category", "cause_summary", "action_summary"]:
            if col in events.columns:
                text_pool.extend(events[col].dropna().astype(str).tolist())

    if not sop_df.empty:
        for col in ["step_title", "step_content", "standard_text", "safety_note"]:
            if col in sop_df.columns:
                text_pool.extend(sop_df[col].dropna().astype(str).tolist())

    all_text = " ".join(text_pool)

    # 備件推估規則
    part_rules = {
        "軸承": ["軸承", "bearing"],
        "軋輥": ["軋輥", "roll", "Roll", "TC Roll"],
        "感測器": ["感測器", "sensor", "訊號", "signal", "loss"],
        "噴嘴": ["噴嘴", "水量", "冷卻", "堵塞"],
        "皮帶/傳動件": ["皮帶", "傳動", "打滑", "馬達", "鏈條"],
        "油壓/潤滑相關零件": ["油壓", "潤滑", "油", "壓力"],
        "電控元件": ["電流", "電壓", "PLC", "控制", "訊號"],
        "加熱爐相關零件": ["加熱", "爐", "溫度", "燃燒"]
    }

    recommended_parts = []

    for part_name, keywords in part_rules.items():
        matched_keywords = [kw for kw in keywords if kw in all_text]
        if matched_keywords:
            recommended_parts.append({
                "part": part_name,
                "matched_keywords": matched_keywords,
                "reason": f"相關資料中出現關鍵字：{', '.join(matched_keywords)}"
            })

    lines = []
    lines.append("【Parts Agent｜備件與物料確認建議】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append("")

    if recommended_parts:
        lines.append("建議優先確認以下備件或物料：")
        for idx, item in enumerate(recommended_parts, start=1):
            lines.append(f"{idx}. {item['part']}")
            lines.append(f"   - 推估原因：{item['reason']}")
    else:
        lines.append("目前資料中沒有明確備件線索。")
        lines.append("建議至少確認：常用消耗品、設備專用備件、感測器與安全防護用品。")

    lines.append("")
    lines.append("補充說明：")
    lines.append("目前資料若沒有正式庫存表，Parts Agent 採用 SOP 與歷史案例文字進行關鍵字推估。")
    lines.append("若後續取得材料庫存資料，可再改成直接查詢庫存數量與安全庫存。")

    return {
        "agent": "Parts Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "parts": recommended_parts
    }

## 測試 Parts Agent

本段會測試 Parts Agent 是否能輸出可能需要確認的備件或物料。


In [58]:
# Step 52：測試 Parts Agent

parts_result = parts_agent("HRM", "打滑")
print(parts_result["text"])

【Parts Agent｜備件與物料確認建議】
查詢設備：HRM
異常狀況：打滑

建議優先確認以下備件或物料：
1. 軋輥
   - 推估原因：相關資料中出現關鍵字：軋輥
2. 噴嘴
   - 推估原因：相關資料中出現關鍵字：噴嘴, 水量
3. 皮帶/傳動件
   - 推估原因：相關資料中出現關鍵字：打滑
4. 加熱爐相關零件
   - 推估原因：相關資料中出現關鍵字：加熱, 爐, 溫度

補充說明：
目前資料若沒有正式庫存表，Parts Agent 採用 SOP 與歷史案例文字進行關鍵字推估。
若後續取得材料庫存資料，可再改成直接查詢庫存數量與安全庫存。


## 建立 Notification Agent

本段建立 Notification Agent。

它負責把異常診斷結果整理成可通報給現場人員、主管或維修單位的文字。


### 說明

負責：

1. 產生正式通報訊息

2. 包含設備、異常、嚴重度、可能原因、SOP、建議處理、通報對象


In [59]:
# Step 53：Notification Agent

def notification_agent(equipment_code, state_name, diagnosis_result=None, sop_result=None):
    """
    Notification Agent：產生正式異常通報訊息。
    """

    events = get_matched_events(equipment_code, state_name)

    if events.empty:
        severity = "未判定"
        avg_downtime = None
        event_count = 0
    else:
        severity = events["severity"].mode().iloc[0] if not events["severity"].dropna().empty else "未判定"
        avg_downtime = events["downtime_min"].mean()
        event_count = len(events)

    # 取得可能原因
    possible_causes = []
    if diagnosis_result and diagnosis_result.get("possible_causes"):
        possible_causes = [
            item["cause"] for item in diagnosis_result["possible_causes"]
        ]

    cause_text = "、".join(possible_causes) if possible_causes else "目前資料不足，需現場進一步確認"

    # 取得 SOP 數量
    sop_count = 0
    if sop_result:
        sop_count = sop_result.get("sop_count", 0)

    # 通報單位推估
    notify_units = ["設備維修單位", "製程工程單位"]

    if str(severity).upper() == "A":
        notify_units.append("產線主管")
        notify_units.append("品質工程單位")
    elif str(severity).upper() == "B":
        notify_units.append("班長")

    lines = []
    lines.append("【Notification Agent｜異常通報內容】")
    lines.append("")
    lines.append("主旨：製造設備異常通報")
    lines.append("")
    lines.append("通報內容：")
    lines.append(f"設備 {equipment_code} 發生「{state_name}」異常。")
    lines.append(f"依歷史資料判斷，此類事件常見嚴重度為 {severity}。")

    if avg_downtime is not None:
        lines.append(f"系統找到 {event_count} 筆相似歷史事件，平均停機時間約 {avg_downtime:.1f} 分鐘。")
    else:
        lines.append("目前系統尚未找到相似歷史事件。")

    lines.append(f"初步可能原因包含：{cause_text}。")

    if sop_count > 0:
        lines.append(f"系統已找到 {sop_count} 份可能對應 SOP，建議現場人員依 SOP 步驟進行檢查。")
    else:
        lines.append("目前未找到明確對應 SOP，建議由現場工程人員建立或補充處理流程。")

    lines.append("")
    lines.append("建議處理：")
    lines.append("1. 先確認設備是否仍處於異常狀態。")
    lines.append("2. 依 SOP 檢查關鍵參數與設備部位。")
    lines.append("3. 若感測器數值超出標準，請優先排除該項目。")
    lines.append("4. 處理完成後記錄原因、處置方式與復機結果。")

    lines.append("")
    lines.append("建議通報單位：")
    for unit in notify_units:
        lines.append(f"- {unit}")

    return {
        "agent": "Notification Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "notify_units": notify_units,
        "severity": severity
    }

## 測試 Notification Agent

本段會測試 Notification Agent 是否能產生可讀的異常通報內容。


In [60]:
# Step 54：測試 Notification Agent

notification_result = notification_agent(
    "HRM",
    "打滑",
    diagnosis_result=diagnosis_result,
    sop_result=sop_result
)

print(notification_result["text"])

【Notification Agent｜異常通報內容】

主旨：製造設備異常通報

通報內容：
設備 HRM 發生「打滑」異常。
依歷史資料判斷，此類事件常見嚴重度為 B。
系統找到 12 筆相似歷史事件，平均停機時間約 40.6 分鐘。
初步可能原因包含：人工操作疏失、原料/鋼種條件差異、設備磨耗。
系統已找到 1 份可能對應 SOP，建議現場人員依 SOP 步驟進行檢查。

建議處理：
1. 先確認設備是否仍處於異常狀態。
2. 依 SOP 檢查關鍵參數與設備部位。
3. 若感測器數值超出標準，請優先排除該項目。
4. 處理完成後記錄原因、處置方式與復機結果。

建議通報單位：
- 設備維修單位
- 製程工程單位
- 班長


## 建立 Multi-Agent Orchestrator

本段建立 Multi-Agent 總控流程。

使用者只要輸入設備與異常狀況，系統就會依序呼叫不同 Agent，最後整合成一份完整的診斷結果。


### 說明

現在五個 Agent 都有了，我們要建立一個「總控流程」。

使用者只要輸入：

設備：HRM

異常：打滑

系統就會依序呼叫：

Diagnosis Agent → SOP Agent → Human Assistance Agent → Parts Agent → Notification Agent → 整合成最終診斷報告


In [61]:
# Step 55：強化版 Multi-Agent Orchestrator

def multi_agent_diagnosis(equipment_code, state_name):
    """
    多 Agent 總控流程：
    1. Diagnosis Agent：可能原因判斷
    2. SOP Agent：標準處理流程
    3. Human Assistance Agent：人工輔助檢查與處置
    4. Parts Agent：備件與物料建議
    5. Notification Agent：通報訊息
    6. 整合最終報告
    """

    diagnosis = diagnosis_agent(equipment_code, state_name)
    sop = sop_agent(equipment_code, state_name)
    human = human_assistance_agent(equipment_code, state_name)
    parts = parts_agent(equipment_code, state_name)
    notification = notification_agent(
        equipment_code,
        state_name,
        diagnosis_result=diagnosis,
        sop_result=sop
    )

    final_lines = []
    final_lines.append("=" * 80)
    final_lines.append("生成式 AI 製造設備異常診斷報告")
    final_lines.append("=" * 80)
    final_lines.append("")

    final_lines.append("一、Diagnosis Agent：可能原因判斷")
    final_lines.append(diagnosis["text"])
    final_lines.append("")

    final_lines.append("二、SOP Agent：標準處理流程")
    final_lines.append(sop["text"])
    final_lines.append("")

    final_lines.append("三、Human Assistance Agent：人工輔助檢查與處置")
    final_lines.append(human["text"])
    final_lines.append("")

    final_lines.append("四、Parts Agent：備件與物料建議")
    final_lines.append(parts["text"])
    final_lines.append("")

    final_lines.append("五、Notification Agent：異常通報建議")
    final_lines.append(notification["text"])
    final_lines.append("")

    final_lines.append("=" * 80)
    final_lines.append("整合結論")
    final_lines.append("=" * 80)
    final_lines.append(
        "本系統先由 Diagnosis Agent 根據歷史事件與感測器資料判斷可能原因，"
        "再由 SOP Agent 找出標準處理流程，接著由 Human Assistance Agent "
        "補充現場人員可執行的人工檢查、處置方式、佐證資料與升級條件，"
        "最後由 Parts Agent 與 Notification Agent 分別提供備件確認與通報建議。"
    )

    return {
        "diagnosis": diagnosis,
        "sop": sop,
        "human": human,
        "parts": parts,
        "notification": notification,
        "final_report": "\n".join(final_lines)
    }


## 測試完整多 Agent 流程

本段會測試完整的多 Agent 流程。

目標是確認 Diagnosis Agent、SOP Agent、Human Assistance Agent、Parts Agent 與 Notification Agent 能否順利串接。


In [62]:
# Step 56：測試完整多 Agent 流程

multi_result = multi_agent_diagnosis("HRM", "打滑")

print(multi_result["final_report"])

生成式 AI 製造設備異常診斷報告

一、Diagnosis Agent：可能原因判斷
【Diagnosis Agent｜異常診斷結果】
查詢設備：HRM
異常狀況：打滑
找到相似歷史事件：12 筆
平均停機時間：約 40.6 分鐘
停機時間範圍：4 至 83 分鐘

可能原因排序：
1. 人工操作疏失：歷史相似事件中出現 4 次
2. 原料/鋼種條件差異：歷史相似事件中出現 3 次
3. 設備磨耗：歷史相似事件中出現 2 次

感測器異常觀察：
- Guide Roll振動值：曾出現 4 次異常判斷
- 加熱爐溫度：曾出現 3 次異常判斷
- 通訊訊號狀態：曾出現 3 次異常判斷
- HRM水量設定：曾出現 3 次異常判斷
- PI11夾輥作動夾持秒：曾出現 2 次異常判斷

二、SOP Agent：標準處理流程
【SOP Agent｜強化版 SOP 流程建議】
查詢設備：HRM
異常狀況：打滑

找到對應 SOP：1 份

【SOP 使用原則】
1. 先確認異常是否仍持續發生，避免對已消失的短暫異常過度處置。
2. 涉及人員安全、設備保護或可能擴大損壞時，應先停機或通知班長。
3. 可自動判斷的項目，應優先比對系統數值與標準範圍。
4. 需要人工判斷的項目，應留下照片、備註或現場確認紀錄。
5. 若依 SOP 處置後仍未改善，應升級通知製程工程師與設備工程師。

【SOP001｜HRM打滑】
SOP 說明：HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷
主要負責角色：軋製課/現場班長
版本：v1.0
狀態：active

【1. 異常確認】
1. 根節點
   執行內容：HRM打滑
   判斷方式：人工確認
   監控項目：無，需由現場人員判斷
   判斷標準：異常起點
   需留存佐證：安全確認紀錄
   安全提醒：先確認現場安全
   異常時下一步：若此步驟判定異常，請依 SOP 分支進入下一個檢查節點，並留下處置紀錄。

1. [->] 判斷是否持續打滑
   執行內容：當支退爐、在軋一支是否打滑
   判斷方式：人工確認
   監控項目：無，需由現場人員判斷
   判斷標準：確認是否只發生單支或持續發生
   需留存佐證：安全確認紀錄
   安全提醒：避免人員靠近入料區
   異常時下一步：若此步驟判定異常，請依 SOP 分支進入下

### 如果 HRM 打滑查不到，先找可用案例

本段說明「如果 HRM 打滑查不到，先找可用案例」的用途。請搭配下方程式碼觀察輸出結果，確認資料處理流程是否正確。


In [63]:
# Step 57：找出資料中可展示的設備 + 異常組合

demo_pairs = (
    clean_event_view
    .groupby(["equipment_code", "state_name"])
    .size()
    .reset_index(name="event_count")
    .sort_values("event_count", ascending=False)
)

demo_pairs.head(30)

,equipment_code,state_name,event_count
53,TMB1,軋輥破裂,16
4,FB,TC Roll破裂,15
24,KB2,Guide Roll破裂,14
11,FEEDING CHANNEL,Roll軸心斷裂,13
21,KB1,傳動軸斷裂,13
3,FB,Guide Roll破裂,13
26,KB2,傳動軸斷裂,12
19,KB1,Guide Roll破裂,12
29,KB2,軸承破裂,11
56,TMB2,TC Roll破裂,11


### 說明

可以從表格中挑一組，例如：equipment_code = KB2、state_name = 軸承破裂


In [64]:
multi_result = multi_agent_diagnosis("KB2", "軸承破裂")
print(multi_result["final_report"])

生成式 AI 製造設備異常診斷報告

一、Diagnosis Agent：可能原因判斷
【Diagnosis Agent｜異常診斷結果】
查詢設備：KB2
異常狀況：軸承破裂
找到相似歷史事件：11 筆
平均停機時間：約 75.5 分鐘
停機時間範圍：4 至 150 分鐘

可能原因排序：
1. 設備磨耗：歷史相似事件中出現 5 次
2. 原料/鋼種條件差異：歷史相似事件中出現 2 次
3. 保養不足：歷史相似事件中出現 1 次

感測器異常觀察：
- HRM水量設定：曾出現 6 次異常判斷
- 通訊訊號狀態：曾出現 4 次異常判斷
- Guide Roll振動值：曾出現 4 次異常判斷
- PI11夾輥作動夾持秒：曾出現 1 次異常判斷
- 加熱爐在爐時間：曾出現 1 次異常判斷

二、SOP Agent：標準處理流程
【SOP Agent｜強化版 SOP 流程建議】
查詢設備：KB2
異常狀況：軸承破裂

找到對應 SOP：1 份

【SOP 使用原則】
1. 先確認異常是否仍持續發生，避免對已消失的短暫異常過度處置。
2. 涉及人員安全、設備保護或可能擴大損壞時，應先停機或通知班長。
3. 可自動判斷的項目，應優先比對系統數值與標準範圍。
4. 需要人工判斷的項目，應留下照片、備註或現場確認紀錄。
5. 若依 SOP 處置後仍未改善，應升級通知製程工程師與設備工程師。

【SOP011｜KB2-軸承破裂異常處理】
SOP 說明：KB2發生軸承破裂時的標準處置流程
主要負責角色：設備/製程工程師
版本：v0.8
狀態：draft

【1. 異常確認】
2. [否] 持續觀察
   執行內容：若為瞬間訊號，先觀察下一支或下一批次
   判斷方式：系統輔助 + 人工確認
   監控項目：Guide Roll振動值
   判斷標準：觀察紀錄
   需留存佐證：系統參數截圖 / 感測數值
   異常時下一步：若訊號異常，請確認 PLC、通訊狀態與感測器連線。

【2. 安全處置】
1. [是] 安全隔離
   執行內容：若異常可能造成人員或設備風險，先停機並通知班長
   判斷方式：系統輔助 + 人工確認
   監控項目：Guide Roll振動值
   判斷標準：安全優先
   需留存佐證：系統參數截圖 / 感測數值、安全確認紀錄
   安全提醒：先確認停機與能源隔離

## 建立真正給 LLM 用的 Multi-Agent Prompt

本段會把多個 Agent 的輸出整理成給 LLM 使用的 Prompt。

這樣之後若要接 Gemini、Groq 或 OpenAI，就可以把診斷結果交給模型產生更自然的摘要。


### 說明

現在的 Agent 是規則式產生結果。
接下來我們要把五個 Agent 的結果整理成一個 Prompt，之後可以丟給 Gemini / Groq / OpenAI。


In [65]:
# Step 58：建立給 LLM 的 Multi-Agent Prompt

def build_multi_agent_llm_prompt(equipment_code, state_name):
    """
    將多 Agent 的輸出整理成 LLM Prompt。
    """

    result = multi_agent_diagnosis(equipment_code, state_name)

    prompt = f"""
你是一位製造設備異常診斷助理，請根據下方多 Agent 系統輸出，
產生一份更自然、更適合現場工程師閱讀的診斷摘要。

請注意：
1. 僅能根據下方資料回答，不要憑空補充。
2. 若資料不足，請直接說明資料不足。
3. 請使用繁體中文。
4. 請用正式但清楚的語氣。
5. 請輸出以下段落：
   - 異常概況
   - 初步判斷
   - SOP 流程建議
   - 人工輔助檢查與處置
   - 備件與物料確認
   - 通報建議
   - 後續資料改善建議

【多 Agent 系統輸出】
{result["final_report"]}

請產生最終診斷摘要：
"""

    return prompt


## 查看 LLM Prompt

本段會查看最終提供給 LLM 的 Prompt。

重點是確認 Prompt 是否包含足夠的診斷依據、SOP 流程、人工提醒、備件建議與通報內容。


In [66]:
# Step 59：查看 Prompt

llm_prompt = build_multi_agent_llm_prompt("HRM", "打滑")
print(llm_prompt)


你是一位製造設備異常診斷助理，請根據下方多 Agent 系統輸出，
產生一份更自然、更適合現場工程師閱讀的診斷摘要。

請注意：
1. 僅能根據下方資料回答，不要憑空補充。
2. 若資料不足，請直接說明資料不足。
3. 請使用繁體中文。
4. 請用正式但清楚的語氣。
5. 請輸出以下段落：
   - 異常概況
   - 初步判斷
   - SOP 流程建議
   - 人工輔助檢查與處置
   - 備件與物料確認
   - 通報建議
   - 後續資料改善建議

【多 Agent 系統輸出】
生成式 AI 製造設備異常診斷報告

一、Diagnosis Agent：可能原因判斷
【Diagnosis Agent｜異常診斷結果】
查詢設備：HRM
異常狀況：打滑
找到相似歷史事件：12 筆
平均停機時間：約 40.6 分鐘
停機時間範圍：4 至 83 分鐘

可能原因排序：
1. 人工操作疏失：歷史相似事件中出現 4 次
2. 原料/鋼種條件差異：歷史相似事件中出現 3 次
3. 設備磨耗：歷史相似事件中出現 2 次

感測器異常觀察：
- Guide Roll振動值：曾出現 4 次異常判斷
- 加熱爐溫度：曾出現 3 次異常判斷
- 通訊訊號狀態：曾出現 3 次異常判斷
- HRM水量設定：曾出現 3 次異常判斷
- PI11夾輥作動夾持秒：曾出現 2 次異常判斷

二、SOP Agent：標準處理流程
【SOP Agent｜強化版 SOP 流程建議】
查詢設備：HRM
異常狀況：打滑

找到對應 SOP：1 份

【SOP 使用原則】
1. 先確認異常是否仍持續發生，避免對已消失的短暫異常過度處置。
2. 涉及人員安全、設備保護或可能擴大損壞時，應先停機或通知班長。
3. 可自動判斷的項目，應優先比對系統數值與標準範圍。
4. 需要人工判斷的項目，應留下照片、備註或現場確認紀錄。
5. 若依 SOP 處置後仍未改善，應升級通知製程工程師與設備工程師。

【SOP001｜HRM打滑】
SOP 說明：HRM打滑異常處理SOP，含加熱、夾輥、水量、咬入點、入口檔板等判斷
主要負責角色：軋製課/現場班長
版本：v1.0
狀態：active

【1. 異常確認】
1. 根節點
   執行內容：HRM打滑
   判斷方式：人工確認
   監控項目：無，需由現場人員判斷
   

### 說明

多 Agent 先各自完成診斷、SOP、人工輔助、備件與通報任務，最後再交給 LLM 產生自然語言診斷摘要。


## 先做不用 API 的最終生成式摘要

本段會建立不使用外部 API 的最終摘要版本。

它會整合多 Agent 的輸出，產生可直接展示的異常診斷摘要。


In [67]:
# Step 60：不用 API 的最終摘要產生器

def final_demo_generative_summary(equipment_code, state_name):
    """
    不使用外部 API 的生成式摘要 Demo。
    加入 Human Assistance Agent 後，讓最終摘要不只顯示 AI 判斷，
    也顯示人工可以實際檢查與處置的項目。
    """

    result = multi_agent_diagnosis(equipment_code, state_name)

    diagnosis = result["diagnosis"]
    sop = result["sop"]
    human = result["human"]
    parts = result["parts"]
    notification = result["notification"]

    lines = []
    lines.append("【最終生成式 AI 診斷摘要 Demo】")
    lines.append("")

    lines.append("一、異常概況")
    lines.append(
        f"本次查詢設備為 {equipment_code}，異常狀況為「{state_name}」。"
        f"系統已根據歷史異常事件、SOP 流程、感測器資料、人工處置經驗與原因分類進行綜合分析。"
    )
    lines.append("")

    lines.append("二、初步判斷")
    if diagnosis["status"] == "ok" and diagnosis.get("possible_causes"):
        cause_text = "、".join([item["cause"] for item in diagnosis["possible_causes"]])
        lines.append(f"依據歷史案例統計，可能原因包含：{cause_text}。")
        lines.append(
            f"系統共找到 {diagnosis['matched_event_count']} 筆相似事件，"
            f"平均停機時間約 {diagnosis['avg_downtime']:.1f} 分鐘。"
        )
    else:
        lines.append("目前歷史資料不足，需依現場檢查結果進一步判斷。")
    lines.append("")

    lines.append("三、標準 SOP 流程")
    if sop["status"] == "ok":
        lines.append(f"系統找到 {sop['sop_count']} 份對應 SOP，建議依 SOP 階段式流程進行檢查。")
    else:
        lines.append("目前尚未找到完整 SOP，需由現場人員依經驗處理並補回紀錄。")
    lines.append("")

    lines.append("四、人工輔助檢查與處置")
    if human["status"] == "ok":
        assist_df = human["assist_df"]

        for idx, row in assist_df.head(5).reset_index(drop=True).iterrows():
            check_item = row.get("check_item", "依現場狀況確認")
            human_action = row.get("human_action", "依班長或工程師指示處理")
            evidence = row.get("evidence_required", "人工確認紀錄")

            lines.append(f"{idx + 1}. 人工需確認：{check_item}")
            lines.append(f"   建議處置：{human_action}")
            lines.append(f"   佐證資料：{evidence}")
            lines.append("")
    else:
        lines.append("目前缺少對應人工輔助資料，建議將本次人工處置經驗回填至知識庫。")
    lines.append("")

    lines.append("五、備件與物料確認")
    if parts["status"] == "ok" and parts.get("parts"):
        part_text = "、".join([item["part"] for item in parts["parts"]])
        lines.append(f"建議優先確認以下備件或物料：{part_text}。")
    else:
        lines.append("目前資料中沒有明確備件線索，建議由維修單位依現場狀況確認。")
    lines.append("")

    lines.append("六、通報建議")
    notify_units = notification.get("notify_units", [])
    if notify_units:
        lines.append("建議通報單位：" + "、".join(notify_units) + "。")
    else:
        lines.append("建議至少通報設備維修單位與製程工程單位。")

    lines.append("")
    lines.append("七、後續資料改善建議")
    lines.append("1. 將人工檢查結果回填到 event_step_check。")
    lines.append("2. 將有效處置方式回填到 abnormal_event 的 action_summary。")
    lines.append("3. 若人工檢查項目多次出現，應補入 sop_step 成為正式 SOP 節點。")
    lines.append("4. 若某些人工檢查可被數值化，應新增 monitor_function 與 parameter_spec。")

    return "\n".join(lines)


## 測試最終摘要

本段會測試最終摘要能否根據設備與異常狀況產生完整內容。


In [68]:
# Step 61：測試最終生成式摘要 Demo

print(final_demo_generative_summary("HRM", "打滑"))

【最終生成式 AI 診斷摘要 Demo】

一、異常概況
本次查詢設備為 HRM，異常狀況為「打滑」。系統已根據歷史異常事件、SOP 流程、感測器資料、人工處置經驗與原因分類進行綜合分析。

二、初步判斷
依據歷史案例統計，可能原因包含：人工操作疏失、原料/鋼種條件差異、設備磨耗。
系統共找到 12 筆相似事件，平均停機時間約 40.6 分鐘。

三、標準 SOP 流程
系統找到 1 份對應 SOP，建議依 SOP 階段式流程進行檢查。

四、人工輔助檢查與處置
1. 人工需確認：判斷是否持續打滑
   建議處置：當支退爐、在軋一支是否打滑
   佐證資料：安全確認紀錄

2. 人工需確認：持續觀察
   建議處置：持續軋延分析打滑鋼種是否異常
   佐證資料：人工確認備註

3. 人工需確認：修正加熱條件
   建議處置：依據加熱溫度管制作業
   佐證資料：人工確認備註

4. 人工需確認：檢查水量與噴嘴
   建議處置：HRM水量設定與噴嘴阻塞與角度目視檢查
   佐證資料：系統參數截圖 / 感測數值、現場照片

5. 人工需確認：調整夾持秒數
   建議處置：依據鋼種夾持規定秒數或視情況增加秒數
   佐證資料：人工確認備註


五、備件與物料確認
建議優先確認以下備件或物料：軋輥、噴嘴、皮帶/傳動件、加熱爐相關零件。

六、通報建議
建議通報單位：設備維修單位、製程工程單位、班長。

七、後續資料改善建議
1. 將人工檢查結果回填到 event_step_check。
2. 將有效處置方式回填到 abnormal_event 的 action_summary。
3. 若人工檢查項目多次出現，應補入 sop_step 成為正式 SOP 節點。
4. 若某些人工檢查可被數值化，應新增 monitor_function 與 parameter_spec。


## 把結果存成文字檔

本段會把診斷結果輸出成文字檔。

這可以作為報告附件，也可以讓使用者下載 AI 產生的異常診斷建議。


In [69]:
# Step 62：輸出診斷報告文字檔

equipment_code = "HRM"
state_name = "打滑"

report_text = final_demo_generative_summary(equipment_code, state_name)

with open("output/multi_agent_diagnosis_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print("已輸出：output/multi_agent_diagnosis_report.txt")

已輸出：output/multi_agent_diagnosis_report.txt


In [70]:
from google.colab import files
files.download("output/multi_agent_diagnosis_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 說明

本專題設計多 Agent 協作架構，將製造設備異常診斷任務拆分為五個角色。Diagnosis Agent 負責根據歷史異常事件與感測器資料判斷可能原因；SOP Agent 負責查詢並整理對應的標準作業流程，且將 SOP 從單純表格強化為階段式流程；Human Assistance Agent 則根據原始人工檢查表與案例筆記，整理現場人員可以檢查什麼、怎麼處理、需要留下什麼佐證，以及什麼情況需要升級通報；Parts Agent 根據 SOP 與歷史處理內容推估可能需要確認的備件；Notification Agent 則產生正式的異常通報內容。

多 Agent 架構的優點在於每個 Agent 負責明確任務，能降低單一模型直接生成答案時的混亂與幻覺風險。系統會先透過 GraphRAG 查詢設備、異常狀況、SOP、監控項目與歷史案例，再將查詢結果分別交由各 Agent 處理，最後整合成完整診斷摘要。此設計使生成式 AI 不只是文字生成工具，而是能結合企業內部知識庫、標準 SOP 與人工現場經驗進行輔助診斷與通報的應用系統。


## 先確認 output 檔案都有存好

本段會確認輸出的文字檔與資料檔是否已成功儲存。


In [71]:
import os

print("目前資料夾：", os.listdir())

if os.path.exists("output"):
    print("output 內容：")
    print(os.listdir("output"))
else:
    print("沒有 output 資料夾，請先回前面步驟輸出 clean_event_view 等資料。")

目前資料夾： ['.config', 'output', '資料集.xlsx', 'sample_data']
output 內容：
['clean_sop_view.csv', 'clean_sensor_view.csv', 'graph_edges.csv', 'graph_nodes.csv', 'human_assist_view.csv', 'multi_agent_diagnosis_report.txt', 'sop_detail_view.csv', 'clean_event_view.csv']


## 建立 Streamlit 主程式 app.py

本段會建立 Streamlit 主程式 `app.py`。

這是把 notebook 中的資料處理、查詢、看板與 AI 診斷邏輯整合成互動式網頁系統的關鍵步驟。


> v15 更新：將「互動式知識圖譜」改為穩定的 Plotly 視覺化，並補上用途說明與關係篩選，避免 pyvis 在 Streamlit iframe 中顯示空白。

In [72]:
%%writefile app.py
import os
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
import html
import math
from datetime import datetime

# =========================================================
# Streamlit 基本設定
# =========================================================

st.set_page_config(
    page_title="生成式 AI 製造設備異常診斷系統",
    page_icon="🛠️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =========================================================
# 欄位設定
# =========================================================

EVENT_COLS = [
    "event_id",
    "occurred_at",
    "equipment_code",
    "equipment_name",
    "state_name",
    "severity",
    "downtime_min",
    "root_cause_category",
    "cause_summary",
    "action_summary",
    "sop_id",
    "sop_name"
]

SOP_COLS = [
    "sop_id",
    "sop_name",
    "sop_desc",
    "equipment_code",
    "equipment_name",
    "state_name",
    "step_id",
    "parent_step_id",
    "owner_role",
    "sort_order",
    "branch_label",
    "step_title",
    "step_content",
    "check_type",
    "monitor_id",
    "monitor_name",
    "standard_text",
    "image_needed",
    "safety_note",
    "sop_stage",
    "check_method_label",
    "evidence_required",
    "next_action_hint"
]

SENSOR_COLS = [
    "snapshot_id",
    "event_id",
    "captured_at",
    "monitor_id",
    "monitor_name",
    "parameter_name",
    "actual_value",
    "unit",
    "spec_lower",
    "spec_upper",
    "judgement",
    "source_system",
    "source_note"
]

HUMAN_ASSIST_COLS = [
    "source_table",
    "equipment_code",
    "state_keyword",
    "assist_type",
    "check_item",
    "human_action",
    "related_monitor_id",
    "evidence_required",
    "escalation_rule",
    "source_note"
]


# =========================================================
# 安全讀檔函式
# =========================================================

def safe_read_csv(path, usecols=None):
    """
    安全讀取 CSV。
    如果指定欄位不存在，會自動忽略不存在欄位。
    """
    if not os.path.exists(path):
        return pd.DataFrame()

    try:
        if usecols is None:
            return pd.read_csv(path)

        return pd.read_csv(
            path,
            usecols=lambda col: col in usecols
        )
    except Exception:
        return pd.read_csv(path)


# =========================================================
# 讀取資料
# =========================================================

@st.cache_data(show_spinner=False)
def load_data():
    clean_event_view = safe_read_csv(
        "output/clean_event_view.csv",
        usecols=EVENT_COLS
    )

    # 優先讀取強化 SOP 表，若不存在則讀原本 clean_sop_view
    clean_sop_view = safe_read_csv(
        "output/sop_detail_view.csv",
        usecols=SOP_COLS
    )

    if clean_sop_view.empty:
        clean_sop_view = safe_read_csv(
            "output/clean_sop_view.csv",
            usecols=SOP_COLS
        )

    clean_sensor_view = safe_read_csv(
        "output/clean_sensor_view.csv",
        usecols=SENSOR_COLS
    )

    human_assist_view = safe_read_csv(
        "output/human_assist_view.csv",
        usecols=HUMAN_ASSIST_COLS
    )

    graph_nodes = None
    graph_edges = None

    if os.path.exists("output/graph_nodes.csv"):
        graph_nodes = safe_read_csv("output/graph_nodes.csv")

    if os.path.exists("output/graph_edges.csv"):
        graph_edges = safe_read_csv("output/graph_edges.csv")

    for col in ["equipment_code", "state_name", "severity", "root_cause_category"]:
        if col in clean_event_view.columns:
            clean_event_view[col] = clean_event_view[col].astype(str)

    for col in ["equipment_code", "state_name", "sop_id", "step_id", "parent_step_id"]:
        if col in clean_sop_view.columns:
            clean_sop_view[col] = clean_sop_view[col].astype(str).replace("nan", pd.NA)

    for col in ["equipment_code", "state_keyword", "check_item", "human_action"]:
        if col in human_assist_view.columns:
            human_assist_view[col] = human_assist_view[col].astype(str).replace("nan", pd.NA)

    if "downtime_min" in clean_event_view.columns:
        clean_event_view["downtime_min"] = pd.to_numeric(
            clean_event_view["downtime_min"],
            errors="coerce"
        )

    if "occurred_at" in clean_event_view.columns:
        clean_event_view["occurred_at"] = pd.to_datetime(
            clean_event_view["occurred_at"],
            errors="coerce"
        )

    if "captured_at" in clean_sensor_view.columns:
        clean_sensor_view["captured_at"] = pd.to_datetime(
            clean_sensor_view["captured_at"],
            errors="coerce"
        )

    for col in ["actual_value", "spec_lower", "spec_upper"]:
        if col in clean_sensor_view.columns:
            clean_sensor_view[col] = pd.to_numeric(clean_sensor_view[col], errors="coerce")

    for col in ["monitor_name", "parameter_name", "judgement", "unit"]:
        if col in clean_sensor_view.columns:
            clean_sensor_view[col] = clean_sensor_view[col].astype(str).replace("nan", pd.NA)

    if "sort_order" in clean_sop_view.columns:
        clean_sop_view["sort_order"] = pd.to_numeric(
            clean_sop_view["sort_order"],
            errors="coerce"
        )

    return clean_event_view, clean_sop_view, clean_sensor_view, graph_nodes, graph_edges, human_assist_view


with st.spinner("載入製造異常資料中..."):
    clean_event_view, clean_sop_view, clean_sensor_view, graph_nodes, graph_edges, human_assist_view = load_data()


# =========================================================
# 預先彙總 Dashboard 統計
# =========================================================

@st.cache_data(show_spinner=False)
def build_dashboard_summary(event_df):
    summary = {}

    summary["total_events"] = len(event_df)

    if "equipment_code" in event_df.columns:
        summary["equipment_count"] = event_df["equipment_code"].nunique()

        top_equipment = (
            event_df["equipment_code"]
            .dropna()
            .value_counts()
            .head(10)
            .reset_index()
        )
        top_equipment.columns = ["equipment_code", "count"]
    else:
        summary["equipment_count"] = 0
        top_equipment = pd.DataFrame(columns=["equipment_code", "count"])

    if "state_name" in event_df.columns:
        top_state = (
            event_df["state_name"]
            .dropna()
            .value_counts()
            .head(10)
            .reset_index()
        )
        top_state.columns = ["state_name", "count"]
    else:
        top_state = pd.DataFrame(columns=["state_name", "count"])

    if "severity" in event_df.columns:
        severity_df = (
            event_df["severity"]
            .dropna()
            .value_counts()
            .reset_index()
        )
        severity_df.columns = ["severity", "count"]
    else:
        severity_df = pd.DataFrame(columns=["severity", "count"])

    if "root_cause_category" in event_df.columns:
        cause_df = (
            event_df["root_cause_category"]
            .dropna()
            .value_counts()
            .reset_index()
        )
        cause_df.columns = ["root_cause_category", "count"]
    else:
        cause_df = pd.DataFrame(columns=["root_cause_category", "count"])

    if "downtime_min" in event_df.columns:
        summary["avg_downtime"] = event_df["downtime_min"].mean()
        summary["max_downtime"] = event_df["downtime_min"].max()
    else:
        summary["avg_downtime"] = None
        summary["max_downtime"] = None

    if "occurred_at" in event_df.columns:
        trend_df = event_df.dropna(subset=["occurred_at"]).copy()

        if len(trend_df) > 0:
            trend_df["month"] = trend_df["occurred_at"].dt.to_period("M").astype(str)

            monthly_count = (
                trend_df
                .groupby("month")
                .size()
                .reset_index(name="count")
                .sort_values("month")
            )
        else:
            monthly_count = pd.DataFrame(columns=["month", "count"])
    else:
        monthly_count = pd.DataFrame(columns=["month", "count"])

    return summary, top_equipment, top_state, severity_df, cause_df, monthly_count


dashboard_summary, top_equipment_df, top_state_df, severity_df, cause_df, monthly_count_df = build_dashboard_summary(clean_event_view)


# =========================================================
# 加強版：相似案例推薦與 SOP 改善分析
# =========================================================

def get_recommended_cases(equipment_code, state_name, top_n=5):
    """
    依同設備、同異常狀況、處置紀錄與停機時間排序，推薦最值得參考的歷史案例。
    """
    events = get_matched_events(equipment_code, state_name).copy()

    if events.empty:
        return pd.DataFrame()

    score = pd.Series(0, index=events.index, dtype="float")

    if "equipment_code" in events.columns:
        score += (events["equipment_code"].astype(str) == str(equipment_code)).astype(int) * 3

    if "state_name" in events.columns:
        score += (events["state_name"].astype(str) == str(state_name)).astype(int) * 3

    if "action_summary" in events.columns:
        action_text = events["action_summary"].fillna("").astype(str)
        score += action_text.str.len().clip(upper=100) / 100

        success_keywords = "復機|改善|恢復|完成|排除|正常|更換|調整|校正"
        score += action_text.str.contains(success_keywords, regex=True, na=False).astype(int) * 2

    if "downtime_min" in events.columns:
        downtime = pd.to_numeric(events["downtime_min"], errors="coerce")
        if downtime.notna().any():
            # 停機時間較短代表該案例可能較有效，作為小幅加分。
            max_dt = downtime.max()
            min_dt = downtime.min()
            if pd.notna(max_dt) and pd.notna(min_dt) and max_dt != min_dt:
                score += (1 - (downtime - min_dt) / (max_dt - min_dt)).fillna(0)
            events["_downtime_for_sort"] = downtime
        else:
            events["_downtime_for_sort"] = pd.NA
    else:
        events["_downtime_for_sort"] = pd.NA

    events["_recommend_score"] = score.round(2)

    sort_cols = ["_recommend_score"]
    ascending = [False]
    if "_downtime_for_sort" in events.columns:
        sort_cols.append("_downtime_for_sort")
        ascending.append(True)

    return events.sort_values(sort_cols, ascending=ascending).head(top_n)


def build_formal_ai_report(equipment_code, state_name, result):
    """
    將多 Agent 結果整理成老師與現場人員較容易閱讀的正式診斷報告格式。
    """
    diagnosis = result.get("diagnosis", {})
    sop = result.get("sop", {})
    human = result.get("human", {})
    parts = result.get("parts", {})
    notification = result.get("notification", {})

    possible_causes = diagnosis.get("possible_causes", [])
    matched_event_count = diagnosis.get("matched_event_count", 0)
    avg_downtime = diagnosis.get("avg_downtime", None)

    if possible_causes:
        cause_lines = []
        for idx, item in enumerate(possible_causes, start=1):
            cause_lines.append(f"{idx}. {item.get('cause', '未分類原因')}：{item.get('evidence', '歷史事件曾出現')}")
    else:
        cause_lines = ["1. 目前資料不足，尚無法判定明確主因，需先補齊現場觀察與感測器紀錄。"]

    if avg_downtime is not None and pd.notna(avg_downtime):
        downtime_text = f"相似事件 {matched_event_count} 筆，平均停機約 {avg_downtime:.1f} 分鐘。"
    else:
        downtime_text = f"相似事件 {matched_event_count} 筆，停機時間資料不足。"

    if human.get("assist_df") is not None and not human["assist_df"].empty:
        manual_df = human["assist_df"].head(4)
        manual_lines = []
        for idx, row in manual_df.iterrows():
            manual_lines.append(f"- {row.get('check_item', '依現場狀況確認')}；建議佐證：{row.get('evidence_required', '現場紀錄或照片')}")
    else:
        manual_lines = [
            "- 確認現場異常是否仍持續，並留下照片或班長確認紀錄。",
            "- 確認感測器是否有 loss、資料缺漏或異常跳動。",
            "- 確認 SOP 是否符合實際作業方式，若不符需回饋修訂。"
        ]

    if parts.get("status") == "ok" and parts.get("parts"):
        parts_lines = [f"- {item.get('part', '待確認備件')}：{item.get('reason', '依歷史處置或 SOP 關鍵字推估')}" for item in parts["parts"][:4]]
    else:
        parts_lines = ["- 目前沒有明確備件線索，需由維修單位依現場狀況確認。"]

    notify_units = notification.get("notify_units", [])
    notify_text = "、".join(notify_units) if notify_units else "設備維修單位、製程工程單位"

    lines = []
    lines.append("【正式 AI 異常診斷報告】")
    lines.append("")
    lines.append("一、異常概況")
    lines.append(f"- 設備：{equipment_code}")
    lines.append(f"- 異常狀況：{state_name}")
    lines.append(f"- 歷史資料基礎：{downtime_text}")
    lines.append("")
    lines.append("二、可能原因排序")
    lines.extend(cause_lines)
    lines.append("")
    lines.append("三、建議處置順序")
    lines.append("1. 先確認異常是否仍持續發生，避免處理已恢復但未紀錄的事件。")
    lines.append("2. 依對應 SOP 檢查關鍵節點，優先檢查可由感測器判斷的項目。")
    lines.append("3. 對 AI 無法自動判斷的節點，要求現場人員補上照片、量測值或備註。")
    lines.append("4. 比對下方相似案例，優先參考過去曾成功復機且停機時間較短的處置方式。")
    lines.append("5. 完成處置後回填原因、動作與復機結果，作為後續 SOP 改善依據。")
    lines.append("")
    lines.append("四、需要人工確認的項目")
    lines.extend(manual_lines)
    lines.append("")
    lines.append("五、備件或物料確認方向")
    lines.extend(parts_lines)
    lines.append("")
    lines.append("六、建議通報對象")
    lines.append(f"- {notify_text}")
    lines.append("")
    lines.append("七、使用限制")
    lines.append("- 本報告為生成式 AI 輔助整理，不取代工程師、安全人員與現場主管的判斷。")
    lines.append("- 若感測器資料缺漏、現場環境不明或 SOP 與實況不一致，應以人工確認結果為準。")

    return "\n".join(lines)


def build_sop_improvement_tables(event_df, sop_df):
    """
    建立 SOP 改善頁面需要的三種分析表：
    1. 停機時間較高的設備與異常組合
    2. 最常需要人工確認的 SOP 步驟
    3. 最常出現感測器 / 規則異常的設備異常組合
    """
    if event_df.empty:
        high_downtime = pd.DataFrame()
        repeated_abnormal = pd.DataFrame()
    else:
        high_downtime = event_df.copy()
        if "downtime_min" in high_downtime.columns:
            high_downtime["downtime_min"] = pd.to_numeric(high_downtime["downtime_min"], errors="coerce")
            group_cols = [c for c in ["equipment_code", "state_name"] if c in high_downtime.columns]
            if group_cols:
                high_downtime = (
                    high_downtime
                    .groupby(group_cols, dropna=False)
                    .agg(
                        event_count=("event_id", "count") if "event_id" in high_downtime.columns else ("downtime_min", "size"),
                        avg_downtime_min=("downtime_min", "mean"),
                        max_downtime_min=("downtime_min", "max")
                    )
                    .reset_index()
                    .sort_values(["avg_downtime_min", "event_count"], ascending=[False, False])
                    .head(15)
                )
            else:
                high_downtime = pd.DataFrame()
        else:
            high_downtime = pd.DataFrame()

        if "root_cause_category" in event_df.columns:
            group_cols = [c for c in ["equipment_code", "state_name", "root_cause_category"] if c in event_df.columns]
            if group_cols:
                repeated_abnormal = (
                    event_df
                    .groupby(group_cols, dropna=False)
                    .size()
                    .reset_index(name="event_count")
                    .sort_values("event_count", ascending=False)
                    .head(15)
                )
            else:
                repeated_abnormal = pd.DataFrame()
        else:
            repeated_abnormal = pd.DataFrame()

    if sop_df.empty:
        manual_steps = pd.DataFrame()
    else:
        manual_df = sop_df.copy()
        check_cols = [c for c in ["check_type", "image_needed", "evidence_required", "owner_role"] if c in manual_df.columns]

        def _manual_score(row):
            text = " ".join([str(row.get(c, "")) for c in check_cols])
            score = 0
            if any(k in text for k in ["人工", "目視", "照片", "拍照", "確認", "現場", "手動"]):
                score += 1
            if str(row.get("image_needed", "")).lower() in ["y", "yes", "true", "1", "需要"]:
                score += 1
            if pd.notna(row.get("evidence_required", pd.NA)) and str(row.get("evidence_required", "")).strip() not in ["", "nan", "None"]:
                score += 1
            return score

        manual_df["_manual_score"] = manual_df.apply(_manual_score, axis=1)
        manual_steps = manual_df[manual_df["_manual_score"] > 0].copy()

        display_cols = [
            "equipment_code", "state_name", "sop_id", "sop_name", "step_id",
            "step_title", "check_type", "owner_role", "evidence_required", "_manual_score"
        ]
        existing_cols = [c for c in display_cols if c in manual_steps.columns]
        manual_steps = (
            manual_steps[existing_cols]
            .sort_values("_manual_score", ascending=False)
            .head(20)
        )

    return high_downtime, manual_steps, repeated_abnormal



# =========================================================
# 共用查詢函式
# =========================================================

def get_matched_events(equipment_code=None, state_name=None):
    df = clean_event_view

    if equipment_code and equipment_code != "全部" and "equipment_code" in df.columns:
        df = df[df["equipment_code"] == str(equipment_code)]

    if state_name and state_name != "全部" and "state_name" in df.columns:
        df = df[df["state_name"] == str(state_name)]

    return df.copy()


def get_sop_by_equipment_state(equipment_code=None, state_name=None):
    events = get_matched_events(equipment_code, state_name)

    if events.empty:
        return pd.DataFrame()

    if "sop_id" not in events.columns or "sop_id" not in clean_sop_view.columns:
        return pd.DataFrame()

    sop_ids = events["sop_id"].dropna().unique().tolist()

    sop_df = clean_sop_view[
        clean_sop_view["sop_id"].isin(sop_ids)
    ].copy()

    if sop_df.empty:
        return sop_df

    if "sort_order" in sop_df.columns:
        return sop_df.sort_values(["sop_id", "sort_order"])

    return sop_df.sort_values(["sop_id"])


def get_sensor_by_events(event_ids):
    if "event_id" not in clean_sensor_view.columns:
        return pd.DataFrame()

    return clean_sensor_view[
        clean_sensor_view["event_id"].isin(event_ids)
    ].copy()




def _clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def _escape_dot(value):
    text = _clean_text(value)
    text = text.replace("\\", "\\\\").replace('"', '\\"')
    text = text.replace("\n", "\\n")
    return text


def classify_sop_stage(step_title, step_content):
    text = f"{step_title} {step_content}"

    if any(k in text for k in ["根節點", "確認是否", "判斷是否", "確認異常", "持續", "觀察", "是否仍"]):
        return "1. 異常確認"
    if any(k in text for k in ["安全", "停機", "隔離", "防護", "通知", "危險"]):
        return "2. 安全處置"
    if any(k in text for k in ["檢查", "確認", "比對", "量測", "查看", "判斷", "監控"]):
        return "3. 原因排查"
    if any(k in text for k in ["調整", "更換", "清除", "修正", "處理", "復歸", "退爐", "降速"]):
        return "4. 處置修正"
    if any(k in text for k in ["復機", "試軋", "恢復", "結案", "再確認", "OK"]):
        return "5. 復機確認"
    if any(k in text for k in ["紀錄", "回報", "上傳", "改善", "修訂", "回填"]):
        return "6. 紀錄回饋"
    return "3. 原因排查"


def build_sop_tree_dot(sop_df):
    """把 sop_step 的 parent_step_id / branch_label 轉成 Graphviz 樹狀圖。
    若原始資料只提供單一路徑，會補上「未完成 / 需升級」等合理分支，避免流程看起來像只有一條路。
    """
    if sop_df.empty or "step_id" not in sop_df.columns:
        return None

    df = sop_df.copy()
    if "sort_order" in df.columns:
        df = df.sort_values(["sort_order", "step_id"], na_position="last")

    valid_step_ids = set(df["step_id"].dropna().astype(str))

    lines = [
        "digraph SOP {",
        "rankdir=TB;",
        "graph [splines=ortho, nodesep=0.45, ranksep=0.65];",
        "node [shape=box, style=\"rounded,filled\", fillcolor=\"#EAF3FF\", color=\"#7EA6D8\", fontname=\"Microsoft JhengHei\", fontsize=11];",
        "edge [color=\"#6B7280\", fontname=\"Microsoft JhengHei\", fontsize=10];"
    ]

    fallback_ids = []
    row_by_id = {}
    child_labels_by_parent = {}

    for i, row in df.reset_index(drop=True).iterrows():
        step_id = _clean_text(row.get("step_id")) or f"STEP_AUTO_{i}"
        fallback_ids.append(step_id)
        row_by_id[step_id] = row
        title = _clean_text(row.get("step_title"))
        content = _clean_text(row.get("step_content"))
        stage = _clean_text(row.get("sop_stage")) or classify_sop_stage(title, content)
        check_type = _clean_text(row.get("check_method_label")) or _clean_text(row.get("check_type"))

        label_parts = [title]
        if content:
            label_parts.append(content[:38] + ("..." if len(content) > 38 else ""))
        if stage:
            label_parts.append(stage)
        if check_type:
            label_parts.append(f"判斷：{check_type}")

        label = "\\n".join([_escape_dot(x) for x in label_parts if x])
        lines.append(f'"{_escape_dot(step_id)}" [label="{label}"];')

    has_edge = False
    for _, row in df.iterrows():
        step_id = _clean_text(row.get("step_id"))
        parent = _clean_text(row.get("parent_step_id"))
        branch = _clean_text(row.get("branch_label"))

        if step_id and parent and parent in valid_step_ids and parent != step_id:
            label = f' [label="{_escape_dot(branch)}"]' if branch else ""
            lines.append(f'"{_escape_dot(parent)}" -> "{_escape_dot(step_id)}"{label};')
            child_labels_by_parent.setdefault(parent, []).append(branch)
            has_edge = True

    # 補齊缺漏分支：例如只有「是」沒有「否」、只有「完成」沒有「未完成」
    for parent, labels in child_labels_by_parent.items():
        if len(labels) == 1 and parent in row_by_id:
            missing_label = suggest_missing_branch_label(labels[0])
            virtual_id = f"VIRTUAL_{parent}_{abs(hash(missing_label)) % 100000}"
            parent_title = _clean_text(row_by_id[parent].get("step_title"))
            virtual_label = f"{missing_label}\\n需人工確認 / 升級處理\\n依終點分析結果處理"
            lines.append(f'"{_escape_dot(virtual_id)}" [label="{_escape_dot(virtual_label)}", fillcolor="#FFF7ED", color="#FDBA74"];')
            lines.append(f'"{_escape_dot(parent)}" -> "{_escape_dot(virtual_id)}" [label="{_escape_dot(missing_label)}"];')

    if not has_edge and len(fallback_ids) > 1:
        for a, b in zip(fallback_ids[:-1], fallback_ids[1:]):
            lines.append(f'"{_escape_dot(a)}" -> "{_escape_dot(b)}";')

    lines.append("}")
    return "\n".join(lines)

def _row_evidence(row):
    evidence = []
    if pd.notna(row.get("monitor_name")):
        evidence.append("系統參數截圖 / 感測數值")
    if str(row.get("image_needed", "")).lower() in ["true", "1", "yes", "y"]:
        evidence.append("現場照片")
    if pd.notna(row.get("safety_note")):
        evidence.append("安全確認紀錄")
    if not evidence:
        evidence.append("人工確認備註")
    return "、".join(evidence)


def _row_escalation(row):
    text = f"{row.get('step_title', '')} {row.get('step_content', '')}"
    if any(k in text for k in ["安全", "隔離", "停機", "危險"]):
        return "若涉及安全風險，立即停機並通知班長。"
    if any(k in text for k in ["軸承", "傳動", "馬達", "電流", "Guide", "Roll"]):
        return "若設備機構異常或處置後仍無法復機，通知設備工程師 / 保全單位。"
    if any(k in text for k in ["加熱", "溫度", "在爐", "鋼種", "製程"]):
        return "若製程條件與鋼種標準不一致，通知製程工程師確認。"
    if any(k in text for k in ["訊號", "PLC", "通訊", "sensor", "感測"]):
        return "若訊號異常或感測值不可信，通知自動化 / 設備單位確認。"
    return "若依此步驟確認後仍無法改善，通知班長並升級給製程或設備工程師。"


def _assist_match_score(row, state_name):
    state = _clean_text(state_name)
    keyword = _clean_text(row.get("state_keyword"))
    check_item = _clean_text(row.get("check_item"))
    human_action = _clean_text(row.get("human_action"))
    source_table = _clean_text(row.get("source_table"))

    score = 0
    if keyword == state:
        score += 100
    elif state and keyword and (state in keyword or keyword in state):
        score += 65
    if state and state in check_item:
        score += 25
    if state and state in human_action:
        score += 20
    if source_table == "12_sop_step" and keyword == state:
        score += 20
    return score


def build_sop_based_human_assist(equipment_code, state_name, max_items=8):
    sop_df = get_sop_by_equipment_state(equipment_code, state_name)
    if sop_df.empty:
        return pd.DataFrame(columns=HUMAN_ASSIST_COLS)

    records = []
    for _, row in sop_df.iterrows():
        title = _clean_text(row.get("step_title"))
        if title in ["", "根節點"]:
            continue
        check_type = _clean_text(row.get("check_type")).lower()
        is_manual = (
            check_type in ["manual", "hybrid"]
            or pd.isna(row.get("monitor_name"))
            or str(row.get("image_needed", "")).lower() in ["true", "1", "yes", "y"]
            or pd.notna(row.get("safety_note"))
        )
        if not is_manual:
            continue
        records.append({
            "source_table": "12_sop_step",
            "equipment_code": equipment_code,
            "state_keyword": state_name,
            "assist_type": _clean_text(row.get("sop_stage")) or classify_sop_stage(title, row.get("step_content", "")),
            "check_item": title,
            "human_action": _clean_text(row.get("step_content")),
            "related_monitor_id": row.get("monitor_id"),
            "evidence_required": row.get("evidence_required") if pd.notna(row.get("evidence_required")) else _row_evidence(row),
            "escalation_rule": _row_escalation(row),
            "source_note": f"來自 {row.get('sop_id', '')}｜{row.get('sop_name', '')}"
        })
    return pd.DataFrame(records).head(max_items)


def get_human_assistance(equipment_code, state_name, max_items=8):
    candidates = []

    if human_assist_view is not None and not human_assist_view.empty:
        df = human_assist_view.copy()
        df = df[df["equipment_code"].astype(str).str.upper() == str(equipment_code).upper()].copy()
        if not df.empty:
            df["match_score"] = df.apply(lambda row: _assist_match_score(row, state_name), axis=1)
            df = df[df["match_score"] > 0].sort_values("match_score", ascending=False)
            if not df.empty:
                candidates.append(df.drop(columns=["match_score"], errors="ignore"))

    # 永遠補上該設備、該異常的 SOP 人工確認項目，確保不同異常顯示不同內容
    sop_based = build_sop_based_human_assist(equipment_code, state_name, max_items=max_items)
    if not sop_based.empty:
        candidates.append(sop_based)

    if not candidates:
        return pd.DataFrame(columns=HUMAN_ASSIST_COLS)

    result = pd.concat(candidates, ignore_index=True)
    result = result.drop_duplicates(
        subset=["equipment_code", "state_keyword", "check_item", "human_action"],
        keep="first"
    )
    return result.head(max_items)


def human_assistance_agent(equipment_code, state_name, max_items=8):
    assist_df = get_human_assistance(equipment_code, state_name, max_items=max_items)

    lines = []
    lines.append("【Human Assistance Agent｜人工輔助檢查與處置】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append("")

    if assist_df.empty:
        lines.append("目前沒有找到此設備與異常狀況的專屬人工輔助資料。")
        lines.append("建議先依 SOP 執行人工確認，並將本次現場處置結果回填到異常事件紀錄。")
        return {
            "agent": "Human Assistance Agent",
            "status": "no_data",
            "text": "\n".join(lines),
            "assist_df": assist_df
        }

    lines.append("此 Agent 會依目前選定的設備與異常狀況，整理原始檔與 SOP 中可由人工執行的檢查、處置、佐證與升級條件。")
    lines.append("")

    for idx, row in assist_df.reset_index(drop=True).iterrows():
        lines.append(f"【人工輔助項目 {idx + 1}】")
        lines.append(f"類型：{row.get('assist_type', '人工輔助')}")
        lines.append(f"人工需確認：{row.get('check_item', '依現場狀況確認')}")
        lines.append(f"建議人工處置：{row.get('human_action', '依班長或工程師指示處理')}")

        if pd.notna(row.get("related_monitor_id")) and _clean_text(row.get("related_monitor_id")):
            lines.append(f"可搭配監控項目：{row.get('related_monitor_id')}")
        else:
            lines.append("可搭配監控項目：無明確監控項目，偏人工經驗判斷")

        lines.append(f"建議留下佐證：{row.get('evidence_required', '人工確認紀錄')}")
        lines.append(f"升級通報條件：{row.get('escalation_rule', '若處置後仍未改善，需升級通報')}")
        lines.append(f"資料來源：{row.get('source_note', row.get('source_table', '原始資料'))}")
        lines.append("")

    return {
        "agent": "Human Assistance Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "assist_df": assist_df
    }

def format_empty_message(agent_name):
    return f"【{agent_name}】目前資料不足，無法產生完整建議。"


# =========================================================
# Agent 1：Diagnosis Agent
# =========================================================

def diagnosis_agent(equipment_code, state_name, top_n_causes=3):
    events = get_matched_events(equipment_code, state_name)

    if events.empty:
        return {
            "agent": "Diagnosis Agent",
            "status": "no_data",
            "text": format_empty_message("Diagnosis Agent"),
            "possible_causes": [],
            "matched_event_count": 0,
            "avg_downtime": None
        }

    possible_causes = []

    if "root_cause_category" in events.columns:
        cause_counts = (
            events["root_cause_category"]
            .dropna()
            .value_counts()
            .head(top_n_causes)
        )

        for cause, count in cause_counts.items():
            possible_causes.append({
                "cause": cause,
                "count": int(count),
                "evidence": f"歷史相似事件中出現 {count} 次"
            })

    if "event_id" in events.columns:
        event_ids = events["event_id"].dropna().unique().tolist()
    else:
        event_ids = []

    sensor_df = get_sensor_by_events(event_ids)

    abnormal_sensors = pd.DataFrame()

    if not sensor_df.empty and "judgement" in sensor_df.columns:
        abnormal_sensors = sensor_df[
            sensor_df["judgement"].astype(str).str.contains(
                "異常|NG|超出|不正常|Fail|fail",
                case=False,
                na=False
            )
        ]

    if "downtime_min" in events.columns:
        avg_downtime = events["downtime_min"].mean()
        max_downtime = events["downtime_min"].max()
        min_downtime = events["downtime_min"].min()
    else:
        avg_downtime = None
        max_downtime = None
        min_downtime = None

    lines = []
    lines.append("【Diagnosis Agent｜異常診斷結果】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append(f"找到相似歷史事件：{len(events)} 筆")

    if avg_downtime is not None and pd.notna(avg_downtime):
        lines.append(f"平均停機時間：約 {avg_downtime:.1f} 分鐘")
        lines.append(f"停機時間範圍：{min_downtime:.0f} 至 {max_downtime:.0f} 分鐘")
    else:
        lines.append("平均停機時間：目前無可用資料")

    lines.append("")
    lines.append("可能原因排序：")

    if possible_causes:
        for idx, item in enumerate(possible_causes, start=1):
            lines.append(f"{idx}. {item['cause']}：{item['evidence']}")
    else:
        lines.append("目前歷史資料中沒有明確的原因分類。")

    lines.append("")
    lines.append("生成式診斷推理：")

    if possible_causes:
        cause_text = "、".join([item["cause"] for item in possible_causes])
        lines.append(
            f"根據相似歷史事件的原因分類，此次「{state_name}」異常較可能與「{cause_text}」相關。"
            "建議現場不要只依單一原因判斷，而應搭配 SOP、感測器資料與現場觀察逐項排查。"
        )
    else:
        lines.append(
            "目前原因分類資料不足，因此系統無法提出明確主因。"
            "建議先確認設備狀態、感測器訊號、近期保養紀錄與現場操作條件。"
        )

    lines.append("")
    lines.append("感測器異常觀察：")

    if abnormal_sensors.empty:
        lines.append("目前未從相似事件中整理出明確的感測器異常紀錄。")
    else:
        if "monitor_name" in abnormal_sensors.columns:
            sensor_summary = (
                abnormal_sensors["monitor_name"]
                .dropna()
                .value_counts()
                .head(5)
            )

            for monitor_name, count in sensor_summary.items():
                lines.append(f"- {monitor_name}：曾出現 {count} 次異常判斷")
        else:
            lines.append("感測器資料中有異常判斷，但缺少 monitor_name 欄位。")

    lines.append("")
    lines.append("建議下一步檢查：")
    lines.append("1. 確認異常是否仍持續發生。")
    lines.append("2. 比對 SOP 中的關鍵檢查點。")
    lines.append("3. 檢查感測器資料是否有 loss、超出規格或缺漏。")
    lines.append("4. 查詢設備近期是否有維修、保養或更換零件紀錄。")

    return {
        "agent": "Diagnosis Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "possible_causes": possible_causes,
        "matched_event_count": len(events),
        "avg_downtime": avg_downtime
    }


# =========================================================
# Agent 2：SOP Agent
# =========================================================

def sop_agent(equipment_code, state_name):
    sop_df = get_sop_by_equipment_state(equipment_code, state_name)

    if sop_df.empty:
        return {
            "agent": "SOP Agent",
            "status": "no_data",
            "text": format_empty_message("SOP Agent"),
            "sop_count": 0,
            "sop_df": sop_df
        }

    if "sop_stage" not in sop_df.columns:
        sop_df = sop_df.copy()
        sop_df["sop_stage"] = sop_df.apply(
            lambda row: classify_sop_stage(row.get("step_title", ""), row.get("step_content", "")),
            axis=1
        )

    lines = []
    lines.append("【SOP Agent｜階段式 SOP 流程建議】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append("")

    sop_count = sop_df["sop_id"].nunique() if "sop_id" in sop_df.columns else 0
    lines.append(f"找到對應 SOP：{sop_count} 份")
    lines.append("此版本不只列出 SOP 表格，而是依異常確認、原因排查、處置修正、復機確認與紀錄回饋等階段整理。")
    lines.append("")

    group_iterator = sop_df.groupby("sop_id") if "sop_id" in sop_df.columns else [("未知 SOP", sop_df)]

    for sop_id, group in group_iterator:
        first = group.iloc[0]
        lines.append(f"【{sop_id}｜{first.get('sop_name', '')}】")

        if pd.notna(first.get("sop_desc", None)):
            lines.append(f"SOP 說明：{first.get('sop_desc')}")
        if pd.notna(first.get("owner_role", None)):
            lines.append(f"負責角色：{first.get('owner_role')}")
        lines.append("")

        if "sop_stage" in group.columns:
            stage_groups = group.sort_values(["sop_stage", "sort_order"], na_position="last").groupby("sop_stage", sort=True)
        else:
            stage_groups = [("3. 原因排查", group.sort_values("sort_order", na_position="last"))]

        for stage, stage_df in stage_groups:
            lines.append(f"【{stage}】")
            for _, step in stage_df.iterrows():
                title = step.get("step_title", "")
                if str(title).strip() == "根節點":
                    continue
                sort_order = step.get("sort_order", "")
                branch_label = step.get("branch_label", "")
                prefix = f"{int(sort_order)}." if pd.notna(sort_order) else "-"
                if pd.notna(branch_label) and str(branch_label).strip() not in ["", "nan", "None"]:
                    prefix += f" [{branch_label}]"

                lines.append(f"{prefix} {title}：{step.get('step_content', '')}")
                if pd.notna(step.get("check_method_label", None)):
                    lines.append(f"   - 判斷方式：{step.get('check_method_label')}")
                elif pd.notna(step.get("check_type", None)):
                    lines.append(f"   - 判斷方式：{step.get('check_type')}")
                if pd.notna(step.get("monitor_name", None)):
                    lines.append(f"   - 監控項目：{step.get('monitor_name')}")
                if pd.notna(step.get("standard_text", None)):
                    lines.append(f"   - 判斷標準：{step.get('standard_text')}")
                if pd.notna(step.get("evidence_required", None)):
                    lines.append(f"   - 需留存佐證：{step.get('evidence_required')}")
                if pd.notna(step.get("next_action_hint", None)):
                    lines.append(f"   - 異常時建議：{step.get('next_action_hint')}")
                if pd.notna(step.get("safety_note", None)):
                    lines.append(f"   - 安全提醒：{step.get('safety_note')}")
            lines.append("")

    return {
        "agent": "SOP Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "sop_count": sop_count,
        "sop_df": sop_df
    }


# =========================================================
# Agent 3：Parts Agent
# =========================================================

def parts_agent(equipment_code, state_name):
    events = get_matched_events(equipment_code, state_name)
    sop_df = get_sop_by_equipment_state(equipment_code, state_name)

    if events.empty and sop_df.empty:
        return {
            "agent": "Parts Agent",
            "status": "no_data",
            "text": format_empty_message("Parts Agent"),
            "parts": []
        }

    text_pool = []

    if not events.empty:
        for col in ["state_name", "root_cause_category", "cause_summary", "action_summary"]:
            if col in events.columns:
                text_pool.extend(events[col].dropna().astype(str).tolist())

    if not sop_df.empty:
        for col in ["step_title", "step_content", "standard_text", "safety_note"]:
            if col in sop_df.columns:
                text_pool.extend(sop_df[col].dropna().astype(str).tolist())

    all_text = " ".join(text_pool)

    part_rules = {
        "軸承": ["軸承", "bearing"],
        "軋輥": ["軋輥", "roll", "Roll", "TC Roll"],
        "感測器": ["感測器", "sensor", "訊號", "signal", "loss"],
        "噴嘴": ["噴嘴", "水量", "冷卻", "堵塞"],
        "皮帶/傳動件": ["皮帶", "傳動", "打滑", "馬達", "鏈條"],
        "油壓/潤滑相關零件": ["油壓", "潤滑", "油", "壓力"],
        "電控元件": ["電流", "電壓", "PLC", "控制", "訊號"],
        "加熱爐相關零件": ["加熱", "爐", "溫度", "燃燒"]
    }

    recommended_parts = []

    for part_name, keywords in part_rules.items():
        matched_keywords = [kw for kw in keywords if kw in all_text]

        if matched_keywords:
            recommended_parts.append({
                "part": part_name,
                "matched_keywords": matched_keywords,
                "reason": f"相關資料中出現關鍵字：{', '.join(matched_keywords)}"
            })

    lines = []
    lines.append("【Parts Agent｜備件與物料確認建議】")
    lines.append(f"查詢設備：{equipment_code}")
    lines.append(f"異常狀況：{state_name}")
    lines.append("")

    if recommended_parts:
        lines.append("建議優先確認以下備件或物料：")

        for idx, item in enumerate(recommended_parts, start=1):
            lines.append(f"{idx}. {item['part']}")
            lines.append(f"   - 推估原因：{item['reason']}")
    else:
        lines.append("目前資料中沒有明確備件線索。")
        lines.append("建議至少確認：常用消耗品、設備專用備件、感測器與安全防護用品。")

    lines.append("")
    lines.append("生成式備件推理：")
    lines.append(
        "目前系統尚未接入正式庫存表，因此 Parts Agent 先根據 SOP 內容、歷史處置摘要與異常關鍵字推估可能需要確認的備件。"
        "此結果不是直接代表一定需要更換，而是作為維修人員進行現場確認與庫存查詢的優先方向。"
    )
    lines.append("")
    lines.append("若後續取得材料庫存資料，可再改成直接查詢庫存數量、安全庫存與補料需求。")

    return {
        "agent": "Parts Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "parts": recommended_parts
    }


# =========================================================
# Agent 4：Notification Agent
# =========================================================

def notification_agent(equipment_code, state_name, diagnosis_result=None, sop_result=None):
    events = get_matched_events(equipment_code, state_name)

    if events.empty:
        severity = "未判定"
        avg_downtime = None
        event_count = 0
    else:
        if "severity" in events.columns and not events["severity"].dropna().empty:
            severity = events["severity"].mode().iloc[0]
        else:
            severity = "未判定"

        if "downtime_min" in events.columns:
            avg_downtime = events["downtime_min"].mean()
        else:
            avg_downtime = None

        event_count = len(events)

    possible_causes = []

    if diagnosis_result and diagnosis_result.get("possible_causes"):
        possible_causes = [
            item["cause"] for item in diagnosis_result["possible_causes"]
        ]

    cause_text = "、".join(possible_causes) if possible_causes else "目前資料不足，需現場進一步確認"

    sop_count = 0

    if sop_result:
        sop_count = sop_result.get("sop_count", 0)

    notify_units = ["設備維修單位", "製程工程單位"]

    if str(severity).upper() == "A":
        notify_units.append("產線主管")
        notify_units.append("品質工程單位")
    elif str(severity).upper() == "B":
        notify_units.append("班長")

    if avg_downtime is not None and pd.notna(avg_downtime):
        downtime_text = f"系統比對 {event_count} 筆相似歷史事件，平均停機時間約 {avg_downtime:.1f} 分鐘。"
    else:
        downtime_text = "目前系統尚未找到足夠相似歷史事件，停機影響仍需由現場確認。"

    if sop_count > 0:
        sop_text = f"系統已找到 {sop_count} 份可能對應 SOP，建議依標準流程逐項確認。"
    else:
        sop_text = "目前未找到明確對應 SOP，建議由現場工程人員補充處理流程。"

    field_lines = []
    field_lines.append("【現場人員版｜立即處理提醒】")
    field_lines.append(f"設備 {equipment_code} 發生「{state_name}」異常。")
    field_lines.append("請先確認異常是否仍持續發生，並依 SOP 優先檢查設備狀態、關鍵參數與安全條件。")
    field_lines.append(f"初步可能原因：{cause_text}。")
    field_lines.append(sop_text)
    field_lines.append("")
    field_lines.append("建議現場先做：")
    field_lines.append("1. 確認設備是否仍處於異常狀態。")
    field_lines.append("2. 檢查感測器數值是否異常或資料是否缺漏。")
    field_lines.append("3. 依 SOP 檢查關鍵部位、製程條件與安全狀態。")
    field_lines.append("4. 處理完成後記錄原因、處置方式與復機結果。")

    field_version = "\n".join(field_lines)

    manager_lines = []
    manager_lines.append("【主管通報版｜異常影響摘要】")
    manager_lines.append(f"目前 {equipment_code} 發生「{state_name}」異常，系統依歷史資料判斷常見嚴重度為 {severity}。")
    manager_lines.append(downtime_text)
    manager_lines.append(f"初步可能原因包含：{cause_text}。")
    manager_lines.append(sop_text)
    manager_lines.append("")
    manager_lines.append("建議主管關注：")
    manager_lines.append("1. 是否造成產線停機或延誤。")
    manager_lines.append("2. 是否需要設備、製程與品質單位共同處理。")
    manager_lines.append("3. 是否為重複發生異常，需要後續改善 SOP 或保養策略。")
    manager_lines.append("")
    manager_lines.append("建議通報單位：" + "、".join(notify_units) + "。")

    manager_version = "\n".join(manager_lines)

    maintenance_lines = []
    maintenance_lines.append("【維修單位版｜設備檢查建議】")
    maintenance_lines.append(f"異常設備：{equipment_code}")
    maintenance_lines.append(f"異常狀況：{state_name}")
    maintenance_lines.append(f"可能原因方向：{cause_text}")
    maintenance_lines.append("")
    maintenance_lines.append("建議維修單位優先確認：")
    maintenance_lines.append("1. 相關機構是否磨耗、鬆動、卡滯或破損。")
    maintenance_lines.append("2. 感測器訊號、PLC / 通訊狀態是否正常。")
    maintenance_lines.append("3. SOP 中列出的關鍵檢查點是否符合標準。")
    maintenance_lines.append("4. 若需更換零件，請同步確認備件與安全庫存。")
    maintenance_lines.append("")
    maintenance_lines.append(sop_text)

    maintenance_version = "\n".join(maintenance_lines)

    human_check_items = [
        "現場異常是否仍持續發生，或已經恢復正常。",
        "感測器資料是否可信，例如是否有訊號 loss、資料缺漏或異常跳動。",
        "設備近期是否已有保養、維修、更換零件或異常紀錄。",
        "目前 SOP 是否仍符合最新產線狀態與現場作業方式。",
        "AI 產生的原因推測是否與現場人員觀察一致。"
    ]

    human_check_lines = ["【仍需人工確認事項】"]

    for idx, item in enumerate(human_check_items, start=1):
        human_check_lines.append(f"{idx}. {item}")

    human_check_text = "\n".join(human_check_lines)

    lines = []
    lines.append("【Notification Agent｜多角色通報內容】")
    lines.append("")
    lines.append("本 Agent 依據異常設備、歷史事件、嚴重度、可能原因與 SOP 狀態，生成三種不同對象的通報版本。")
    lines.append("")
    lines.append(field_version)
    lines.append("")
    lines.append(manager_version)
    lines.append("")
    lines.append(maintenance_version)
    lines.append("")
    lines.append(human_check_text)

    return {
        "agent": "Notification Agent",
        "status": "ok",
        "text": "\n".join(lines),
        "notify_units": notify_units,
        "severity": severity,
        "role_versions": {
            "現場人員版": field_version,
            "主管通報版": manager_version,
            "維修單位版": maintenance_version
        },
        "human_check_items": human_check_items,
        "human_check_text": human_check_text
    }


# =========================================================
# 多 Agent 總控
# =========================================================

def multi_agent_diagnosis(equipment_code, state_name):
    diagnosis = diagnosis_agent(equipment_code, state_name)
    sop = sop_agent(equipment_code, state_name)
    human = human_assistance_agent(equipment_code, state_name)
    parts = parts_agent(equipment_code, state_name)
    notification = notification_agent(
        equipment_code,
        state_name,
        diagnosis_result=diagnosis,
        sop_result=sop
    )

    return {
        "diagnosis": diagnosis,
        "sop": sop,
        "human": human,
        "parts": parts,
        "notification": notification
    }


def final_demo_generative_summary(equipment_code, state_name):
    result = multi_agent_diagnosis(equipment_code, state_name)

    diagnosis = result["diagnosis"]
    sop = result["sop"]
    human = result["human"]
    parts = result["parts"]
    notification = result["notification"]

    if diagnosis["status"] == "ok" and diagnosis.get("possible_causes"):
        cause_text = "、".join([item["cause"] for item in diagnosis["possible_causes"]])
        matched_event_count = diagnosis.get("matched_event_count", 0)
        avg_downtime = diagnosis.get("avg_downtime", None)
    else:
        cause_text = "目前資料不足，尚無法判定明確主因"
        matched_event_count = 0
        avg_downtime = None

    if avg_downtime is not None and pd.notna(avg_downtime):
        downtime_sentence = f"系統找到 {matched_event_count} 筆相似歷史事件，平均停機時間約 {avg_downtime:.1f} 分鐘。"
    else:
        downtime_sentence = "目前相似事件資料不足，停機時間影響仍需由現場進一步確認。"

    if sop["status"] == "ok":
        sop_sentence = f"系統找到 {sop['sop_count']} 份可能對應 SOP，並已整理為階段式處理流程。"
    else:
        sop_sentence = "目前尚未找到明確 SOP，建議後續補建此異常的標準處理流程。"

    if human["status"] == "ok":
        assist_count = len(human.get("assist_df", pd.DataFrame()))
        human_sentence = f"系統另外找到 {assist_count} 項與此異常相關的人工輔助檢查 / 處置建議。"
    else:
        human_sentence = "目前缺少專屬人工輔助資料，建議將本次現場處置經驗回填成新案例。"

    if parts["status"] == "ok" and parts.get("parts"):
        part_text = "、".join([item["part"] for item in parts["parts"]])
    else:
        part_text = "目前沒有明確備件線索，需由維修單位依現場狀況確認"

    notify_units = notification.get("notify_units", [])
    notify_text = "、".join(notify_units) if notify_units else "設備維修單位、製程工程單位"

    lines = []
    lines.append("【AI 生成式分析摘要】")
    lines.append("")
    lines.append(
        f"系統針對設備「{equipment_code}」發生「{state_name}」的情境，"
        "整合歷史異常事件、SOP 流程、感測器資料、人工輔助知識與原因分類後，產生以下診斷建議。"
    )
    lines.append("")
    lines.append(f"從歷史資料來看，此類異常較可能與「{cause_text}」有關。{downtime_sentence}")
    lines.append("")
    lines.append(
        f"在處理流程上，{sop_sentence}建議現場人員依 SOP 樹狀流程先確認異常是否仍持續，"
        "再依原因排查、處置修正、復機確認與紀錄回饋的順序處理。"
    )
    lines.append("")
    lines.append(
        f"人工輔助方面，{human_sentence}人工輔助內容會依目前選擇的設備與異常狀況動態篩選，"
        "不會把所有異常都套用同一套確認事項。"
    )
    lines.append("")
    lines.append(
        f"備件與物料方面，系統推估可優先確認：{part_text}。"
        "若後續能加入正式庫存資料，系統可進一步提供庫存數量、安全庫存與是否需補料的建議。"
    )
    lines.append("")
    lines.append(
        f"通報方面，建議至少通知：{notify_text}。"
        "若事件嚴重度較高或已造成停機，應同步通知主管與相關支援單位，以降低處理延誤。"
    )
    lines.append("")
    lines.append(
        "補充說明：本摘要屬於生成式 AI 輔助判斷，主要用於協助現場快速整理線索，"
        "不應取代工程師的現場判斷與安全確認。"
    )

    return "\n".join(lines), result




# =========================================================
# SOP 互動呈現輔助函式
# =========================================================

def normalize_branch_label(label, default="下一步"):
    text = _clean_text(label)
    if text in ["", "None", "nan"]:
        return default
    return text


def suggest_missing_branch_label(existing_label):
    """依現有分支補上常見的相反分支，避免互動流程只有單一路徑。"""
    label = normalize_branch_label(existing_label, default="").replace(" ", "")
    mapping = {
        "是": "否 / 未完成",
        "否": "是 / 已確認",
        "正常": "異常 / 未完成",
        "異常": "正常 / 已排除",
        "完成": "未完成 / 需升級",
        "未完成": "完成 / 可結案",
        "可修復": "不可修復 / 需升級",
        "不可修復": "可修復 / 完成處置",
        "可處理": "不可處理 / 需升級",
        "不可處理": "可處理 / 依 SOP 處置",
        "可復機": "不可復機 / 需升級",
        "不可復機": "可復機 / 結案回饋",
    }
    if label in mapping:
        return mapping[label]
    if "是" in label:
        return "否 / 未完成"
    if "否" in label:
        return "是 / 已確認"
    if "正常" in label:
        return "異常 / 未完成"
    if "異常" in label:
        return "正常 / 已排除"
    if "完成" in label or "可" in label:
        return "未完成 / 需升級"
    return "未完成 / 需人工判斷"


def make_virtual_terminal_step(parent_row, missing_label, state_name=""):
    """建立補齊用的終點節點：代表此分支尚未完成、無法確認或需要升級。"""
    parent_id = _clean_text(parent_row.get("_node_id")) or _clean_text(parent_row.get("step_id")) or "PARENT"
    virtual_id = f"VIRTUAL_{parent_id}_{abs(hash(missing_label)) % 100000}"
    parent_title = _clean_text(parent_row.get("step_title"))
    parent_content = _clean_text(parent_row.get("step_content"))

    virtual = dict(parent_row)
    virtual.update({
        "step_id": virtual_id,
        "_node_id": virtual_id,
        "parent_step_id": parent_id,
        "_parent_node_id": parent_id,
        "branch_label": missing_label,
        "sort_order": 999,
        "step_title": "未完成 / 需升級處理",
        "step_content": f"此分支代表「{parent_title}」尚未完成、條件不符合或現場無法確認。請先保留現場紀錄，再依終點分析結果處理。",
        "sop_stage": "5. 復機確認 / 升級處理",
        "check_type": "manual",
        "check_method_label": "人工確認",
        "monitor_name": "無，需由現場人員確認",
        "standard_text": "確認此步驟尚未完成、無法判定或不符合預期。",
        "evidence_required": "現場照片 / 人工確認備註 / 異常截圖 / 通報紀錄",
        "safety_note": "若有安全疑慮或設備風險，先停機並通知班長。",
        "next_action_hint": "若此分支被選到，代表流程未能正常完成；請通知班長，並視情況升級給設備工程師或製程工程師。",
        "_virtual_terminal": True,
        "_virtual_reason": f"原始 SOP 只有單一分支，系統補上「{missing_label}」作為未完成或例外處理路徑。",
    })
    return pd.Series(virtual)


def prepare_sop_nodes(sop_group):
    """將 SOP 資料轉成便於樹狀圖與逐步導覽使用的節點結構。
    會自動補齊常見缺漏分支，例如只有「是」時補上「否 / 未完成」。
    """
    df = sop_group.copy().reset_index(drop=True)

    if "sop_stage" not in df.columns:
        df["sop_stage"] = df.apply(
            lambda row: classify_sop_stage(row.get("step_title", ""), row.get("step_content", "")),
            axis=1
        )

    if "check_method_label" not in df.columns:
        def _check_label(x):
            x = _clean_text(x).lower()
            if x == "auto":
                return "系統自動判斷"
            if x == "hybrid":
                return "系統輔助 + 人工確認"
            return "人工確認"
        df["check_method_label"] = df.get("check_type", pd.Series([""] * len(df))).apply(_check_label)

    df = df.sort_values(["sort_order", "step_id"], na_position="last").reset_index(drop=True)

    node_ids = []
    raw_to_node = {}
    for idx, row in df.iterrows():
        raw_id = _clean_text(row.get("step_id"))
        node_id = raw_id if raw_id else f"STEP_AUTO_{idx}"
        node_ids.append(node_id)
        if raw_id:
            raw_to_node[raw_id] = node_id
    df["_node_id"] = node_ids

    parent_ids = []
    for idx, row in df.iterrows():
        parent_raw = _clean_text(row.get("parent_step_id"))
        parent_ids.append(raw_to_node.get(parent_raw, ""))
    df["_parent_node_id"] = parent_ids

    step_map = {row["_node_id"]: row for _, row in df.iterrows()}
    children_map = {node_id: [] for node_id in df["_node_id"]}
    roots = []

    for _, row in df.iterrows():
        node_id = row["_node_id"]
        parent_id = row["_parent_node_id"]
        if parent_id and parent_id in children_map and parent_id != node_id:
            children_map[parent_id].append(row)
        else:
            roots.append(node_id)

    if not roots and len(df) > 0:
        roots = [df.iloc[0]["_node_id"]]

    for parent_id in list(children_map.keys()):
        children_map[parent_id] = sorted(
            children_map[parent_id],
            key=lambda r: (999999 if pd.isna(r.get("sort_order")) else r.get("sort_order"), _clean_text(r.get("_node_id")))
        )

    # 補齊只有單一路徑的節點，讓互動流程可以處理「未完成 / 例外 / 需升級」
    for parent_id in list(children_map.keys()):
        children = children_map.get(parent_id, [])
        if len(children) == 1 and parent_id in step_map:
            current_title = _clean_text(step_map[parent_id].get("step_title"))
            if current_title != "根節點":
                existing_label = normalize_branch_label(children[0].get("branch_label"), default="下一步")
                missing_label = suggest_missing_branch_label(existing_label)
                virtual = make_virtual_terminal_step(step_map[parent_id], missing_label)
                vid = virtual.get("_node_id")
                step_map[vid] = virtual
                children_map.setdefault(vid, [])
                children_map[parent_id].append(virtual)

    return df, step_map, children_map, roots


def find_related_human_assistance(step_row, assist_df, state_name="", max_items=3):
    if assist_df is None or assist_df.empty:
        return pd.DataFrame(columns=HUMAN_ASSIST_COLS)

    title = _clean_text(step_row.get("step_title"))
    content = _clean_text(step_row.get("step_content"))
    stage = _clean_text(step_row.get("sop_stage"))

    candidates = assist_df.copy()

    def _score(row):
        score = 0
        for field in ["check_item", "human_action", "assist_type", "source_note"]:
            text = _clean_text(row.get(field))
            if title and title in text:
                score += 90
            if content and content[:12] and content[:12] in text:
                score += 25
            if stage and stage in text:
                score += 10
        if state_name and _clean_text(row.get("state_keyword")) == _clean_text(state_name):
            score += 15
        return score

    candidates["_score"] = candidates.apply(_score, axis=1)
    matched = candidates[candidates["_score"] > 0].sort_values("_score", ascending=False)

    if matched.empty:
        matched = candidates.head(max_items)
    else:
        matched = matched.drop(columns=["_score"], errors="ignore").head(max_items)

    return matched


def render_human_assistance_cards(assist_df, section_title="Human Assistance Agent 建議"):
    if assist_df is None or assist_df.empty:
        st.info("目前沒有對應的人工輔助建議。")
        return

    st.markdown(f"**{section_title}**")

    for idx, row in assist_df.reset_index(drop=True).iterrows():
        with st.container(border=True):
            st.markdown(f"##### 建議 {idx + 1}｜{row.get('assist_type', '人工輔助')}")
            st.write(f"**人工需確認：** {row.get('check_item', '依現場狀況確認')}")
            st.write(f"**建議人工處置：** {row.get('human_action', '依班長或工程師指示處理')}")
            st.write(f"**建議留下佐證：** {row.get('evidence_required', '人工確認紀錄')}")
            st.write(f"**升級通報條件：** {row.get('escalation_rule', '若處置後仍未改善，需升級通報')}")
            source_note = row.get("source_note", row.get("source_table", "原始資料"))
            if pd.notna(source_note) and _clean_text(source_note):
                st.caption(f"資料來源：{source_note}")


def render_step_detail_card(step, equipment_code, state_name, assist_df=None, prefix_text=None, show_assist=True):
    title = _clean_text(step.get("step_title"))
    if title == "根節點":
        return

    sort_order = step.get("sort_order", "")
    branch_label = normalize_branch_label(step.get("branch_label"), default="")

    if prefix_text:
        header = prefix_text
    else:
        if bool(step.get("_virtual_terminal", False)):
            header = "例外分支"
        else:
            header = f"步驟 {int(sort_order)}" if pd.notna(sort_order) else "步驟"
        if branch_label:
            header += f"｜分支：{branch_label}"

    st.markdown(f"#### {header}：{title}")
    st.write(step.get("step_content", ""))

    if bool(step.get("_virtual_terminal", False)):
        st.warning(step.get("_virtual_reason", "此節點為系統補齊的未完成 / 例外處理分支。"))

    c1, c2 = st.columns(2)

    with c1:
        st.markdown("**判斷方式**")
        check_label = step.get("check_method_label", None)
        check_type = _clean_text(step.get("check_type")).lower()
        if pd.notna(check_label) and _clean_text(check_label):
            st.info(check_label)
        elif check_type == "auto":
            st.success("系統自動判斷")
        elif check_type == "hybrid":
            st.warning("系統輔助 + 人工確認")
        else:
            st.info("人工確認")

        st.markdown("**監控項目**")
        if pd.notna(step.get("monitor_name", None)) and _clean_text(step.get("monitor_name")):
            st.write(step.get("monitor_name"))
        else:
            st.write("無，需由現場人員確認")

        st.markdown("**建議佐證資料**")
        if pd.notna(step.get("evidence_required", None)) and _clean_text(step.get("evidence_required")):
            st.write(step.get("evidence_required"))
        else:
            st.write("人工確認備註")

    with c2:
        st.markdown("**判斷標準**")
        if pd.notna(step.get("standard_text", None)) and _clean_text(step.get("standard_text")):
            st.write(step.get("standard_text"))
        else:
            st.write("目前未提供標準，需依現場經驗或主管指示")

        st.markdown("**安全提醒**")
        if pd.notna(step.get("safety_note", None)) and _clean_text(step.get("safety_note")):
            st.warning(step.get("safety_note"))
        else:
            st.write("無特別安全提醒")

        st.markdown("**異常時下一步建議**")
        if pd.notna(step.get("next_action_hint", None)) and _clean_text(step.get("next_action_hint")):
            st.write(step.get("next_action_hint"))
        else:
            st.write(_row_escalation(step))

    if show_assist and assist_df is not None:
        related_assist = find_related_human_assistance(step, assist_df, state_name=state_name, max_items=2)
        render_human_assistance_cards(related_assist, section_title="對應人工輔助建議")


def render_version_one(group, equipment_code, state_name):
    overall_assist_df = get_human_assistance(equipment_code, state_name, max_items=8)

    st.markdown("### 版本一：樹狀架構＋詳細內容卡片")
    st.caption("先用樹狀圖看整體分支，再往下看每一步的詳細內容與對應人工輔助建議。")

    dot = build_sop_tree_dot(group)
    if dot:
        st.graphviz_chart(dot, use_container_width=True)
    else:
        st.warning("此 SOP 缺少 step_id / parent_step_id，無法繪製樹狀圖。")

    st.markdown("---")
    st.markdown("### 步驟詳細說明")

    step_df, _, _, _ = prepare_sop_nodes(group)
    step_df = step_df[step_df["step_title"].astype(str).str.strip() != "根節點"]

    for stage, stage_df in step_df.groupby("sop_stage", sort=True):
        st.markdown(f"## {stage}")
        for _, step in stage_df.iterrows():
            with st.container(border=True):
                render_step_detail_card(step, equipment_code, state_name, assist_df=overall_assist_df)

    st.markdown("---")
    st.markdown("### 全部人工輔助建議總覽")
    render_human_assistance_cards(overall_assist_df, section_title="此異常狀況的人工輔助建議")


def render_version_two(group, equipment_code, state_name, sop_id):
    st.markdown("### 逐步 SOP 導覽")
    st.caption("先確認當前問題，再依『是 / 否 / 正常 / 異常 / 完成 / 未完成』等分支往下走；只有走到沒有下一步時，才顯示分析結果與人工輔助建議。")

    step_df, step_map, children_map, roots = prepare_sop_nodes(group)
    if len(step_df) == 0:
        st.warning("此 SOP 沒有可用步驟。")
        return

    visible_roots = [r for r in roots if _clean_text(step_map[r].get("step_title")) != "根節點"]
    if visible_roots:
        root_id = visible_roots[0]
    elif roots and children_map.get(roots[0]):
        root_id = children_map[roots[0]][0].get("_node_id")
    else:
        root_id = roots[0]

    state_key = f"wizard_{sop_id}_current"
    path_key = f"wizard_{sop_id}_path"

    if state_key not in st.session_state:
        st.session_state[state_key] = root_id
    if path_key not in st.session_state:
        st.session_state[path_key] = [root_id]

    current_id = st.session_state[state_key]
    if current_id not in step_map:
        st.session_state[state_key] = root_id
        st.session_state[path_key] = [root_id]
        current_id = root_id

    current_step = step_map[current_id]
    overall_assist_df = get_human_assistance(equipment_code, state_name, max_items=8)
    path_ids = st.session_state[path_key]
    children = children_map.get(current_id, [])

    st.info(f"目前確認項目：{_clean_text(current_step.get('step_title'))}")

    # 先呈現問題確認 / 目前步驟，不先丟分析結果，避免還沒走完流程就給結論
    with st.container(border=True):
        render_step_detail_card(
            current_step,
            equipment_code,
            state_name,
            assist_df=None,
            prefix_text="問題確認 / 目前步驟" if len(path_ids) == 1 else None,
            show_assist=False
        )

    path_titles = [
        _clean_text(step_map[x].get("step_title"))
        for x in path_ids if x in step_map and _clean_text(step_map[x].get("step_title")) != "根節點"
    ]
    if len(path_titles) > 1:
        st.caption("目前路徑：" + " → ".join(path_titles))

    control_col1, control_col2 = st.columns([1, 1])
    with control_col1:
        if st.button("🔄 重新開始", key=f"restart_{sop_id}"):
            st.session_state[state_key] = root_id
            st.session_state[path_key] = [root_id]
            st.rerun()
    with control_col2:
        if len(path_ids) > 1:
            if st.button("⬅️ 回上一步", key=f"back_{sop_id}"):
                new_path = path_ids[:-1]
                st.session_state[path_key] = new_path
                st.session_state[state_key] = new_path[-1] if new_path else root_id
                st.rerun()

    st.markdown("---")
    if children:
        st.markdown("#### 請選擇下一步分支")
        cols = st.columns(min(3, max(1, len(children))))
        for idx, child in enumerate(children):
            branch = normalize_branch_label(child.get("branch_label"), default="下一步")
            next_title = _clean_text(child.get("step_title"))
            target_id = child.get("_node_id")
            if bool(child.get("_virtual_terminal", False)):
                btn_label = f"{branch}"
            else:
                btn_label = f"{branch} → {next_title}"
            if cols[idx % len(cols)].button(btn_label, key=f"goto_{sop_id}_{target_id}"):
                st.session_state[state_key] = target_id
                st.session_state[path_key] = path_ids + [target_id]
                st.rerun()
    else:
        st.success("此流程已走到終點，以下顯示分析結果。")
        st.markdown("### 分析結果")

        st.markdown("#### 1. 終點判斷")
        if pd.notna(current_step.get("next_action_hint", None)) and _clean_text(current_step.get("next_action_hint")):
            st.write(current_step.get("next_action_hint"))
        else:
            st.write(_row_escalation(current_step))

        st.markdown("#### 2. 對應人工輔助建議")
        related_assist = find_related_human_assistance(current_step, overall_assist_df, state_name=state_name, max_items=4)
        render_human_assistance_cards(related_assist, section_title="Human Assistance Agent 建議")

        path_rows = [step_map[x] for x in path_ids if x in step_map and _clean_text(step_map[x].get("step_title")) != "根節點"]
        if path_rows:
            st.markdown("#### 3. 本次判斷路徑摘要")
            for idx, step in enumerate(path_rows, start=1):
                branch = normalize_branch_label(step.get("branch_label"), default="")
                label = f"步驟 {idx}"
                if branch:
                    label += f"｜分支：{branch}"
                st.write(f"**{label}** {step.get('step_title', '')}：{step.get('step_content', '')}")

        st.markdown("#### 4. 回填建議")
        st.write("請將本次確認結果、照片 / 截圖、處置方式、是否復機與升級通報結果回填到異常事件紀錄，作為後續相似案例推薦與 SOP 改善依據。")



# =========================================================
# 加強版：多模態、互動圖譜、主動預警工具函式
# =========================================================

def _safe_text(value, default="未提供"):
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    text = str(value).strip()
    if text.lower() in ["", "nan", "none", "nat"]:
        return default
    return text



def analyze_uploaded_photo_basic(uploaded_file):
    """
    展示版的照片基礎品質分析。
    這不是 Vision LLM，也不會判斷照片中的零件是什麼；
    但會依實際上傳圖片的尺寸、亮度、對比與清晰度產生不同提醒，
    避免「上傳任何照片結果都一樣」。
    """
    if uploaded_file is None:
        return {
            "has_photo": False,
            "filename": "未上傳照片",
            "width": None,
            "height": None,
            "brightness": None,
            "contrast": None,
            "sharpness": None,
            "quality_level": "未上傳",
            "quality_notes": [
                "尚未上傳照片，因此系統只能依設備、異常狀況、SOP 與歷史案例給出檢查方向。"
            ]
        }

    filename = getattr(uploaded_file, "name", "未命名照片")
    try:
        from PIL import Image
        import numpy as np

        uploaded_file.seek(0)
        img = Image.open(uploaded_file).convert("RGB")
        arr = np.asarray(img).astype("float32")
        gray = arr.mean(axis=2)

        width, height = img.size
        brightness = float(gray.mean())
        contrast = float(gray.std())
        # 不使用 OpenCV，改用 numpy gradient 做簡易清晰度估計。
        gy, gx = np.gradient(gray)
        sharpness = float((gx ** 2 + gy ** 2).mean())

        notes = []
        score = 0

        if width < 600 or height < 400:
            notes.append("照片解析度偏低，建議補拍近照或使用較高解析度照片。")
        else:
            score += 1
            notes.append("照片解析度足夠，可作為現場佐證。")

        if brightness < 55:
            notes.append("照片偏暗，可能看不清楚裂痕、磨耗或漏油位置。")
        elif brightness > 210:
            notes.append("照片偏亮，可能過曝而看不清楚表面細節。")
        else:
            score += 1
            notes.append("照片亮度大致正常。")

        if contrast < 22:
            notes.append("照片對比偏低，建議補拍有明確光線與角度的照片。")
        else:
            score += 1
            notes.append("照片對比尚可，有助於辨識邊界與異常區域。")

        if sharpness < 18:
            notes.append("照片可能模糊，建議補拍異常部位近照。")
        else:
            score += 1
            notes.append("照片清晰度尚可，可支援後續人工或 Vision 模型判讀。")

        if score >= 4:
            level = "高"
        elif score >= 2:
            level = "中"
        else:
            level = "低"

        uploaded_file.seek(0)
        return {
            "has_photo": True,
            "filename": filename,
            "width": width,
            "height": height,
            "brightness": brightness,
            "contrast": contrast,
            "sharpness": sharpness,
            "quality_level": level,
            "quality_notes": notes
        }
    except Exception as exc:
        try:
            uploaded_file.seek(0)
        except Exception:
            pass
        return {
            "has_photo": True,
            "filename": filename,
            "width": None,
            "height": None,
            "brightness": None,
            "contrast": None,
            "sharpness": None,
            "quality_level": "無法判讀",
            "quality_notes": [f"系統無法讀取此圖片的基礎資訊，請確認檔案格式是否正確。錯誤訊息：{exc}"]
        }


def infer_visual_focus_from_context(filename, equipment_code, state_name):
    """
    依檔名、設備與異常狀況推論「應該優先看照片哪裡」。
    這是展示版的情境推論，不是影像內容辨識。
    """
    filename_text = _safe_text(filename, "")
    equip_text = _safe_text(equipment_code, "未指定設備")
    state_text = _safe_text(state_name, "未指定異常")
    combined = f"{filename_text} {equip_text} {state_text}".lower()

    rules = [
        (
            ["斷", "裂", "break", "broken", "crack", "lever", "軸承破裂", "破裂"],
            "破損 / 斷裂類",
            [
                "優先看零件是否有裂痕、斷裂、變形、固定點鬆脫。",
                "補拍破損位置近照，並盡量拍到設備編號或相對位置。",
                "確認是否需要停機隔離，避免二次損壞或安全風險。"
            ]
        ),
        (
            ["磨", "耗", "wear", "knife", "切刀", "roll", "roller", "軋輥", "刮傷"],
            "磨耗 / 表面劣化類",
            [
                "優先看接觸面是否有磨耗、刮傷、缺角、表面剝落。",
                "建議補拍正常側與異常側對比照片，方便工程師判斷磨耗程度。",
                "同步確認更換週期、材料條件與近期是否重複發生。"
            ]
        ),
        (
            ["漏", "油", "water", "水", "leak", "液", "壓力", "流量"],
            "漏液 / 水油壓異常類",
            [
                "優先看現場是否有漏油、漏水、液體殘留或噴濺痕跡。",
                "補拍管線接頭、閥件、地面殘留與壓力表位置。",
                "同步檢查 sensor snapshot 中的水量、壓力、流量或溫度是否異常。"
            ]
        ),
        (
            ["偏", "卡", "cobble", "jam", "塞", "misalign", "打滑", "蛇行", "偏移"],
            "材料偏移 / 卡料 / 打滑類",
            [
                "優先看材料是否偏移、卡滯、打滑，或導引位置是否異常。",
                "補拍入口、出口、導板、夾輥與材料相對位置。",
                "同步確認加熱條件、速度設定與夾輥作動是否符合 SOP。"
            ]
        ),
        (
            ["異音", "vibration", "震", "振", "bearing", "軸承", "馬達"],
            "振動 / 異音 / 傳動類",
            [
                "照片本身不一定能判斷異音，建議搭配影片、聲音描述或振動數值。",
                "優先拍攝軸承座、馬達、聯軸器、固定螺絲與潤滑位置。",
                "同步檢查 sensor snapshot 中的振動、溫度與電流變化。"
            ]
        ),
    ]

    for keywords, label, hints in rules:
        if any(k in combined for k in keywords):
            return label, hints

    return "一般現場佐證類", [
        "請確認照片是否拍到異常部位、設備編號與周邊環境。",
        "建議至少補齊全景照、異常部位近照、正常/異常對比照。",
        "若照片無法直接說明原因，應以 SOP、sensor snapshot 與班長確認紀錄一起判斷。"
    ]


def generate_vision_agent_report(uploaded_file, equipment_code, state_name, related_cases=None, sop_df=None):
    """
    展示版 Vision Agent：
    不宣稱真正辨識照片內容；但會依「實際上傳照片品質」以及「檔名/設備/異常情境」產生不同檢查建議。
    未來可將 image_observation 區塊替換成 GPT-4o / Gemini Vision 的影像辨識結果。
    """
    photo_info = analyze_uploaded_photo_basic(uploaded_file)
    filename = photo_info["filename"]
    has_photo = photo_info["has_photo"]
    state_text = _safe_text(state_name, "未指定異常")
    equip_text = _safe_text(equipment_code, "未指定設備")
    focus_label, photo_clues = infer_visual_focus_from_context(filename, equipment_code, state_name)

    # SOP：優先抓需要照片/人工佐證的節點，避免文字爆量與重複。
    sop_hints = []
    if sop_df is not None and not sop_df.empty:
        temp = sop_df.copy()
        if "image_needed" in temp.columns:
            img_temp = temp[temp["image_needed"].astype(str).str.lower().isin(["y", "yes", "true", "1", "需要", "是"])]
            if not img_temp.empty:
                temp = img_temp

        for _, row in temp.head(3).iterrows():
            title = _safe_text(row.get("step_title"), "SOP 檢查節點")
            content = _safe_text(row.get("step_content"), "")
            evidence = _safe_text(row.get("evidence_required"), "照片、量測值或班長確認紀錄")
            owner = _safe_text(row.get("owner_role"), "現場人員")
            if content and content != "未提供":
                sop_hints.append(f"- {title}：{content}｜負責：{owner}｜建議佐證：{evidence}")
            else:
                sop_hints.append(f"- {title}｜負責：{owner}｜建議佐證：{evidence}")

    if not sop_hints:
        sop_hints = [
            "- 目前未找到對應 SOP 節點，建議先由班長確認異常部位並建立補充紀錄。",
            "- 後續可將本次照片、處置結果與實際原因回填，作為 SOP 改善資料。"
        ]

    # 歷史案例：只放最相近一筆，不重複展開過多案例。
    case_lines = []
    if related_cases is not None and not related_cases.empty:
        top_case = related_cases.iloc[0]
        case_lines.append(f"- 參考案例：{_safe_text(top_case.get('event_id'), '未知事件')}")
        case_lines.append(f"- 曾記錄原因：{_safe_text(top_case.get('cause_summary'), '未記錄')}")
        case_lines.append(f"- 曾採取處置：{_safe_text(top_case.get('action_summary'), '未記錄')}")
        if "downtime_min" in top_case.index:
            case_lines.append(f"- 該案例停機時間：{_safe_text(top_case.get('downtime_min'), '未記錄')} 分鐘")
    else:
        case_lines = ["- 目前沒有足夠相似案例可引用，建議以 SOP 與現場人工判斷為主。"]

    metric_lines = []
    if has_photo and photo_info.get("width"):
        metric_lines = [
            f"- 圖片尺寸：{photo_info['width']} × {photo_info['height']}",
            f"- 基礎品質等級：{photo_info['quality_level']}（依解析度、亮度、對比、清晰度估計）",
        ]
    else:
        metric_lines = [f"- 基礎品質等級：{photo_info['quality_level']}"]

    lines = [
        "【Vision Agent 綜合診斷摘要｜展示版】",
        "",
        "0. 目前定位",
        "- 這不是正式影像辨識模型，而是照片輔助診斷流程展示。",
        "- 本版會依實際照片品質、照片檔名、設備、異常狀況、SOP 與歷史案例產生不同建議。",
        "- 未來串接 GPT-4o / Gemini Vision 後，可把照片內容轉成結構化觀察，再交給 Agent 綜合判斷。",
        "",
        "1. 輸入資訊",
        f"- 照片狀態：{'已上傳' if has_photo else '未上傳'}",
        f"- 照片檔名：{filename}",
        f"- 設備：{equip_text}",
        f"- 異常狀況：{state_text}",
        f"- 情境推論類型：{focus_label}",
        *metric_lines,
        "",
        "2. 照片品質提醒",
        *[f"- {item}" for item in photo_info.get("quality_notes", [])],
        "",
        "3. 優先檢查方向",
        *[f"- {item}" for item in photo_clues],
        "",
        "4. 對應 SOP 確認項目",
        *sop_hints,
        "",
        "5. 歷史案例參考",
        *case_lines,
        "",
        "6. 建議處置順序",
        "1. 先判斷照片品質是否足夠；若偏暗、模糊或只拍局部，先補拍。",
        "2. 依情境推論類型確認重點部位，例如破損、磨耗、漏液、卡料、偏移或振動相關位置。",
        "3. 對照 SOP 節點，確認照片是否能支撐該步驟判斷。",
        "4. 同步保存 sensor snapshot、人工備註與處置結果，避免只靠照片下結論。",
        "5. 處理完成後回填實際原因，作為相似案例推薦與 SOP 改善資料。"
    ]
    return "\n".join(lines)


def _graph_columns(edges_df):
    src_col = "source" if "source" in edges_df.columns else edges_df.columns[0]
    tgt_col = "target" if "target" in edges_df.columns else edges_df.columns[1]
    rel_col = "relation" if "relation" in edges_df.columns else ("label" if "label" in edges_df.columns else None)
    return src_col, tgt_col, rel_col


def filter_graph_edges(edges_df, selected_node=None, relation_filter=None, max_edges=90):
    """把圖譜資料篩成適合展示的子圖，避免整張圖太大或空白。"""
    if edges_df is None or edges_df.empty:
        return pd.DataFrame()
    edf = edges_df.copy()
    src_col, tgt_col, rel_col = _graph_columns(edf)

    if selected_node and selected_node != "全部":
        selected = str(selected_node)
        mask = (edf[src_col].astype(str).str.contains(selected, case=False, na=False)) | (edf[tgt_col].astype(str).str.contains(selected, case=False, na=False))
        edf = edf[mask]

    if relation_filter and relation_filter != "全部" and rel_col:
        edf = edf[edf[rel_col].astype(str) == relation_filter]

    return edf.head(max_edges).copy()


def build_plotly_knowledge_graph(edges_df, nodes_df=None, selected_node=None, relation_filter=None, max_edges=90):
    """用 Plotly 畫穩定可顯示的知識圖譜；不用 pyvis，避免 Streamlit iframe 空白。"""
    if edges_df is None or edges_df.empty:
        return None, pd.DataFrame()

    edf = filter_graph_edges(edges_df, selected_node=selected_node, relation_filter=relation_filter, max_edges=max_edges)
    if edf.empty:
        return None, edf

    src_col, tgt_col, rel_col = _graph_columns(edf)
    node_list = sorted(set(edf[src_col].dropna().astype(str)) | set(edf[tgt_col].dropna().astype(str)))
    if not node_list:
        return None, edf

    # 先嘗試用 networkx spring layout；如果環境沒有 networkx，就用圓形配置。
    positions = {}
    try:
        import networkx as nx
        g = nx.Graph()
        for _, row in edf.iterrows():
            g.add_edge(str(row[src_col]), str(row[tgt_col]))
        positions = nx.spring_layout(g, seed=42, k=0.65, iterations=80)
    except Exception:
        n = len(node_list)
        for i, node in enumerate(node_list):
            angle = 2 * math.pi * i / max(n, 1)
            positions[node] = (math.cos(angle), math.sin(angle))

    label_lookup = {}
    type_lookup = {}
    if nodes_df is not None and not nodes_df.empty:
        id_col = "id" if "id" in nodes_df.columns else ("node_id" if "node_id" in nodes_df.columns else nodes_df.columns[0])
        label_col = "label" if "label" in nodes_df.columns else id_col
        type_col = "type" if "type" in nodes_df.columns else ("node_type" if "node_type" in nodes_df.columns else None)
        for _, row in nodes_df.iterrows():
            nid = str(row.get(id_col))
            label_lookup[nid] = _safe_text(row.get(label_col), nid)
            if type_col:
                type_lookup[nid] = _safe_text(row.get(type_col), "node")

    edge_x, edge_y, edge_hover = [], [], []
    mid_x, mid_y, mid_text = [], [], []
    for _, row in edf.iterrows():
        src, tgt = str(row[src_col]), str(row[tgt_col])
        if src not in positions or tgt not in positions:
            continue
        x0, y0 = positions[src]
        x1, y1 = positions[tgt]
        rel = _safe_text(row.get(rel_col), "關聯") if rel_col else "關聯"
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
        edge_hover.append(f"{src} → {tgt}: {rel}")
        mid_x.append((x0 + x1) / 2)
        mid_y.append((y0 + y1) / 2)
        mid_text.append(rel[:12])

    node_x, node_y, node_text, node_hover, node_size = [], [], [], [], []
    degree_count = {node: 0 for node in node_list}
    for _, row in edf.iterrows():
        degree_count[str(row[src_col])] = degree_count.get(str(row[src_col]), 0) + 1
        degree_count[str(row[tgt_col])] = degree_count.get(str(row[tgt_col]), 0) + 1

    for node in node_list:
        x, y = positions[node]
        nlabel = label_lookup.get(node, node)
        ntype = type_lookup.get(node, "node")
        node_x.append(x)
        node_y.append(y)
        node_text.append(nlabel[:24])
        node_hover.append(f"節點：{nlabel}<br>類型：{ntype}<br>關聯數：{degree_count.get(node, 0)}")
        node_size.append(16 + min(degree_count.get(node, 0), 12) * 2)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(width=1.4, color="rgba(90,90,90,0.45)"),
        hoverinfo="none", name="關係"
    ))
    fig.add_trace(go.Scatter(
        x=mid_x, y=mid_y, mode="text",
        text=mid_text, textfont=dict(size=10, color="rgba(70,70,70,0.75)"),
        hoverinfo="skip", showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode="markers+text",
        text=node_text, textposition="top center",
        hovertext=node_hover, hoverinfo="text",
        marker=dict(size=node_size, color="#4C78A8", line=dict(width=1.5, color="#FFFFFF")),
        name="節點"
    ))
    fig.update_layout(
        title="設備—異常—SOP—歷史事件關聯圖",
        height=650,
        margin=dict(l=10, r=10, t=45, b=10),
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
    )
    return fig, edf


def build_predictive_maintenance_alerts(sensor_df, event_df=None):
    """以現有 sensor snapshot 做簡化風險排序：超出規格、接近規格、異常 judgement、近期連續出現皆加分。"""
    if sensor_df is None or sensor_df.empty:
        return pd.DataFrame()

    df = sensor_df.copy()
    for col in ["actual_value", "spec_lower", "spec_upper"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "captured_at" in df.columns:
        df["captured_at"] = pd.to_datetime(df["captured_at"], errors="coerce")

    if event_df is not None and not event_df.empty and "event_id" in df.columns and "event_id" in event_df.columns:
        join_cols = [c for c in ["event_id", "equipment_code", "equipment_name", "state_name"] if c in event_df.columns]
        df = df.merge(event_df[join_cols].drop_duplicates("event_id"), on="event_id", how="left")

    group_cols = [c for c in ["equipment_code", "equipment_name", "monitor_name", "parameter_name", "unit"] if c in df.columns]
    if not group_cols:
        group_cols = [c for c in ["monitor_name", "parameter_name"] if c in df.columns]
    if not group_cols:
        return pd.DataFrame()

    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        meta = dict(zip(group_cols, keys))
        gg = g.copy()
        if "captured_at" in gg.columns:
            gg = gg.sort_values("captured_at")
        values = gg["actual_value"] if "actual_value" in gg.columns else pd.Series(dtype="float")
        lower = pd.to_numeric(gg["spec_lower"], errors="coerce") if "spec_lower" in gg.columns else pd.Series(dtype="float")
        upper = pd.to_numeric(gg["spec_upper"], errors="coerce") if "spec_upper" in gg.columns else pd.Series(dtype="float")
        judgement_text = gg["judgement"].fillna("").astype(str) if "judgement" in gg.columns else pd.Series([""] * len(gg))

        abnormal_count = judgement_text.str.contains("異常|NG|超|HIGH|LOW|ALARM|警", case=False, regex=True, na=False).sum()
        risk_score = abnormal_count * 3
        near_count = 0
        out_count = 0
        latest_value = None
        latest_upper = None
        latest_lower = None
        trend = "資料不足"

        if values.notna().any():
            latest_value = values.dropna().iloc[-1]
            if upper.notna().any():
                latest_upper = upper.dropna().iloc[-1]
            if lower.notna().any():
                latest_lower = lower.dropna().iloc[-1]

            valid = pd.DataFrame({"value": values, "lower": lower, "upper": upper}).dropna(subset=["value"])
            if not valid.empty:
                if "upper" in valid and valid["upper"].notna().any():
                    out_count += (valid["value"] > valid["upper"]).sum()
                    denom = (valid["upper"] - valid["lower"]).replace(0, pd.NA) if valid["lower"].notna().any() else valid["upper"].abs().replace(0, pd.NA)
                    ratio_to_upper = (valid["upper"] - valid["value"]) / denom
                    near_count += ((ratio_to_upper >= 0) & (ratio_to_upper <= 0.15)).sum()
                if "lower" in valid and valid["lower"].notna().any():
                    out_count += (valid["value"] < valid["lower"]).sum()
                risk_score += int(out_count) * 5 + int(near_count) * 2

            tail = values.dropna().tail(5)
            if len(tail) >= 3:
                delta = tail.iloc[-1] - tail.iloc[0]
                if abs(delta) < 1e-9:
                    trend = "持平"
                elif delta > 0:
                    trend = "上升"
                    risk_score += 1
                else:
                    trend = "下降"

        sample_count = len(gg)
        if sample_count >= 5:
            risk_score += 1

        if risk_score >= 8:
            level = "高"
        elif risk_score >= 4:
            level = "中"
        else:
            level = "低"

        reason_parts = []
        if abnormal_count:
            reason_parts.append(f"異常判定 {abnormal_count} 次")
        if out_count:
            reason_parts.append(f"超出規格 {int(out_count)} 次")
        if near_count:
            reason_parts.append(f"接近上下限 {int(near_count)} 次")
        if trend in ["上升", "下降"]:
            reason_parts.append(f"近期趨勢{trend}")
        if not reason_parts:
            reason_parts.append("目前未看到明顯超規或異常字樣")

        rows.append({
            **meta,
            "sample_count": sample_count,
            "latest_value": latest_value,
            "spec_lower": latest_lower,
            "spec_upper": latest_upper,
            "trend": trend,
            "abnormal_count": int(abnormal_count),
            "near_limit_count": int(near_count),
            "risk_score": round(float(risk_score), 2),
            "risk_level": level,
            "risk_reason": "、".join(reason_parts),
            "suggestion": "提前安排巡檢，確認軸承、潤滑、固定件、噴嘴或感測器狀態；若再次升高，建議停機檢查。" if level in ["中", "高"] else "維持觀察，保留趨勢紀錄並確認資料是否完整。"
        })

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    risk_order = {"高": 3, "中": 2, "低": 1}
    result["_risk_order"] = result["risk_level"].map(risk_order).fillna(0)
    return result.sort_values(["_risk_order", "risk_score", "sample_count"], ascending=[False, False, False]).drop(columns=["_risk_order"])

# =========================================================
# Sidebar
# =========================================================

st.sidebar.title("🛠️ 製造異常診斷系統")
st.sidebar.caption("生成式 AI × GraphRAG × Multi-Agent")
st.sidebar.markdown("---")

st.sidebar.markdown("### 建議展示順序")
st.sidebar.caption("先看系統與資料概況，再進入異常處理流程，最後查看預警、改善與設備原因關聯。")

MENU_GROUPS = {
    "系統與資料概況": [
        "系統總覽",
        "異常分析看板",
    ],
    "異常處理流程": [
        "異常事件查詢",
        "SOP 處理流程查詢",
        "AI 異常診斷建議",
        "現場照片佐證紀錄",
    ],
    "預警與改善": [
        "主動預警 Agent",
        "SOP 改善建議",
    ],
    "知識圖譜展示": [
        "設備原因查詢",
    ],
}

if "selected_page" not in st.session_state:
    st.session_state["selected_page"] = "系統總覽"

st.sidebar.markdown("#### 請選擇功能頁面")

for group_title, group_pages in MENU_GROUPS.items():
    st.sidebar.markdown(f"**{group_title}**")
    for menu_item in group_pages:
        is_selected = st.session_state["selected_page"] == menu_item
        label = f"• {menu_item}"
        if st.sidebar.button(
            label,
            key=f"nav_{menu_item}",
            use_container_width=True,
            type="primary" if is_selected else "secondary",
        ):
            st.session_state["selected_page"] = menu_item
    st.sidebar.markdown("")

page = st.session_state["selected_page"]

st.sidebar.markdown("---")
st.sidebar.subheader("資料概況")

st.sidebar.metric("異常事件數", f"{len(clean_event_view):,}")

if "equipment_code" in clean_event_view.columns:
    st.sidebar.metric("涉及設備數", f"{clean_event_view['equipment_code'].nunique():,}")

st.sidebar.metric("SOP 步驟數", f"{len(clean_sop_view):,}")
st.sidebar.metric("感測器快照數", f"{len(clean_sensor_view):,}")

if graph_nodes is not None:
    st.sidebar.metric("知識節點數", f"{len(graph_nodes):,}")

if graph_edges is not None:
    st.sidebar.metric("知識關係數", f"{len(graph_edges):,}")

if "occurred_at" in clean_event_view.columns:
    valid_dates = clean_event_view["occurred_at"].dropna()
    if len(valid_dates) > 0:
        st.sidebar.caption(
            f"資料期間：{valid_dates.min().date()} ～ {valid_dates.max().date()}"
        )


# =========================================================
# Page 1：系統總覽
# =========================================================

if page == "系統總覽":
    st.title("🛠️ 生成式 AI 製造異常診斷與 SOP 查詢輔助系統")
    st.caption("把製造異常資料、SOP、感測器快照與歷史案例串成知識圖譜，讓 AI 協助現場診斷、查 SOP、找相似案例與提出改善建議。")

    k1, k2, k3, k4 = st.columns(4)
    k1.metric("異常事件數", f"{len(clean_event_view):,}")

    if "equipment_code" in clean_event_view.columns:
        k2.metric("涉及設備數", f"{clean_event_view['equipment_code'].nunique():,}")
    else:
        k2.metric("涉及設備數", "無資料")

    k3.metric("SOP 步驟數", f"{len(clean_sop_view):,}")
    k4.metric("感測器快照數", f"{len(clean_sensor_view):,}")

    st.markdown("---")

    st.success("""
    **本專題的核心價值：** 不只是做異常 Dashboard，而是讓系統在異常發生時，
    依照「設備 → 異常狀況 → SOP → 感測器 → 歷史案例 → 人工確認」的順序整理線索，
    協助現場人員更快知道要查什麼、先處理什麼、哪些地方不能只靠 AI 判斷。
    """)

    tab_intro, tab_ai, tab_flow, tab_limit = st.tabs([
        "📌 專題背景",
        "🤖 生成式 AI 重點",
        "🧭 系統流程",
        "⚠️ 限制與未來改善"
    ])

    with tab_intro:
        st.markdown("""
        ### 專題背景與系統目的

        在傳統製造現場中，異常處理資訊常分散在 Excel、SOP 文件、歷史報告與工程師經驗中。
        當設備發生異常時，現場人員往往需要花時間查找過去案例、確認 SOP 流程、比對感測資料，
        並整理通報內容，整體處理流程容易受到經驗差異與資料分散影響。

        因此，本專題將異常事件、設備、異常狀況、SOP、感測器資料與歷史案例整理為結構化知識庫，
        並結合 GraphRAG 與 Multi-Agent 架構，讓系統能根據使用者選擇的設備與異常狀況，
        自動查詢相似事件、整理 SOP 步驟、推估可能原因、提醒人工確認項目，
        並產生初步診斷摘要與通報內容。
        """)

        st.markdown("### 展示時可以主打的 4 個亮點")
        st.markdown("""
        1. **資料可讀化**：把原本代碼型資料轉成設備、異常、SOP、感測器與歷史事件的可讀資料表。
        2. **GraphRAG 查詢**：不是讓 AI 直接亂猜，而是先取回與設備、異常、SOP、歷史案例相關的資料。
        3. **Multi-Agent 分工**：Diagnosis、SOP、Human Assistance、Parts、Notification 各自負責不同任務。
        4. **SOP 持續改善**：利用歷史事件與人工確認紀錄，找出最常出錯或最需要補資料的 SOP 節點。
        """)

    with tab_ai:
        st.markdown("""
        ### 生成式 AI 應用重點

        本系統的重點不只是資料查詢，而是將查詢到的歷史案例、SOP、感測器與原因分類資料，
        轉換成現場人員可閱讀的自然語言診斷摘要。
        """)

        agent_table = pd.DataFrame([
            {"Agent": "Diagnosis Agent", "負責內容": "根據歷史事件、停機時間、原因分類與感測器快照推估可能原因"},
            {"Agent": "SOP Agent", "負責內容": "查詢設備與異常狀況對應 SOP，整理為現場可理解的處理步驟"},
            {"Agent": "Human Assistance Agent", "負責內容": "找出需要現場目視、拍照、量測或人工補充的確認項目"},
            {"Agent": "Parts Agent", "負責內容": "根據 SOP 文字與歷史處置紀錄推估可能需要確認的備件或物料"},
            {"Agent": "Notification Agent", "負責內容": "依不同閱讀對象產生現場人員版、主管版與維修單位版通報"}
        ])
        st.dataframe(agent_table, use_container_width=True, hide_index=True)

        st.info("""
        **Human Assistance Agent 的意義：** 它不是表示 AI 失敗，而是表示製造現場有些判斷本來就必須由人確認，
        例如異音、現場照片、設備外觀、鋼材狀態、安全風險與復機條件。
        """)

    with tab_flow:
        st.markdown("### 系統展示流程")

        st.code("""
異常事件發生
↓
選擇設備與異常狀況
↓
GraphRAG 取回相關資料
↓
查到 SOP / 感測器 / 歷史案例
↓
Multi-Agent 分工診斷
↓
輸出處置建議、人工確認項目、備件提醒與通報內容
↓
回填處置結果，作為相似案例推薦與 SOP 改善依據
        """)

        st.markdown("### 系統架構")

        st.code("""
原始異常資料與 SOP 文件
↓
資料清理與欄位可讀化
↓
建立設備、異常狀況、SOP、感測器與歷史事件知識庫
↓
建立知識圖譜節點與關係
↓
依使用者輸入進行 GraphRAG 關聯查詢
↓
由多個 Agent 分工完成診斷、SOP、人工確認、備件與通報任務
↓
整合產生生成式 AI 診斷摘要與角色化通報內容
↓
透過 Streamlit UI 提供互動式查詢與展示
        """)

    with tab_limit:
        st.markdown("""
        ### 系統限制

        1. 部分資料屬於模擬或整理後資料，仍需更多真實事件驗證。
        2. AI 診斷目前是輔助整理線索，不能取代工程師與現場主管的判斷。
        3. 感測器判斷依賴資料完整性，若缺少即時資料仍需人工補充。
        4. 相似案例推薦目前以設備、異常狀況、原因與處置文字為主，未來可加入向量檢索與權重排序。
        5. 未來可把 SOP 改善建議回寫資料庫，形成持續學習機制。
        """)



# =========================================================
# Page 2：設備原因查詢
# =========================================================

elif page == "設備原因查詢":
    st.title("🔎 設備原因查詢")
    st.caption("用下拉選單直接查看某設備常見異常、可能原因與歷史處置，不需要使用者自己從表格中找線索。")

    st.info("""
    這頁保留知識圖譜的概念，但改成更實用的查詢方式：
    使用者先選設備，再看該設備過去常見的異常狀況、原因分類、停機時間與處置摘要。
    這比直接顯示一大張空白或難讀的圖更適合期末展示，也比較接近現場人員真正會使用的方式。
    """)

    if clean_event_view.empty or "equipment_code" not in clean_event_view.columns:
        st.warning("目前沒有異常事件資料，無法建立設備原因查詢。")
    else:
        eq_options = sorted(clean_event_view["equipment_code"].dropna().astype(str).unique().tolist())
        selected_eq = st.selectbox("選擇設備", eq_options, key="cause_lookup_equipment")

        eq_events = clean_event_view[clean_event_view["equipment_code"].astype(str) == selected_eq].copy()

        state_options = ["全部"]
        if "state_name" in eq_events.columns:
            state_options += sorted(eq_events["state_name"].dropna().astype(str).unique().tolist())
        selected_state_for_cause = st.selectbox("異常狀況", state_options, key="cause_lookup_state")

        if selected_state_for_cause != "全部" and "state_name" in eq_events.columns:
            eq_events = eq_events[eq_events["state_name"].astype(str) == selected_state_for_cause]

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("歷史事件數", f"{len(eq_events):,}")
        if "downtime_min" in eq_events.columns and not eq_events["downtime_min"].dropna().empty:
            c2.metric("平均停機分鐘", f"{eq_events['downtime_min'].mean():.1f}")
            c3.metric("最高停機分鐘", f"{eq_events['downtime_min'].max():.0f}")
        else:
            c2.metric("平均停機分鐘", "無資料")
            c3.metric("最高停機分鐘", "無資料")
        if "state_name" in eq_events.columns and not eq_events.empty:
            most_state = eq_events["state_name"].value_counts().index[0]
            c4.metric("最常見異常", most_state)
        else:
            c4.metric("最常見異常", "無資料")

        if eq_events.empty:
            st.warning("目前這個篩選條件下沒有歷史事件。")
        else:
            tab_state, tab_cause, tab_action, tab_events = st.tabs([
                "常見異常",
                "原因整理",
                "處置整理",
                "歷史事件明細",
            ])

            with tab_state:
                if "state_name" in eq_events.columns:
                    state_count = eq_events["state_name"].fillna("未標記").value_counts().reset_index()
                    state_count.columns = ["異常狀況", "事件數"]
                    col_l, col_r = st.columns([1, 1.3])
                    with col_l:
                        st.dataframe(state_count, use_container_width=True, hide_index=True)
                    with col_r:
                        fig = px.bar(state_count.head(10), x="事件數", y="異常狀況", orientation="h", title=f"{selected_eq} 常見異常 Top 10")
                        fig.update_layout(yaxis={"categoryorder": "total ascending"})
                        st.plotly_chart(fig, use_container_width=True)
                else:
                    st.info("事件資料中沒有 state_name 欄位。")

            with tab_cause:
                if "root_cause_category" in eq_events.columns:
                    cause_cat = eq_events["root_cause_category"].fillna("未分類").astype(str).value_counts().reset_index()
                    cause_cat.columns = ["原因分類", "事件數"]
                    st.markdown("#### 原因分類統計")
                    st.dataframe(cause_cat, use_container_width=True, hide_index=True)

                if "cause_summary" in eq_events.columns:
                    st.markdown("#### 歷史原因摘要")
                    cause_df = eq_events[[c for c in ["event_id", "occurred_at", "state_name", "downtime_min", "root_cause_category", "cause_summary"] if c in eq_events.columns]].copy()
                    st.dataframe(cause_df.head(20), use_container_width=True, hide_index=True)

                    examples = cause_df["cause_summary"].dropna().astype(str).head(5).tolist()
                    if examples:
                        st.markdown("#### 可直接說明的重點")
                        for item in examples:
                            st.write(f"- {item}")
                else:
                    st.info("事件資料中沒有 cause_summary 欄位。")

            with tab_action:
                if "action_summary" in eq_events.columns:
                    action_df = eq_events[[c for c in ["event_id", "occurred_at", "state_name", "downtime_min", "action_summary"] if c in eq_events.columns]].copy()
                    st.dataframe(action_df.head(20), use_container_width=True, hide_index=True)
                    st.markdown("#### 使用方式")
                    st.write("這裡可以快速看過去同設備的處置方式，作為 SOP 查詢與 AI 診斷建議的參考；但仍需由現場人員確認本次狀況是否相同。")
                else:
                    st.info("事件資料中沒有 action_summary 欄位。")

            with tab_events:
                show_cols = [c for c in ["event_id", "occurred_at", "equipment_code", "state_name", "severity", "downtime_min", "root_cause_category", "cause_summary", "action_summary"] if c in eq_events.columns]
                st.dataframe(eq_events[show_cols].sort_values("occurred_at", ascending=False).head(100) if "occurred_at" in eq_events.columns else eq_events[show_cols].head(100), use_container_width=True, hide_index=True)

# =========================================================
# Page 3：異常分析看板
# =========================================================

elif page == "異常分析看板":
    st.title("📊 異常事件分析看板")
    st.caption("完整呈現異常分布、設備問題頻率、嚴重度、原因分類與時間趨勢。")

    c1, c2, c3, c4 = st.columns(4)

    c1.metric("總異常事件數", f"{dashboard_summary['total_events']:,}")

    if dashboard_summary["avg_downtime"] is not None:
        c2.metric("平均停機時間", f"{dashboard_summary['avg_downtime']:.1f} 分鐘")
        c3.metric("最高停機時間", f"{dashboard_summary['max_downtime']:.0f} 分鐘")
    else:
        c2.metric("平均停機時間", "無資料")
        c3.metric("最高停機時間", "無資料")

    c4.metric("涉及設備數", f"{dashboard_summary['equipment_count']:,}")

    st.markdown("---")

    st.markdown("### 異常分布統計")

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Top 10 異常設備")

        fig1 = px.bar(
            top_equipment_df,
            x="equipment_code",
            y="count",
            title="Top 10 異常設備",
            color="count",
            color_continuous_scale="Blues",
            labels={
                "equipment_code": "設備代碼",
                "count": "異常次數"
            }
        )

        fig1.update_layout(coloraxis_showscale=False)
        st.plotly_chart(fig1, use_container_width=True)

        st.dataframe(
            top_equipment_df.rename(
                columns={
                    "equipment_code": "設備代碼",
                    "count": "異常次數"
                }
            ),
            use_container_width=True
        )

    with col2:
        st.subheader("Top 10 異常狀況")

        fig2 = px.bar(
            top_state_df,
            x="count",
            y="state_name",
            orientation="h",
            title="Top 10 異常狀況",
            color="count",
            color_continuous_scale="Viridis",
            labels={
                "state_name": "異常狀況",
                "count": "異常次數"
            }
        )

        fig2.update_layout(
            coloraxis_showscale=False,
            yaxis_title=""
        )

        st.plotly_chart(fig2, use_container_width=True)

        st.dataframe(
            top_state_df.rename(
                columns={
                    "state_name": "異常狀況",
                    "count": "異常次數"
                }
            ),
            use_container_width=True
        )

    st.markdown("---")

    st.markdown("### 嚴重度與原因分類")

    col3, col4 = st.columns(2)

    with col3:
        st.subheader("嚴重度分布")

        fig3 = px.pie(
            severity_df,
            names="severity",
            values="count",
            title="嚴重度分布",
            hole=0.4
        )

        fig3.update_traces(textinfo="percent+label")
        st.plotly_chart(fig3, use_container_width=True)

        st.dataframe(
            severity_df.rename(
                columns={
                    "severity": "嚴重度",
                    "count": "事件數"
                }
            ),
            use_container_width=True
        )

    with col4:
        st.subheader("原因分類分布")

        fig4 = px.pie(
            cause_df,
            names="root_cause_category",
            values="count",
            title="原因分類分布",
            hole=0.4
        )

        fig4.update_traces(textinfo="percent+label")
        st.plotly_chart(fig4, use_container_width=True)

        st.dataframe(
            cause_df.rename(
                columns={
                    "root_cause_category": "原因分類",
                    "count": "事件數"
                }
            ),
            use_container_width=True
        )

    st.markdown("---")

    st.markdown("### 每月異常事件趨勢")
    st.caption("此圖可用來觀察不同月份的異常事件數量變化。")

    if len(monthly_count_df) > 0:
        fig_trend = px.area(
            monthly_count_df,
            x="month",
            y="count",
            title="每月異常事件趨勢",
            color_discrete_sequence=["#667eea"],
            labels={
                "month": "月份",
                "count": "異常事件數"
            }
        )

        fig_trend.update_xaxes(tickangle=45)
        st.plotly_chart(fig_trend, use_container_width=True)

        st.dataframe(
            monthly_count_df.rename(
                columns={
                    "month": "月份",
                    "count": "異常事件數"
                }
            ),
            use_container_width=True
        )
    else:
        st.warning("目前沒有可用的發生時間資料，因此無法產生時間趨勢圖。")


# =========================================================
# Page 4：異常事件查詢
# =========================================================

elif page == "異常事件查詢":
    st.title("🔍 歷史異常事件查詢")
    st.caption("依設備、異常狀況與嚴重度篩選歷史異常事件。")

    col1, col2, col3 = st.columns(3)

    equipment_options = ["全部"] + sorted(
        clean_event_view["equipment_code"]
        .dropna()
        .unique()
        .tolist()
    )

    selected_equipment = col1.selectbox("設備", equipment_options)

    if selected_equipment == "全部":
        state_source_df = clean_event_view.copy()
    else:
        state_source_df = clean_event_view[
            clean_event_view["equipment_code"] == selected_equipment
        ].copy()

    state_options = ["全部"] + sorted(
        state_source_df["state_name"]
        .dropna()
        .unique()
        .tolist()
    )

    selected_state = col2.selectbox("異常狀況", state_options)

    severity_source_df = state_source_df.copy()

    if selected_state != "全部":
        severity_source_df = severity_source_df[
            severity_source_df["state_name"] == selected_state
        ].copy()

    severity_options = ["全部"] + sorted(
        severity_source_df["severity"]
        .dropna()
        .unique()
        .tolist()
    )

    selected_severity = col3.selectbox("嚴重度", severity_options)

    filtered = severity_source_df.copy()

    if selected_severity != "全部":
        filtered = filtered[
            filtered["severity"] == selected_severity
        ].copy()

    st.markdown("---")

    c1, c2, c3 = st.columns(3)
    c1.metric("符合條件事件數", f"{len(filtered):,}")

    if "downtime_min" in filtered.columns and len(filtered) > 0:
        c2.metric("平均停機時間", f"{filtered['downtime_min'].mean():.1f} 分鐘")
        c3.metric("最高停機時間", f"{filtered['downtime_min'].max():.0f} 分鐘")
    else:
        c2.metric("平均停機時間", "無資料")
        c3.metric("最高停機時間", "無資料")

    st.markdown("### 歷史異常事件列表")

    display_cols = [
        "event_id",
        "occurred_at",
        "equipment_code",
        "equipment_name",
        "state_name",
        "severity",
        "downtime_min",
        "root_cause_category",
        "action_summary"
    ]

    existing_cols = [c for c in display_cols if c in filtered.columns]

    st.dataframe(
        filtered[existing_cols],
        use_container_width=True
    )

    with st.expander("查看完整欄位資料"):
        st.dataframe(filtered, use_container_width=True)


# =========================================================
# Page 5：SOP 處理流程查詢
# =========================================================


elif page == "SOP 處理流程查詢":
    st.title("📋 SOP 處理流程查詢｜逐步互動導覽")
    st.caption("依設備與異常狀況查詢對應 SOP，每個步驟以一頁一頁的方式呈現；走到無下一步時才顯示分析結果。")

    col1, col2 = st.columns(2)

    equipment_options = sorted(
        clean_event_view["equipment_code"]
        .dropna()
        .unique()
        .tolist()
    )

    selected_equipment = col1.selectbox("選擇設備", equipment_options)

    related_states = (
        clean_event_view[
            clean_event_view["equipment_code"] == selected_equipment
        ]["state_name"]
        .dropna()
        .unique()
        .tolist()
    )

    related_states = sorted(related_states)

    if len(related_states) == 0:
        st.warning("此設備目前沒有對應的異常狀況資料。")
        st.stop()

    selected_state = col2.selectbox("選擇異常狀況", related_states)

    st.markdown("---")

    sop_df = get_sop_by_equipment_state(selected_equipment, selected_state)

    if sop_df.empty:
        st.warning("目前找不到對應 SOP。")
    else:
        st.success(f"找到 {sop_df['sop_id'].nunique()} 份 SOP")

        for sop_id, group in sop_df.groupby("sop_id"):
            first = group.iloc[0]

            with st.expander(f"{sop_id}｜{first.get('sop_name', '')}", expanded=True):
                c1, c2 = st.columns(2)
                c1.info(f"**SOP 說明**：{first.get('sop_desc', '')}")
                c2.info(f"**負責角色**：{first.get('owner_role', '')}")

                tab_v2, tab_table = st.tabs([
                    "逐步互動導覽",
                    "原始表格"
                ])

                with tab_v2:
                    render_version_two(group, selected_equipment, selected_state, sop_id)

                with tab_table:
                    display_cols = [
                        "sort_order",
                        "branch_label",
                        "sop_stage",
                        "step_title",
                        "step_content",
                        "check_method_label",
                        "monitor_name",
                        "standard_text",
                        "evidence_required",
                        "safety_note"
                    ]
                    existing_cols = [c for c in display_cols if c in group.columns]
                    st.dataframe(group[existing_cols], use_container_width=True)


# =========================================================
# Page 6：AI 異常診斷建議
# =========================================================

elif page == "AI 異常診斷建議":
    st.title("🤖 AI 異常診斷建議")
    st.caption("整合歷史案例、SOP、感測資料與通報規則，生成可供現場工程師參考的診斷摘要。")

    st.warning("""
    使用提醒：本頁面的 AI 診斷只負責整理歷史線索、SOP 與通報內容，不能取代現場判斷。
    涉及安全、設備實況、感測器是否可信、復機條件與最終結案，都必須由現場人員或工程師人工確認。
    """)

    col1, col2 = st.columns(2)

    equipment_options = sorted(
        clean_event_view["equipment_code"]
        .dropna()
        .unique()
        .tolist()
    )

    selected_equipment = col1.selectbox(
        "選擇設備",
        equipment_options,
        key="diag_equipment"
    )

    related_states = (
        clean_event_view[
            clean_event_view["equipment_code"] == selected_equipment
        ]["state_name"]
        .dropna()
        .unique()
        .tolist()
    )

    related_states = sorted(related_states)

    if len(related_states) == 0:
        st.warning("此設備目前沒有對應的異常狀況資料。")
        st.stop()

    selected_state = col2.selectbox(
        "選擇異常狀況",
        related_states,
        key="diag_state"
    )

    st.markdown("---")

    if st.button("產生 AI 診斷建議", type="primary"):
        with st.spinner("系統正在整合歷史事件、SOP、感測資料與 Agent 結果..."):
            summary_text, result = final_demo_generative_summary(
                selected_equipment,
                selected_state
            )

        notification = result["notification"]
        human = result["human"]
        role_versions = notification.get("role_versions", {})

        formal_report = build_formal_ai_report(
            selected_equipment,
            selected_state,
            result
        )

        st.markdown("### 1. 正式 AI 異常診斷報告")
        st.info(formal_report)

        st.markdown("### 2. AI 生成式分析摘要")
        st.caption("這一段保留較口語化的整合摘要，方便放入報告或簡報說明。")
        st.info(summary_text)

        st.markdown("### 3. 需人工確認提醒")
        st.caption("這裡不是要再顯示一次 Human Assistance Agent，而是提醒使用此頁面的人：哪些地方不能只靠 AI 判斷。")

        manual_items = []
        if human.get("assist_df") is not None and not human["assist_df"].empty:
            for _, row in human["assist_df"].head(5).iterrows():
                check_item = row.get("check_item", "依現場狀況確認")
                evidence = row.get("evidence_required", "人工確認紀錄")
                escalation = row.get("escalation_rule", "若處置後仍未改善，需升級通報")
                manual_items.append({
                    "check_item": check_item,
                    "evidence": evidence,
                    "escalation": escalation
                })

        if not manual_items:
            manual_items = [
                {
                    "check_item": "確認現場異常是否仍持續發生，或是否已恢復正常。",
                    "evidence": "現場觀察紀錄、照片或班長確認備註。",
                    "escalation": "若異常持續或涉及安全風險，立即通知班長與工程師。"
                },
                {
                    "check_item": "確認感測器資料是否可信，例如是否有訊號 loss、資料缺漏或異常跳動。",
                    "evidence": "感測器截圖、PLC / SCADA 畫面或人工量測值。",
                    "escalation": "若資料不可信，不應直接採用 AI 判斷，需人工複核。"
                },
                {
                    "check_item": "確認目前 SOP 是否符合現場實際作業方式。",
                    "evidence": "SOP 執行紀錄、現場處置備註。",
                    "escalation": "若 SOP 與現場不符，需回饋製程或設備工程師修訂。"
                }
            ]

        for idx, item in enumerate(manual_items, start=1):
            with st.container(border=True):
                st.markdown(f"#### 人工確認 {idx}")
                st.write(f"**需要人工確認：** {item['check_item']}")
                st.write(f"**建議留下佐證：** {item['evidence']}")
                st.write(f"**升級通報條件：** {item['escalation']}")

        st.markdown("---")

        st.markdown("### 4. 多 Agent 分析細節")
        st.caption("此區把各 Agent 的輸出集中在同一個地方；角色化通報歸在 Notification Agent 底下，避免與前面的診斷摘要重複。")

        tab1, tab2, tab3, tab4 = st.tabs([
            "Diagnosis Agent",
            "SOP Agent",
            "Parts Agent",
            "Notification Agent"
        ])

        with tab1:
            st.markdown("#### Diagnosis Agent｜可能原因判斷")
            st.caption("負責根據設備、異常狀況、歷史案例與感測資料，整理可能原因與風險說明。")
            st.markdown(result["diagnosis"]["text"].replace("\n", "  \n"))

        with tab2:
            st.markdown("#### SOP Agent｜標準流程對應")
            st.caption("負責找出可能對應的 SOP，並整理現場處理時應優先確認的步驟。")
            st.markdown(result["sop"]["text"].replace("\n", "  \n"))

        with tab3:
            st.markdown("#### Parts Agent｜備件與維修提醒")
            st.caption("負責提醒可能牽涉的備件、耗材或維修資源，供維修單位提前準備。")
            st.markdown(result["parts"]["text"].replace("\n", "  \n"))

        with tab4:
            st.markdown("#### Notification Agent｜角色化通報內容")
            st.caption("同一筆異常資料，系統會依不同閱讀對象生成不同版本的通報內容。這一塊原本獨立顯示，現在歸在 Notification Agent 底下，頁面邏輯會更清楚。")

            for role_name, role_desc in [
                ("現場人員版", "立即處理提醒，重點是先做什麼、如何確保安全與留下紀錄。"),
                ("主管通報版", "管理摘要，重點是嚴重度、影響範圍、目前進度與是否需要決策。"),
                ("維修單位版", "維修準備，重點是可能原因、檢修方向、備件與現場佐證。"),
            ]:
                with st.container(border=True):
                    st.markdown(f"##### {role_name}")
                    st.caption(role_desc)
                    st.markdown(role_versions.get(role_name, f"目前沒有{role_name}通報內容。").replace("\n", "  \n"))

        st.markdown("---")

        st.markdown("### 5. 相似歷史事件推薦")
        st.caption("系統優先推薦同設備、同異常、處置紀錄較完整，且停機時間較短的歷史案例。")

        matched_events = get_recommended_cases(selected_equipment, selected_state, top_n=8)

        display_cols = [
            "event_id",
            "occurred_at",
            "equipment_code",
            "state_name",
            "severity",
            "downtime_min",
            "root_cause_category",
            "cause_summary",
            "action_summary",
            "sop_name",
            "_recommend_score"
        ]

        existing_cols = [c for c in display_cols if c in matched_events.columns]

        st.dataframe(
            matched_events[existing_cols],
            use_container_width=True
        )

        manual_text_lines = []
        manual_text_lines.append("【需人工確認提醒】\n")
        for idx, item in enumerate(manual_items, start=1):
            manual_text_lines.append(f"{idx}. 需要人工確認：{item['check_item']}\n")
            manual_text_lines.append(f"   建議留下佐證：{item['evidence']}\n")
            manual_text_lines.append(f"   升級通報條件：{item['escalation']}\n")

        download_text = []
        download_text.append(formal_report)
        download_text.append("\n\n==============================\n")
        download_text.append(summary_text)
        download_text.append("\n\n==============================\n")
        download_text.append("".join(manual_text_lines))
        download_text.append("\n\n==============================\n")
        download_text.append("【角色化通報內容】")
        download_text.append("\n\n--- 現場人員版 ---\n")
        download_text.append(role_versions.get("現場人員版", ""))
        download_text.append("\n\n--- 主管通報版 ---\n")
        download_text.append(role_versions.get("主管通報版", ""))
        download_text.append("\n\n--- 維修單位版 ---\n")
        download_text.append(role_versions.get("維修單位版", ""))

        st.download_button(
            label="下載 AI 診斷與通報摘要 TXT",
            data="".join(download_text),
            file_name=f"{selected_equipment}_{selected_state}_AI_diagnosis.txt",
            mime="text/plain"
        )



# =========================================================
# Page 7：現場照片佐證紀錄
# =========================================================

elif page == "現場照片佐證紀錄":
    st.title("📷 現場照片佐證紀錄")
    st.caption("這頁只負責保存照片佐證與現場補充說明，不再產生重複或不可靠的診斷內容。")

    st.info("""
    **本頁定位：留下證據，不做照片判讀。**

    目前版本不判斷照片中的零件是否真的破裂、磨耗或漏油；照片只作為 SOP 人工確認、異常追蹤與事後改善的佐證。
    真正的原因判斷請回到「AI 異常診斷建議」或「設備原因查詢」頁面。
    """)

    st.markdown("### 建立照片佐證紀錄")
    col1, col2 = st.columns([1, 1])
    with col1:
        equipment_options = ["未指定"]
        if "equipment_code" in clean_event_view.columns:
            equipment_options += sorted(clean_event_view["equipment_code"].dropna().astype(str).unique().tolist())
        selected_equipment = st.selectbox("設備", equipment_options, key="photo_evidence_equipment")

    with col2:
        state_source = clean_event_view.copy()
        if selected_equipment not in ["未指定", "全部"] and "equipment_code" in state_source.columns:
            state_source = state_source[state_source["equipment_code"].astype(str) == selected_equipment]
        state_options = ["未指定"]
        if "state_name" in state_source.columns:
            state_options += sorted(state_source["state_name"].dropna().astype(str).unique().tolist())
        selected_state = st.selectbox("異常狀況", state_options, key="photo_evidence_state")

    col3, col4 = st.columns([1, 1])
    with col3:
        photo_type = st.selectbox(
            "照片用途",
            ["異常部位近照", "設備全景照", "正常/異常對照", "安全環境佐證", "其他"],
            key="photo_evidence_type",
        )
    with col4:
        confirm_role = st.selectbox(
            "確認人員角色",
            ["現場人員", "班長", "設備工程師", "製程工程師", "品質工程師"],
            key="photo_evidence_role",
        )

    uploaded_photo = st.file_uploader("上傳現場照片（jpg / png / jpeg）", type=["jpg", "jpeg", "png"], key="photo_evidence_upload")
    site_note = st.text_area(
        "現場補充說明",
        placeholder="例如：照片拍攝位置、異常發生時間、是否仍持續、是否已先停機、是否已通知班長。",
        height=110,
        key="photo_evidence_note",
    )

    photo_info = analyze_uploaded_photo_basic(uploaded_photo)

    st.markdown("### 照片與品質提醒")
    left, right = st.columns([0.9, 1.1])
    with left:
        if uploaded_photo is not None:
            st.image(uploaded_photo, caption=f"已上傳：{uploaded_photo.name}", use_container_width=True)
        else:
            st.warning("尚未上傳照片。")
            st.write("建議同時保留設備全景照與異常部位近照，避免事後無法判斷位置與細節。")
    with right:
        c1, c2, c3 = st.columns(3)
        c1.metric("照片品質", photo_info.get("quality_level", "未上傳"))
        c2.metric("照片用途", photo_type)
        c3.metric("確認角色", confirm_role)

        quality_notes = photo_info.get("notes", [])
        if quality_notes:
            st.markdown("#### 系統提醒")
            for note in quality_notes[:5]:
                st.write(f"- {note}")
        else:
            st.write("上傳照片後，系統會檢查解析度、亮度、對比與清晰度。")

    st.markdown("### 現場確認清單")
    checklist = [
        "照片是否能看出設備位置？",
        "照片是否能看出異常部位？",
        "是否需要補拍正常狀態作為對照？",
        "是否已記錄感測器數值、處置方式與復機結果？",
    ]
    for idx, item in enumerate(checklist, start=1):
        st.checkbox(item, key=f"photo_check_{idx}")

    st.markdown("### 產生照片佐證紀錄")
    filename = photo_info.get("filename", "未上傳照片")
    quality_notes = photo_info.get("notes", [])
    evidence_lines = [
        "【現場照片佐證紀錄】",
        f"設備：{selected_equipment}",
        f"異常狀況：{selected_state}",
        f"照片檔名：{filename}",
        f"照片用途：{photo_type}",
        f"確認角色：{confirm_role}",
        f"照片品質：{photo_info.get('quality_level', '未上傳')}",
        "",
        "一、現場補充說明",
        site_note.strip() if site_note.strip() else "尚未填寫現場補充說明。",
        "",
        "二、照片品質提醒",
        *([f"- {n}" for n in quality_notes[:5]] if quality_notes else ["- 尚未上傳照片或無明顯品質提醒。"]),
        "",
        "三、後續建議",
        "- 若照片看不清異常部位，請補拍近照與全景照。",
        "- 依 SOP 確認需要人工判斷的節點，並把照片、量測值與處置結果一起保存。",
        "- 原因判斷請搭配 AI 異常診斷建議與設備原因查詢，不要只依照片下結論。",
    ]
    evidence_text = "\n".join(evidence_lines)
    st.text_area("可複製的照片佐證紀錄", evidence_text, height=330)
    st.download_button(
        "下載照片佐證紀錄 TXT",
        data=evidence_text,
        file_name="photo_evidence_record.txt",
        mime="text/plain",
    )


# =========================================================
# Page 8：互動式知識圖譜（已整併）
# =========================================================

elif page == "互動式知識圖譜":
    st.title("互動式知識圖譜已整併")
    st.info("此版本已將原本容易空白的互動式圖譜頁移除，改由『設備原因查詢』提供更穩定、可讀的關聯查詢。")

# =========================================================
# Page 9：主動預警 Agent
# =========================================================

elif page == "主動預警 Agent":
    st.title("📡 主動預警 Agent｜Predictive Maintenance")
    st.caption("從 sensor snapshot 中找出超規、接近上下限或趨勢惡化的監控項目，將系統從被動處置延伸到主動巡檢。")

    alerts = build_predictive_maintenance_alerts(clean_sensor_view, clean_event_view)
    if alerts.empty:
        st.warning("目前 sensor_snapshot 欄位不足，無法建立主動預警表。請確認 clean_sensor_view.csv 是否包含 actual_value、spec_lower、spec_upper、judgement 等欄位。")
    else:
        eq_options = ["全部"]
        if "equipment_code" in alerts.columns:
            eq_options += sorted(alerts["equipment_code"].dropna().astype(str).unique().tolist())
        selected_eq = st.selectbox("選擇設備", eq_options, key="predictive_eq")

        show_alerts = alerts.copy()
        if selected_eq != "全部" and "equipment_code" in show_alerts.columns:
            show_alerts = show_alerts[show_alerts["equipment_code"].astype(str) == selected_eq]

        high_count = (show_alerts["risk_level"] == "高").sum() if "risk_level" in show_alerts.columns else 0
        mid_count = (show_alerts["risk_level"] == "中").sum() if "risk_level" in show_alerts.columns else 0
        low_count = (show_alerts["risk_level"] == "低").sum() if "risk_level" in show_alerts.columns else 0
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("高風險項目", f"{high_count:,}")
        c2.metric("中風險項目", f"{mid_count:,}")
        c3.metric("低風險項目", f"{low_count:,}")
        c4.metric("監控項目數", f"{len(show_alerts):,}")

        st.markdown("### 預警排序表")
        display_cols = [c for c in [
            "equipment_code", "equipment_name", "monitor_name", "parameter_name", "latest_value", "unit",
            "spec_lower", "spec_upper", "trend", "abnormal_count", "near_limit_count", "risk_score", "risk_level", "risk_reason", "suggestion"
        ] if c in show_alerts.columns]
        st.dataframe(show_alerts[display_cols].head(50), use_container_width=True, hide_index=True)

        col_chart1, col_chart2 = st.columns(2)
        with col_chart1:
            if "risk_level" in show_alerts.columns:
                risk_df = show_alerts["risk_level"].value_counts().reset_index()
                risk_df.columns = ["risk_level", "count"]
                fig = px.bar(risk_df, x="risk_level", y="count", title="風險等級分布")
                st.plotly_chart(fig, use_container_width=True)
        with col_chart2:
            top_risk = show_alerts.head(10).copy()
            if "monitor_name" in top_risk.columns and "risk_score" in top_risk.columns:
                label_cols = [c for c in ["equipment_code", "monitor_name", "parameter_name"] if c in top_risk.columns]
                top_risk["label"] = top_risk[label_cols].astype(str).agg("｜".join, axis=1)
                fig = px.bar(top_risk, x="risk_score", y="label", orientation="h", title="Top 10 預警項目")
                fig.update_layout(yaxis={"categoryorder": "total ascending"})
                st.plotly_chart(fig, use_container_width=True)

        st.markdown("### 預警文字範例")
        if not show_alerts.empty:
            top = show_alerts.iloc[0]
            warning_text = f"""【主動預警 Agent 建議】

設備：{_safe_text(top.get('equipment_code'))} {_safe_text(top.get('equipment_name'), '')}
監控項目：{_safe_text(top.get('monitor_name'))} / {_safe_text(top.get('parameter_name'))}
目前風險等級：{_safe_text(top.get('risk_level'))}
近期趨勢：{_safe_text(top.get('trend'))}
判斷理由：{_safe_text(top.get('risk_reason'))}

建議：{_safe_text(top.get('suggestion'))}

注意：目前為簡化版風險排序，並非保證未來一定故障；正式版本需要連續時間序列資料、設備維修紀錄與模型驗證。"""
            st.text_area("可複製到報告 / 通報的預警文字", warning_text, height=280)
            st.download_button("下載主動預警 TXT", warning_text, file_name="predictive_maintenance_alert.txt", mime="text/plain")

        with st.expander("補充說明"):
            st.markdown("""
            本頁將系統從「異常發生後才查 SOP」延伸為「異常發生前先做風險提示」。
            Agent 會檢查 sensor snapshot 中是否有超出上下限、接近規格邊界、異常 judgement 或近期趨勢變化的項目，
            並產生高 / 中 / 低風險等級與巡檢建議。

            由於目前資料主要是異常事件快照，而不是完整連續感測器時間序列，因此本頁採用簡化版趨勢預警；
            未來若接入即時監控資料，可進一步訓練預測性維護模型，提前預測軸承、切刀、導輪或水量異常風險。
            """)


# =========================================================
# Page 10：SOP 改善建議
# =========================================================

elif page == "SOP 改善建議":
    st.title("🧩 SOP 改善建議")
    st.caption("利用歷史異常事件與 SOP 步驟資料，找出最值得優先改善的設備、異常與流程節點。")

    st.info("""
    這一頁的重點是：系統不只是查 SOP，而是能用歷史處置紀錄反推 SOP 哪些地方需要改善。
    例如：哪種設備異常平均停機時間最高、哪些 SOP 節點最常需要人工確認、哪些原因反覆出現。
    """)

    high_downtime, manual_steps, repeated_abnormal = build_sop_improvement_tables(
        clean_event_view,
        clean_sop_view
    )

    tab1, tab2, tab3, tab4 = st.tabs([
        "停機時間優先改善",
        "人工確認節點",
        "重複發生原因",
        "報告可用結論"
    ])

    with tab1:
        st.markdown("### 1. 平均停機時間較高的設備與異常")
        st.caption("平均停機時間越高，代表該類異常對產線影響越大，適合優先檢討 SOP 或設備維護策略。")

        if high_downtime.empty:
            st.warning("目前沒有足夠的停機時間資料。")
        else:
            st.dataframe(high_downtime, use_container_width=True, hide_index=True)

            if {"equipment_code", "state_name", "avg_downtime_min"}.issubset(high_downtime.columns):
                chart_df = high_downtime.copy()
                chart_df["equipment_state"] = chart_df["equipment_code"].astype(str) + "｜" + chart_df["state_name"].astype(str)
                fig = px.bar(
                    chart_df.head(10),
                    x="equipment_state",
                    y="avg_downtime_min",
                    text="event_count",
                    title="平均停機時間 Top 10"
                )
                fig.update_layout(xaxis_title="設備｜異常狀況", yaxis_title="平均停機時間（分鐘）")
                st.plotly_chart(fig, use_container_width=True)

    with tab2:
        st.markdown("### 2. 最需要人工確認的 SOP 節點")
        st.caption("這些節點通常需要照片、目視、現場量測或工程師判斷，是 AI 無法完全自動化的地方。")

        if manual_steps.empty:
            st.warning("目前沒有整理出明確需要人工確認的 SOP 節點。")
        else:
            st.dataframe(manual_steps, use_container_width=True, hide_index=True)

            st.markdown("#### 改善方向")
            st.markdown("""
            - 若同一節點經常需要人工補資料，可以把「需要拍什麼、量什麼、回報什麼」寫進 SOP。
            - 若該節點其實可由感測器判斷，可以新增 monitor_id 或 parameter_spec。
            - 若現場常不知道如何判斷，應補上正常/異常照片或判斷範例。
            """)

    with tab3:
        st.markdown("### 3. 重複發生的設備、異常與原因")
        st.caption("如果同一類原因反覆出現，代表可能不是單次偶發，而是 SOP、設備保養或製程條件需要改善。")

        if repeated_abnormal.empty:
            st.warning("目前沒有足夠的原因分類資料。")
        else:
            st.dataframe(repeated_abnormal, use_container_width=True, hide_index=True)

            if {"equipment_code", "state_name", "root_cause_category", "event_count"}.issubset(repeated_abnormal.columns):
                chart_df = repeated_abnormal.copy()
                chart_df["label"] = (
                    chart_df["equipment_code"].astype(str)
                    + "｜"
                    + chart_df["state_name"].astype(str)
                    + "｜"
                    + chart_df["root_cause_category"].astype(str)
                )
                fig = px.bar(
                    chart_df.head(10),
                    x="label",
                    y="event_count",
                    title="重複發生原因 Top 10"
                )
                fig.update_layout(xaxis_title="設備｜異常｜原因", yaxis_title="事件數")
                st.plotly_chart(fig, use_container_width=True)

    with tab4:
        st.markdown("### 4. 可直接放進報告的結論")
        st.markdown("""
        本系統除了提供 SOP 查詢與 AI 診斷外，也能透過歷史異常資料回饋 SOP 改善方向。
        若某些設備異常的平均停機時間較高，代表該類異常對產線影響較大，應優先檢討處置流程；
        若某些 SOP 節點經常需要人工確認，代表該節點可能需要補充照片、判斷標準或感測器監控；
        若同一原因分類反覆發生，則可進一步檢討設備保養、製程條件或教育訓練是否不足。

        因此，本專題的價值不只是「查 SOP」或「產生 AI 文字」，而是將異常處理資料轉成可持續改善的知識庫，
        讓 SOP 可以隨著歷史案例與現場經驗逐步更新。
        """)

        st.download_button(
            label="下載 SOP 改善結論 TXT",
            data="""【SOP 改善建議結論】

本系統除了提供 SOP 查詢與 AI 診斷外，也能透過歷史異常資料回饋 SOP 改善方向。
若某些設備異常的平均停機時間較高，代表該類異常對產線影響較大，應優先檢討處置流程；
若某些 SOP 節點經常需要人工確認，代表該節點可能需要補充照片、判斷標準或感測器監控；
若同一原因分類反覆發生，則可進一步檢討設備保養、製程條件或教育訓練是否不足。

因此，本專題的價值不只是查 SOP 或產生 AI 文字，而是將異常處理資料轉成可持續改善的知識庫，
讓 SOP 可以隨著歷史案例與現場經驗逐步更新。
""",
            file_name="SOP_improvement_conclusion.txt",
            mime="text/plain"
        )



Writing app.py


## 安裝 Streamlit 與 localtunnel

本段會安裝與啟動 Streamlit 所需的工具。

在 Colab 中執行 Streamlit 時，通常需要透過 localtunnel 或類似工具產生外部網址，讓瀏覽器可以開啟系統。


In [73]:
!pip install streamlit plotly pandas pillow numpy pyvis openpyxl networkx

## 啟動 Streamlit

本段會啟動 Streamlit 網頁系統。

執行後會產生一個網址，點開後即可使用製造異常診斷系統。


> 舊版除錯／修正程式已整合進上方 `app.py`，此格不需要再執行。

> 舊版除錯／修正程式已整合進上方 `app.py`，此格不需要再執行。

In [74]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

> 舊版除錯／修正程式已整合進上方 `app.py`，此格不需要再執行。

> 舊版除錯／修正程式已整合進上方 `app.py`，此格不需要再執行。

> 舊版除錯／修正程式已整合進上方 `app.py`，此格不需要再執行。

In [75]:
!python -m py_compile app.py

In [76]:
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  & ./cloudflared tunnel --url http://localhost:8501

2026-05-24T17:51:28Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-24T17:51:28Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-05-24T17:51:33Z INF +--------------------------------------------------------------------------------------------+
2026-05-24T17:51:33Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-24T17:51:33Z INF |  https://sum-genesis-advisor-settled.trycloudflare.c

## 如果 localtunnel 要你輸入 Endpoint IP

如果 localtunnel 要求輸入 Endpoint IP，可以執行下方程式取得目前 Colab 環境的 IP 位址。

把顯示出來的 IP 貼到 localtunnel 頁面，就能開啟 Streamlit 網頁。
